# 🦷 DentalOPG-1550 | AI vs. Human Observers — End-to-End Research Notebook
**Study title:** *Deep Learning Versus Human Observers for Multi-Class Dental Findings Detection in Panoramic Radiographs: A Comparative Diagnostic Accuracy Study*

**AI models:** YOLOv8s · YOLO11s · RT-DETR-L  
**Six target classes:** Caries · Missing Teeth · Root Canal Treatment · Filling · Crown · Unerupted Wisdom Teeth  
**Human comparison:** Two dentists independently annotate **the same held-out test images**.  
**Notebook version:** Research Notebook 1.0 (English edition) · 12 October 2026

> **How to run:** In Google Colab, select `Runtime → Change runtime type → GPU` (T4, L4, or A100, if available). Run the cells **sequentially**. Some steps require manual review, reader participation, or ethical approval, which code cannot replace. Outputs are stored in the Google Drive research folder. **Do not select “Run all” before passing the required quality-control gates.**

**Core resources:** [Ultralytics](https://docs.ultralytics.com/) · [RT-DETR](https://docs.ultralytics.com/models/rtdetr/) · [STARD-AI](https://www.nature.com/articles/s41591-025-03953-8) · [CLAIM 2024](https://pmc.ncbi.nlm.nih.gov/articles/PMC11304031/) · [DentalOPG-1550 V3](https://data.mendeley.com/datasets/rtt726b26d/3)


## 01 — Environment and library installation
**Dependencies:** Ultralytics, PyTorch, NumPy, Pandas, SciPy, scikit-learn, Pillow, ImageHash, PyYAML, Matplotlib, OpenPyXL, and tqdm.


In [ ]:
%pip -q install "ultralytics>=8.3,<9" "numpy<3" pandas scipy scikit-learn Pillow imagehash PyYAML matplotlib openpyxl tqdm statsmodels

In [ ]:
import os, sys, json, csv, time, random, shutil, re, hashlib, platform, subprocess
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import imagehash
import yaml
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO, RTDETR
from google.colab import drive
from IPython.display import display

drive.mount('/content/drive')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

ROOT = Path('/content/drive/MyDrive/DentalOPG_Research_2026')
SOURCE = ROOT / '01_Datasets/YOLO_Export'
ORIGINAL = ROOT / '01_Datasets/Original_DentalOPG'
PREPARED = ROOT / '01_Datasets/Prepared_6_Classes'
QC = ROOT / '01_Datasets/Dataset_QC'
PROTOCOL = ROOT / '00_Research_Protocol'
RESTRICTED = PROTOCOL / 'Restricted_DONOTSHARE'
MODELS_DIR = ROOT / '03_Trained_Models'
RESULTS_DIR = ROOT / '04_Results'
READERS_DIR = ROOT / '05_Dentist_Evaluation'
MANUSCRIPT = ROOT / '06_Manuscript'

for d in [QC, PREPARED, RESTRICTED, MODELS_DIR, RESULTS_DIR/'Metrics',
          RESULTS_DIR/'Figures', RESULTS_DIR/'Predictions',
          RESULTS_DIR/'Statistical_Analysis',
          READERS_DIR/'Blinded_Images', READERS_DIR/'Reader_1',
          READERS_DIR/'Reader_2', READERS_DIR/'Reference_Standard'/'labels',
          MANUSCRIPT/'Drafts', MANUSCRIPT/'Final_Submission']:
    d.mkdir(parents=True, exist_ok=True)

if not SOURCE.exists():
    raise FileNotFoundError('YOLO_Export directory was not found. Check the Google Drive path.')

SOURCE_CLASS_NAMES = ['BDR', 'Bridge', 'Cap', 'Caries', 'Distoangular impaction',
                      'Filling', 'Horizontal Impaction', 'Missing Teeth',
                      'Mixed Dentition', 'Normal Teeth', 'Periodontities',
                      'Rootcanel', 'Unerupted Wisdom Teeth', 'apical abscss']
SELECTED = [(3,'Caries'), (7,'Missing Teeth'), (11,'Root Canal Treatment'),
            (5,'Filling'), (2,'Crown'), (12,'Unerupted Wisdom Teeth')]
OLD_TO_NEW = {old: new for new, (old, _) in enumerate(SELECTED)}
CLASS_NAMES = [label for _, label in SELECTED]
IMAGE_EXT = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
SPLITS = ('train','valid','test')

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, '| Ultralytics:', ultralytics.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available')
print('Project:', ROOT)
print('Target classes:', dict(enumerate(CLASS_NAMES)))

### 01.1 — Register the protocol and software versions
Save the initial configuration and software environment before viewing final test results. **Change experimental settings only before the final test**, or document subsequent changes explicitly as protocol amendments.


In [ ]:
CONFIG = {
    'research_title': 'Deep Learning Versus Human Observers for Multi-Class Dental Findings Detection in Panoramic Radiographs',
    'source_dataset': 'DentalOPG-1550 (Roboflow YOLO export v4; investigate relation to Mendeley V3)',
    'mendeley_doi': '10.17632/rtt726b26d.3',
    'source_class_names': SOURCE_CLASS_NAMES,
    'selected_original_ids': [x for x,_ in SELECTED],
    'selected_class_names': CLASS_NAMES,
    'seed': SEED,
    'model_weights': {'yolov8s': 'yolov8s.pt', 'yolo11s': 'yolo11s.pt', 'rtdetr_l': 'rtdetr-l.pt'},
    'epochs_max': 100,
    'early_stopping_patience': 20,
    'image_size': 640,
    'test_operating_confidence': 0.25,  # fixed a priori; do not tune on test
    'test_matching_iou': 0.50,
    'training_batch': {'yolov8s': 8, 'yolo11s': 8, 'rtdetr_l': 2},
    'external_validation': False,
    'clinical_reference_status': 'NOT YET ADJUDICATED'
}
config_file = PROTOCOL / 'study_config_v1.json'
if config_file.exists():
    prior = json.loads(config_file.read_text(encoding='utf-8'))
    if prior != CONFIG:
        print('⚠️ The previous protocol file differs. Do not change the protocol after viewing test results without a documented amendment.')
        print('The existing protocol file will be preserved and will not be overwritten.')
else:
    config_file.write_text(json.dumps(CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')

(PROTOCOL / 'environment_versions.txt').write_text(
    '\n'.join([f'Python={platform.python_version()}', f'PyTorch={torch.__version__}',
                 f'Ultralytics={ultralytics.__version__}',
                 subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True).stdout]),
    encoding='utf-8'
)
print('Study configuration:', config_file)

## 02 — Source-dataset quality control
**Do not overlook** missing images or label files, malformed or out-of-bounds bounding boxes, or overlap between training and test splits. The audit checks paths, labels, and annotation values and produces aggregate reports without printing patient names. It also screens possible duplicates using filenames, SHA-256, and perceptual image hashes (pHash). Similar pHash values require visual confirmation; visual similarity alone does not establish duplicate identity.


In [ ]:

def read_yolo_label(path):
    boxes, errors = [], []

    if not path.exists():
        return [], ["MISSING_LABEL"]

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for ln, line in enumerate(lines, 1):
        if not line.strip():
            continue

        vals = line.split()

        try:
            raw_id = float(vals[0])

            if not np.isfinite(raw_id):
                raise ValueError("Non-finite class ID")

            cid = int(raw_id)

            if raw_id != cid or not 0 <= cid < 14:
                raise ValueError("Invalid class ID")

            # Standard YOLO object detection annotation
            if len(vals) == 5:
                xywh = [float(v) for v in vals[1:]]

            # YOLO polygon segmentation annotation
            elif len(vals) >= 7 and len(vals) % 2 == 1:

                # Mixed-format polygons are expected
                # only for the Normal Teeth class
                if cid != 9:
                    raise ValueError(
                        "Unexpected polygon class"
                    )

                coords = [float(v) for v in vals[1:]]

                if not all(
                    np.isfinite(v) and 0 <= v <= 1
                    for v in coords
                ):
                    raise ValueError(
                        "Invalid polygon coordinates"
                    )

                xs = coords[0::2]
                ys = coords[1::2]

                x_min, x_max = min(xs), max(xs)
                y_min, y_max = min(ys), max(ys)

                # Convert polygon to an axis-aligned box
                xywh = [
                    (x_min + x_max) / 2,
                    (y_min + y_max) / 2,
                    x_max - x_min,
                    y_max - y_min
                ]

            else:
                raise ValueError(
                    "Unsupported annotation format"
                )

            x, y, w, h = xywh

            if not (
                all(np.isfinite(v) for v in xywh)
                and all(0 <= v <= 1 for v in xywh)
                and w > 0
                and h > 0
            ):
                raise ValueError(
                    "Invalid bounding box"
                )

            tolerance = 0.002

            if (
                x - w / 2 < -tolerance
                or x + w / 2 > 1 + tolerance
                or y - h / 2 < -tolerance
                or y + h / 2 > 1 + tolerance
            ):
                raise ValueError(
                    "Bounding box exceeds image limits"
                )

            boxes.append((cid, *xywh))

        except (ValueError, IndexError) as e:
            errors.append(
                f"line={ln}:{e}"
            )

    return boxes, errors


def source_image_id(stem):
    # strips Roboflow image-specific export hash, used only as indicative source ID
    return stem.split('.rf.')[0]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

AUDIT = {}
all_records = []
all_errors = []
for split in SPLITS:
    src = SOURCE / split
    ims = {p.stem:p for p in (src/'images').iterdir() if p.suffix.lower() in IMAGE_EXT} if (src/'images').exists() else {}
    lbs = {p.stem:p for p in (src/'labels').glob('*.txt')} if (src/'labels').exists() else {}
    missing = set(ims)-set(lbs)
    orphan = set(lbs)-set(ims)
    counts, positives = Counter(), Counter()
    for stem, im in sorted(ims.items()):
        if stem in lbs:
            boxes, errors = read_yolo_label(lbs[stem])
            for cid in set(c for c,*_ in boxes): positives[cid] += 1
            counts.update(c for c,*_ in boxes)
            for err in errors: all_errors.append((split,'INVALID_LABEL',err))
        else:
            boxes=[]
            all_errors.append((split,'MISSING_LABEL','one image without label'))
        all_records.append({'split':split,'stem':stem,'source_id':source_image_id(stem),
                            'image_path':str(im),'label_path':str(lbs.get(stem, '')),
                            'objects':len(boxes),'sha256':None,'phash':None})
    for _ in orphan: all_errors.append((split,'ORPHAN_LABEL','one label without image'))
    AUDIT[split] = {'images':len(ims),'labels':len(lbs),'paired':len(set(ims)&set(lbs)),
                    'images_without_labels':len(missing),'labels_without_images':len(orphan),
                    'class_object_counts':{str(i):counts[i] for i in range(14)},
                    'class_positive_images':{str(i):positives[i] for i in range(14)}}

print('SPLIT COUNTS')
display(pd.DataFrame({k: {n: AUDIT[k][n] for n in ['images','labels','paired',
                 'images_without_labels','labels_without_images']} for k in SPLITS}).T)
print('TARGET CLASSES: objects / positive images in each split')
rows=[]
for orig_id, cname in SELECTED:
    row={'class':cname,'old_id':orig_id}
    for split in SPLITS:
        row[f'{split}_objects']=AUDIT[split]['class_object_counts'][str(orig_id)]
        row[f'{split}_positive_images']=AUDIT[split]['class_positive_images'][str(orig_id)]
    rows.append(row)
class_counts_df = pd.DataFrame(rows)
display(class_counts_df)
class_counts_df.to_csv(QC/'target_class_distribution.csv',index=False)

print('Input errors:',len(all_errors))
print('Original BMP count:',sum(p.suffix.lower()=='.bmp' for p in ORIGINAL.glob('*')) if ORIGINAL.exists() else 'missing')
(QC/'initial_audit_summary.json').write_text(json.dumps(AUDIT,indent=2),encoding='utf-8')
if all_errors:
    raise RuntimeError('❌ Source dataset has missing or invalid annotations. Fix before proceeding.')
if sum(AUDIT[s]['images'] for s in SPLITS) != 1525:
    print('⚠️ Image count differs from announced 1,525; investigate dataset version before proceeding.')

### 02.1 — Duplicate and split-leakage screening
Analyze image identifiers and calculate SHA-256 and perceptual hashes for **all 1,525 annotated images**. Hashing files stored on Drive may take several minutes. Visually investigate any near-duplicate pHash matches. **Confirmed overlap between training and test images is a stop condition** until a documented corrective decision is made.


In [ ]:
records_df = pd.DataFrame(all_records)
cache_path = QC/'image_hash_cache.csv'
cache = {}
if cache_path.exists():
    old_hash = pd.read_csv(cache_path)
    cache = {r.image_path:(r.sha256,r.phash) for r in old_hash.itertuples(index=False)}

hashes=[]
from tqdm.auto import tqdm
for rec in tqdm(all_records, desc='Hash images'):
    f=rec['image_path']
    if f in cache and pd.notna(cache[f][0]) and pd.notna(cache[f][1]):
        sh,ph=cache[f]
    else:
        sh=sha256_file(f)
        with Image.open(f) as im:
            ph=str(imagehash.phash(im.convert('RGB')))
    hashes.append({'image_path':f,'sha256':sh,'phash':ph})

hash_df=pd.DataFrame(hashes)
hash_df.to_csv(cache_path,index=False)
records_df=records_df.drop(columns=['sha256','phash']).merge(hash_df,on='image_path')

cross_id=records_df.groupby('source_id')['split'].nunique()
cross_sha=records_df.groupby('sha256')['split'].nunique()
flagged_id=set(cross_id[cross_id>1].index)
flagged_sha=set(cross_sha[cross_sha>1].index)

# Candidate near-duplicates; pairwise comparison only across different splits
from itertools import combinations
near=[]
for a,b in combinations(SPLITS,2):
    left=records_df[records_df.split==a][['phash']].drop_duplicates()
    right=records_df[records_df.split==b][['phash']].drop_duplicates()
    # integer XOR Hamming distances, avoids showing patient filenames
    l=[int(h,16) for h in left.phash]
    r=[int(h,16) for h in right.phash]
    for x in l:
        for y in r:
            if (x^y).bit_count()<=4:
                near.append({'split_pair':f'{a}-{b}','phash_hamming':(x^y).bit_count(),
                             'phash_a':f'{x:016x}','phash_b':f'{y:016x}'})

near_df=pd.DataFrame(near)
near_df.to_csv(QC/'near_duplicates_RESTRICTED_review.csv',index=False)
leak_report = {'cross_split_same_source_id_groups':len(flagged_id),
               'cross_split_exact_duplicate_groups':len(flagged_sha),
               'cross_split_near_duplicate_pairs':len(near)}
(QC/'leakage_summary.json').write_text(json.dumps(leak_report,indent=2),encoding='utf-8')
print('LEAKAGE SUMMARY:',leak_report)
if flagged_id or flagged_sha:
    raise RuntimeError('❌ Confirmed/potential duplicate IDs or exact image copies across splits. Resolve and document BEFORE training.')
if near:
    print('⚠️ Near-duplicate candidates need manual review. Report saved under Dataset_QC.')
print('Note: different files from the SAME PATIENT remain possible. Verify patient-level grouping with the source/provider.')

In [ ]:

from pathlib import Path
import pandas as pd
import json

QC = Path(
    "/content/drive/MyDrive/"
    "DentalOPG_Research_2026/"
    "01_Datasets/Dataset_QC"
)

print("===== LEAKAGE REPORT INSPECTION =====")

for file in sorted(QC.iterdir()):

    if not file.is_file():
        continue

    print(f"\nReport: {file.name}")

    if file.suffix.lower() == ".csv":
        try:
            df = pd.read_csv(file)

            print("Rows:", len(df))
            print("Columns:", list(df.columns))

        except Exception as e:
            print("Read error:", str(e))

    elif file.suffix.lower() == ".json":
        try:
            with open(file, encoding="utf-8") as f:
                data = json.load(f)

            if isinstance(data, dict):
                print("Keys:", list(data.keys()))
            elif isinstance(data, list):
                print("Records:", len(data))

        except Exception as e:
            print("Read error:", str(e))

print("\n✅ REPORT INSPECTION COMPLETED")
print("No files were modified.")


In [ ]:

from pathlib import Path
from collections import defaultdict
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display, Image as ColabImage
import pandas as pd
import re
import math

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")
QC = ROOT / "01_Datasets/Dataset_QC"

REVIEW = QC / "Near_Duplicate_Review_RESTRICTED"
SHEETS = REVIEW / "contact_sheets"
SHEETS.mkdir(parents=True, exist_ok=True)

cache = pd.read_csv(
    QC / "image_hash_cache.csv", dtype=str
).fillna("")

pairs = pd.read_csv(
    QC / "near_duplicates_RESTRICTED_review.csv",
    dtype=str
).fillna("")

def normalize_hash(value):
    return str(value).strip().lower().removeprefix("0x")

def get_split(path):
    match = re.search(
        r"[/\\](train|valid|test)[/\\]images[/\\]",
        str(path),
        flags=re.I
    )
    return match.group(1).lower() if match else None

hash_index = defaultdict(list)

for _, row in cache.iterrows():
    h = normalize_hash(row["phash"])
    path = str(row["image_path"])
    split = get_split(path)
    if split:
        hash_index[h].append((path, split))

resolved = []
unresolved = []

for i, row in pairs.iterrows():
    ha = normalize_hash(row["phash_a"])
    hb = normalize_hash(row["phash_b"])

    declared_splits = re.findall(
        r"(?<![a-z])(train|valid|test)(?![a-z])",
        str(row["split_pair"]).lower()
    )

    options = {}

    for pa, sa in hash_index.get(ha, []):
        for pb, sb in hash_index.get(hb, []):
            if pa == pb or sa == sb:
                continue

            if len(declared_splits) == 2:
                if sorted([sa, sb]) != sorted(declared_splits):
                    continue

            key = tuple(sorted([pa, pb]))
            options[key] = (pa, pb, sa, sb)

    if len(options) != 1:
        unresolved.append({
            "report_row": i + 1,
            "candidate_pairs": len(options)
        })
        continue

    pa, pb, sa, sb = next(iter(options.values()))

    resolved.append({
        "pair_id": f"ND-{i+1:03d}",
        "split_a": sa,
        "split_b": sb,
        "hamming": row["phash_hamming"],
        "path_a": pa,
        "path_b": pb
    })

print("===== PAIR RESOLUTION =====")
print("Reported pairs:", len(pairs))
print("Uniquely resolved:", len(resolved))
print("Ambiguous/unresolved:", len(unresolved))

if unresolved:
    display(pd.DataFrame(unresolved).head(20))
    raise RuntimeError(
        "Some hash pairs cannot be uniquely mapped "
        "to images. Review before generating sheets."
    )

# Private mapping: contains original identifying filenames.
pd.DataFrame(resolved).to_csv(
    REVIEW / "RESTRICTED_private_pair_mapping.csv",
    index=False
)

# Reviewer worksheet: contains no original filenames.
review_rows = [
    {
        "pair_id": r["pair_id"],
        "split_a": r["split_a"],
        "split_b": r["split_b"],
        "phash_hamming": r["hamming"],
        "decision": "",
        "reviewer": "",
        "notes": ""
    }
    for r in resolved
]

pd.DataFrame(review_rows).to_csv(
    REVIEW / "near_duplicate_decisions.csv",
    index=False
)

def load_thumbnail(path, max_size=(675, 405)):
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
        image.thumbnail(max_size)
        return image.copy()

PER_SHEET = 4
sheet_paths = []

for start in range(0, len(resolved), PER_SHEET):
    batch = resolved[start:start + PER_SHEET]

    canvas = Image.new("RGB", (1440, 1900), "white")
    draw = ImageDraw.Draw(canvas)

    draw.text(
        (25, 15),
        "NEAR-DUPLICATE REVIEW - RESTRICTED",
        fill="black"
    )

    for j, item in enumerate(batch):
        y = 65 + j * 455

        draw.text(
            (25, y),
            f"{item['pair_id']} | "
            f"Hamming: {item['hamming']}",
            fill="black"
        )

        for col, side in enumerate(["a", "b"]):
            x = 25 + col * 710

            draw.text(
                (x, y + 24),
                f"Image {side.upper()} - "
                f"{item['split_' + side].upper()}",
                fill="black"
            )

            thumb = load_thumbnail(item["path_" + side])
            canvas.paste(thumb, (x, y + 48))

    page_number = start // PER_SHEET + 1
    output_file = SHEETS / f"review_sheet_{page_number:02d}.jpg"

    canvas.save(output_file, quality=92)
    sheet_paths.append(output_file)

print("\n===== REVIEW PACKAGE =====")
print("Contact sheets:", len(sheet_paths))
print("Decision CSV created: YES")
print("Private mapping created: YES")
print("Original dataset modified: NO")

if sheet_paths:
    print("\nFIRST REVIEW SHEET:")
    display(ColabImage(filename=str(sheet_paths[0])))

print("\nREVIEW FOLDER:")
print(REVIEW)


In [ ]:

from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display, clear_output
import ipywidgets as widgets
import pandas as pd
import os

REVIEW = (
    Path("/content/drive/MyDrive/DentalOPG_Research_2026")
    / "01_Datasets/Dataset_QC"
    / "Near_Duplicate_Review_RESTRICTED"
)

DECISIONS = REVIEW / "near_duplicate_decisions.csv"
MAPPING = REVIEW / "RESTRICTED_private_pair_mapping.csv"

df = pd.read_csv(DECISIONS, dtype=str).fillna("")
mapping = pd.read_csv(MAPPING, dtype=str).set_index("pair_id")

assert df["pair_id"].is_unique
assert df["pair_id"].isin(mapping.index).all()

state = {"index": 0}

choices = [
    ("Select decision", ""),
    ("Same radiograph", "SAME_RADIOGRAPH"),
    ("Same patient, different radiograph", "SAME_PATIENT"),
    ("Different patients", "DIFFERENT_PATIENTS"),
    ("Uncertain / needs verification", "UNCERTAIN")
]

decision = widgets.Dropdown(
    options=choices, description="Decision:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="480px")
)

notes = widgets.Textarea(
    description="Notes:",
    placeholder="Optional clinical observations",
    layout=widgets.Layout(width="600px", height="65px"),
    style={"description_width": "initial"}
)

status = widgets.HTML()
output = widgets.Output()

previous = widgets.Button(description="Previous")
save_next = widgets.Button(
    description="Save & Next", button_style="success"
)
next_button = widgets.Button(description="Next")

def thumbnail(path):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im.thumbnail((650, 400))
        result = Image.new("RGB", (650, 400), "white")
        result.paste(
            im, ((650 - im.width)//2, (400 - im.height)//2)
        )
        return result

def render():
    i = state["index"]
    row = df.iloc[i]
    item = mapping.loc[row["pair_id"]]

    saved = int(df["decision"].ne("").sum())
    status.value = (
        f"<b>Pair {i+1}/{len(df)}</b> | "
        f"Reviewed: {saved}/{len(df)}"
    )

    current = row["decision"]
    decision.value = current if current in [
        v for _, v in choices
    ] else ""
    notes.value = row["notes"]

    canvas = Image.new("RGB", (1320, 445), "white")
    draw = ImageDraw.Draw(canvas)

    for col, side in enumerate(["a", "b"]):
        x = col * 660
        draw.text(
            (x+10, 5),
            f"Image {side.upper()} | "
            f"{item['split_' + side].upper()}",
            fill="black"
        )
        canvas.paste(
            thumbnail(item["path_" + side]),
            (x+5, 30)
        )

    with output:
        clear_output(wait=True)
        display(canvas)

def save_current():
    if not decision.value:
        status.value = (
            "<b>Please select a decision before saving.</b>"
        )
        return False

    i = state["index"]
    df.loc[i, "decision"] = decision.value
    df.loc[i, "notes"] = notes.value

    temporary = DECISIONS.with_suffix(".csv.tmp")
    df.to_csv(temporary, index=False)
    os.replace(temporary, DECISIONS)
    return True

def save_and_advance(_):
    if save_current():
        if state["index"] < len(df)-1:
            state["index"] += 1
        render()

def move(step):
    state["index"] = max(
        0, min(len(df)-1, state["index"] + step)
    )
    render()

save_next.on_click(save_and_advance)
previous.on_click(lambda _: move(-1))
next_button.on_click(lambda _: move(1))

display(widgets.VBox([
    status, output, decision, notes,
    widgets.HBox([previous, save_next, next_button])
]))

render()


In [ ]:

import pandas as pd

df = pd.read_csv(DECISIONS).fillna("")

print("===== FINAL DUPLICATE REVIEW =====")

print(
    df["decision"]
    .replace("", "PENDING")
    .value_counts()
    .to_string()
)

print("\nTotal pairs:", len(df))
print("Completed:", df["decision"].ne("").sum())
print("Pending:", df["decision"].eq("").sum())


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")

REVIEW = (
    ROOT / "01_Datasets/Dataset_QC"
    / "Near_Duplicate_Review_RESTRICTED"
)

DECISIONS = REVIEW / "near_duplicate_decisions.csv"

df = pd.read_csv(
    DECISIONS, dtype=str
).fillna("")

df["decision"] = df["decision"].str.strip().str.upper()

allowed = {
    "SAME_RADIOGRAPH",
    "SAME_PATIENT",
    "DIFFERENT_PATIENTS",
    "UNCERTAIN",
    ""
}

invalid = df[~df["decision"].isin(allowed)]

if not invalid.empty:
    raise ValueError(
        "Unexpected decision values: "
        + str(invalid["decision"].unique().tolist())
    )

print("=" * 60)
print("FINAL DUPLICATE REVIEW ANALYSIS")
print("=" * 60)

print("\nTotal pairs:", len(df))
print("Reviewed:", (df["decision"] != "").sum())
print("Pending:", (df["decision"] == "").sum())

print("\n===== DECISION DISTRIBUTION =====")
print(
    df["decision"]
    .replace("", "PENDING")
    .value_counts()
    .to_string()
)

df["split_pair"] = df.apply(
    lambda r: " - ".join(sorted([
        r["split_a"].lower(),
        r["split_b"].lower()
    ])),
    axis=1
)

print("\n===== DECISIONS BY SPLIT =====")
display(pd.crosstab(
    df["split_pair"],
    df["decision"].replace("", "PENDING")
))

risk = df[
    df["decision"].isin([
        "SAME_RADIOGRAPH",
        "SAME_PATIENT",
        "UNCERTAIN"
    ])
]

print("\n===== POSSIBLE LEAKAGE =====")
print("High-risk or unresolved pairs:", len(risk))

if not risk.empty:
    display(risk[[
        "pair_id",
        "split_pair",
        "decision",
        "phash_hamming"
    ]])

pending = (df["decision"] == "").sum()

print("\n===== FINAL STATUS =====")

if pending > 0:
    print("STOP: Some pairs are not reviewed.")
elif len(risk) > 0:
    print("REQUIRES RESOLUTION:")
    print("Potential leakage or uncertain pairs found.")
    print("Do not prepare the final dataset yet.")
else:
    print("PASS: All 68 candidate pairs were")
    print("classified as different patients.")
    print("Independent patient-level verification")
    print("is still recommended.")

print("\nNo original dataset files were modified.")


In [ ]:

from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")
SOURCE = ROOT / "01_Datasets/YOLO_Export"
QC = ROOT / "01_Datasets/Dataset_QC"

REVIEW = QC / "Near_Duplicate_Review_RESTRICTED"
RESOLUTION = QC / "Leakage_Resolution_RESTRICTED"
RESOLUTION.mkdir(parents=True, exist_ok=True)

decisions = pd.read_csv(
    REVIEW / "near_duplicate_decisions.csv",
    dtype=str,
    keep_default_na=False
)

mapping = pd.read_csv(
    REVIEW / "RESTRICTED_private_pair_mapping.csv",
    dtype=str,
    keep_default_na=False
).set_index("pair_id")

decisions["decision"] = (
    decisions["decision"].str.strip().str.upper()
)

# Safety checks
assert len(decisions) == 68
assert decisions["pair_id"].is_unique
assert decisions["pair_id"].isin(mapping.index).all()
assert decisions["decision"].isin([
    "SAME_RADIOGRAPH",
    "DIFFERENT_PATIENTS"
]).all(), "Unresolved decisions remain."

duplicates = decisions[
    decisions["decision"] == "SAME_RADIOGRAPH"
].copy()

expected_pairs = {"ND-018", "ND-019", "ND-020"}

assert set(duplicates["pair_id"]) == expected_pairs, (
    "Duplicate decisions differ from the reviewed report."
)

records = []

for _, row in duplicates.iterrows():
    pair_id = row["pair_id"]
    item = mapping.loc[pair_id]

    sides = {
        str(item["split_a"]).lower(): Path(item["path_a"]),
        str(item["split_b"]).lower(): Path(item["path_b"])
    }

    assert set(sides) == {"train", "valid"}, (
        f"Unexpected split combination: {pair_id}"
    )

    train_image = sides["train"].resolve(strict=True)
    valid_image = sides["valid"].resolve(strict=True)

    assert train_image.parent == (
        SOURCE / "train/images"
    ).resolve()

    assert valid_image.parent == (
        SOURCE / "valid/images"
    ).resolve()

    train_label = (
        SOURCE / "train/labels" /
        f"{train_image.stem}.txt"
    ).resolve(strict=True)

    valid_label = (
        SOURCE / "valid/labels" /
        f"{valid_image.stem}.txt"
    ).resolve(strict=True)

    records.append({
        "pair_id": pair_id,
        "decision": "SAME_RADIOGRAPH",
        "excluded_split": "train",
        "excluded_image_relpath": str(
            train_image.relative_to(SOURCE.resolve())
        ),
        "excluded_label_relpath": str(
            train_label.relative_to(SOURCE.resolve())
        ),
        "retained_split": "valid",
        "retained_image_relpath": str(
            valid_image.relative_to(SOURCE.resolve())
        ),
        "retained_label_relpath": str(
            valid_label.relative_to(SOURCE.resolve())
        )
    })

manifest = pd.DataFrame(records)

# Confirm that all three pairs refer to distinct images.
assert manifest["excluded_image_relpath"].is_unique
assert manifest["retained_image_relpath"].is_unique

manifest_path = (
    RESOLUTION / "training_exclusion_manifest.csv"
)

manifest.to_csv(manifest_path, index=False)

# Simulate exclusion without changing source files.
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

train_images = [
    p for p in (SOURCE / "train/images").iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXT
]

excluded = set(manifest["excluded_image_relpath"])

after_train = [
    p for p in train_images
    if str(p.relative_to(SOURCE.resolve())) not in excluded
]

valid_count = sum(
    p.is_file() and p.suffix.lower() in IMAGE_EXT
    for p in (SOURCE / "valid/images").iterdir()
)

test_count = sum(
    p.is_file() and p.suffix.lower() in IMAGE_EXT
    for p in (SOURCE / "test/images").iterdir()
)

assert len(train_images) - len(after_train) == len(excluded)

summary = {
    "reviewed_pairs": len(decisions),
    "same_radiograph_pairs": len(duplicates),
    "excluded_training_images": len(excluded),
    "train_before": len(train_images),
    "train_after": len(after_train),
    "valid_after": valid_count,
    "test_after": test_count,
    "original_dataset_modified": False
}

summary_path = (
    QC / "leakage_resolution_summary.json"
)

summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("=" * 55)
print("LEAKAGE RESOLUTION PLAN")
print("=" * 55)

print("Reviewed pairs:", summary["reviewed_pairs"])
print("Confirmed duplicate pairs:",
      summary["same_radiograph_pairs"])
print("Training images to exclude:",
      summary["excluded_training_images"])

print("\n===== PROPOSED DATASET COUNTS =====")
print("Train before:", summary["train_before"])
print("Train after:", summary["train_after"])
print("Validation:", summary["valid_after"])
print("Test:", summary["test_after"])

print("\n===== EXCLUDED PAIR IDS =====")
print(manifest["pair_id"].tolist())

print("\n===== SAFETY STATUS =====")
print("Manifest created:", manifest_path.exists())
print("Original dataset modified: NO")
print("Exclusions applied to dataset: NOT YET")
print("Leakage resolution plan: READY")


In [ ]:

from pathlib import Path
from collections import Counter
import pandas as pd
import shutil
import json
import tempfile

# ==================================================
# STEP: PREPARE CLEAN SIX-CLASS DATASET
# ==================================================

ROOT = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026"
)

SOURCE = ROOT / "01_Datasets/YOLO_Export"
QC = ROOT / "01_Datasets/Dataset_QC"

PREPARED = ROOT / "01_Datasets/Prepared_6_Classes"
DEST = PREPARED / "v1_cleaned"

MANIFEST = (
    QC / "Leakage_Resolution_RESTRICTED"
    / "training_exclusion_manifest.csv"
)

RESTRICTED = QC / "RESTRICTED_provenance"

CLASS_MAP = {
    3: (0, "Caries"),
    7: (1, "Missing Teeth"),
    11: (2, "Root Canal Treatment"),
    5: (3, "Filling"),
    2: (4, "Crown"),
    12: (5, "Unerupted Wisdom Teeth")
}

EXPECTED = {
    "train": 1065,
    "valid": 303,
    "test": 154
}

EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp"
}

# ==================================================
# 1. PRE-FLIGHT SAFETY CHECKS
# ==================================================

if not MANIFEST.exists():
    raise RuntimeError(
        "STOP: Training exclusion manifest not found. "
        "Run the leakage resolution step first."
    )

if "read_yolo_label" not in globals():
    raise RuntimeError(
        "STOP: Run the updated Dataset Audit cell "
        "before this preparation step."
    )

manifest = pd.read_csv(
    MANIFEST, dtype=str
).fillna("")

assert len(manifest) == 3, (
    "Expected exactly 3 reviewed exclusions."
)

assert set(manifest["pair_id"]) == {
    "ND-018", "ND-019", "ND-020"
}

excluded = set(
    manifest["excluded_image_relpath"]
)

assert len(excluded) == 3

for rel in excluded:
    rel_path = Path(rel)

    assert not rel_path.is_absolute()
    assert ".." not in rel_path.parts
    assert rel_path.parts[:2] == (
        "train", "images"
    )
    assert (SOURCE / rel_path).is_file()

for _, row in manifest.iterrows():
    for field in [
        "excluded_label_relpath",
        "retained_image_relpath",
        "retained_label_relpath"
    ]:
        rel = Path(row[field])
        assert not rel.is_absolute()
        assert ".." not in rel.parts
        assert (SOURCE / rel).is_file()

if DEST.exists():
    raise RuntimeError(
        "STOP: v1_cleaned already exists. "
        "Do not overwrite the locked research dataset."
    )

PREPARED.mkdir(parents=True, exist_ok=True)
RESTRICTED.mkdir(parents=True, exist_ok=True)

# ==================================================
# 2. BUILD DATASET IN A TEMPORARY DIRECTORY
# ==================================================

STAGING = Path(
    tempfile.mkdtemp(
        prefix="six_class_staging_",
        dir=str(PREPARED)
    )
)

summary = {}
private_mapping = []

try:
    for split in ["train", "valid", "test"]:

        source_images = SOURCE / split / "images"
        source_labels = SOURCE / split / "labels"

        target_images = STAGING / split / "images"
        target_labels = STAGING / split / "labels"

        target_images.mkdir(parents=True)
        target_labels.mkdir(parents=True)

        image_files = sorted(
            p for p in source_images.iterdir()
            if p.is_file()
            and p.suffix.lower() in EXTENSIONS
        )

        counts = Counter()
        positive_images = Counter()
        empty_labels = 0
        copied = 0
        skipped = 0

        for image_file in image_files:

            relative_path = image_file.relative_to(SOURCE)
            label_file = (
                source_labels /
                f"{image_file.stem}.txt"
            )

            if split == "train" and str(
                relative_path
            ) in excluded:
                skipped += 1
                continue

            if not label_file.is_file():
                raise RuntimeError(
                    "Missing label during dataset build."
                )

            boxes, errors = read_yolo_label(
                label_file
            )

            if errors:
                raise RuntimeError(
                    f"Invalid annotation detected: {errors[:3]}"
                )

            selected_boxes = []
            present_classes = set()

            for cid, x, y, w, h in boxes:

                if cid not in CLASS_MAP:
                    continue

                # Strict geometry check for training labels.
                if not (
                    0 <= x-w/2 <= x+w/2 <= 1
                    and 0 <= y-h/2 <= y+h/2 <= 1
                ):
                    raise RuntimeError(
                        "Target bounding box exceeds image bounds."
                    )

                new_id = CLASS_MAP[cid][0]

                selected_boxes.append(
                    (new_id, x, y, w, h)
                )

                counts[new_id] += 1
                present_classes.add(new_id)

            for new_id in present_classes:
                positive_images[new_id] += 1

            copied += 1

            # Pseudonymize filenames in the research copy.
            new_stem = f"{split}_{copied:05d}"
            new_image = (
                target_images /
                f"{new_stem}{image_file.suffix.lower()}"
            )
            new_label = (
                target_labels / f"{new_stem}.txt"
            )

            shutil.copy2(image_file, new_image)

            lines = [
                f"{cid} {x:.10f} {y:.10f} "
                f"{w:.10f} {h:.10f}"
                for cid, x, y, w, h in selected_boxes
            ]

            new_label.write_text(
                "\n".join(lines) + ("\n" if lines else ""),
                encoding="utf-8"
            )

            if not lines:
                empty_labels += 1

            # Keep identifiable source names only in
            # the restricted provenance mapping.
            private_mapping.append({
                "split": split,
                "source_image": str(relative_path),
                "source_label": str(
                    label_file.relative_to(SOURCE)
                ),
                "prepared_image": str(
                    new_image.relative_to(STAGING)
                ),
                "prepared_label": str(
                    new_label.relative_to(STAGING)
                )
            })

        if copied != EXPECTED[split]:
            raise RuntimeError(
                f"{split}: expected {EXPECTED[split]} "
                f"images, found {copied}"
            )

        if split == "train" and skipped != 3:
            raise RuntimeError(
                "Expected to exclude exactly 3 "
                "training images."
            )

        summary[split] = {
            "images": copied,
            "excluded": skipped,
            "empty_target_labels": empty_labels,
            "class_objects": {
                str(i): counts[i] for i in range(6)
            },
            "class_positive_images": {
                str(i): positive_images[i]
                for i in range(6)
            }
        }

    # ==================================================
    # 3. GENERATE DATA.YAML
    # ==================================================

    yaml_text = f"""path: {DEST}
train: train/images
val: valid/images
test: test/images

nc: 6
names:
  0: Caries
  1: Missing Teeth
  2: Root Canal Treatment
  3: Filling
  4: Crown
  5: Unerupted Wisdom Teeth
"""

    (STAGING / "data.yaml").write_text(
        yaml_text, encoding="utf-8"
    )

    (STAGING / "build_summary.json").write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8"
    )

    # Publish the validated derived dataset.
    STAGING.rename(DEST)

except Exception:
    # Remove only this run's incomplete staging files.
    if STAGING.exists():
        shutil.rmtree(STAGING)
    raise

# ==================================================
# 4. SAVE PROVENANCE AND PRINT SUMMARY
# ==================================================

pd.DataFrame(private_mapping).to_csv(
    RESTRICTED / "six_class_v1_file_mapping.csv",
    index=False
)

(QC / "six_class_v1_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("=" * 60)
print("SIX-CLASS DATASET PREPARATION COMPLETED")
print("=" * 60)

for split, result in summary.items():
    print(f"\n{split.upper()}")
    print("Images:", result["images"])
    print("Excluded:", result["excluded"])
    print("Empty target labels:",
          result["empty_target_labels"])

print("\nCLASS MAPPING")
for old_id, (new_id, name) in CLASS_MAP.items():
    print(f"{old_id} -> {new_id}: {name}")

print("\nDATASET LOCATION:")
print(DEST)

print("\nDATA.YAML:")
print(DEST / "data.yaml")

print("\nOriginal files modified: NO")
print("Dataset preparation: COMPLETE")


In [ ]:

from pathlib import Path
from collections import Counter
from PIL import Image
import pandas as pd
import numpy as np
import json
import random

# =====================================================
# FINAL SIX-CLASS DATASET INTEGRITY CHECK
# =====================================================

ROOT = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026"
)

DATA = ROOT / "01_Datasets/Prepared_6_Classes/v1_cleaned"
QC = ROOT / "01_Datasets/Dataset_QC"

EXPECTED = {
    "train": 1065,
    "valid": 303,
    "test": 154
}

CLASSES = [
    "Caries",
    "Missing Teeth",
    "Root Canal Treatment",
    "Filling",
    "Crown",
    "Unerupted Wisdom Teeth"
]

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
rng = random.Random(42)

issues = []
summary = {}
class_rows = []

print("=" * 65)
print("FINAL DATASET INTEGRITY CHECK")
print("=" * 65)

assert DATA.exists(), "Prepared dataset not found."
assert (DATA / "data.yaml").exists(), "Missing data.yaml."

# =====================================================
# 1. CHECK ALL THREE DATASET SPLITS
# =====================================================

for split in ["train", "valid", "test"]:

    image_dir = DATA / split / "images"
    label_dir = DATA / split / "labels"

    assert image_dir.exists(), f"Missing {split}/images"
    assert label_dir.exists(), f"Missing {split}/labels"

    images = sorted(
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXT
    )

    labels = sorted(label_dir.glob("*.txt"))

    image_stems = [p.stem for p in images]
    label_stems = [p.stem for p in labels]

    image_set = set(image_stems)
    label_set = set(label_stems)

    if len(image_set) != len(images):
        issues.append(f"{split}: duplicate image stems")

    missing_labels = image_set - label_set
    orphan_labels = label_set - image_set

    if missing_labels:
        issues.append(
            f"{split}: {len(missing_labels)} missing labels"
        )

    if orphan_labels:
        issues.append(
            f"{split}: {len(orphan_labels)} orphan labels"
        )

    if len(images) != EXPECTED[split]:
        issues.append(
            f"{split}: unexpected image count {len(images)}"
        )

    counts = Counter()
    positive_images = Counter()
    empty_count = 0
    invalid_count = 0

    for label_file in labels:

        present = set()

        lines = label_file.read_text(
            encoding="utf-8",
            errors="replace"
        ).splitlines()

        if not any(line.strip() for line in lines):
            empty_count += 1

        for line in lines:

            if not line.strip():
                continue

            try:
                parts = line.split()

                if len(parts) != 5:
                    raise ValueError("Invalid YOLO format")

                raw_id = float(parts[0])
                cid = int(raw_id)
                x, y, w, h = map(float, parts[1:])

                if raw_id != cid or not 0 <= cid < 6:
                    raise ValueError("Invalid class ID")

                values = [x, y, w, h]

                if not all(np.isfinite(v) for v in values):
                    raise ValueError("Non-finite value")

                if not (
                    0 <= x <= 1 and 0 <= y <= 1
                    and 0 < w <= 1 and 0 < h <= 1
                ):
                    raise ValueError("Invalid coordinates")

                tolerance = 0.00001

                if (
                    x - w / 2 < -tolerance
                    or x + w / 2 > 1 + tolerance
                    or y - h / 2 < -tolerance
                    or y + h / 2 > 1 + tolerance
                ):
                    raise ValueError("Box outside image")

                counts[cid] += 1
                present.add(cid)

            except (ValueError, OverflowError):
                invalid_count += 1

        for cid in present:
            positive_images[cid] += 1

    if invalid_count:
        issues.append(
            f"{split}: {invalid_count} invalid annotations"
        )

    # Test image readability using a reproducible sample.
    sample = rng.sample(images, min(6, len(images)))
    unreadable = 0

    for image_path in sample:
        try:
            with Image.open(image_path) as im:
                im.verify()
        except Exception:
            unreadable += 1

    if unreadable:
        issues.append(
            f"{split}: {unreadable} unreadable sampled images"
        )

    summary[split] = {
        "images": len(images),
        "labels": len(labels),
        "matched": len(image_set & label_set),
        "empty_labels": empty_count,
        "annotations": sum(counts.values()),
        "invalid_annotations": invalid_count,
        "sampled_images_checked": len(sample),
        "unreadable_sampled_images": unreadable
    }

    for cid, name in enumerate(CLASSES):
        class_rows.append({
            "split": split,
            "class_id": cid,
            "class": name,
            "objects": counts[cid],
            "positive_images": positive_images[cid]
        })

# =====================================================
# 2. VERIFY LEAKAGE EXCLUSION MANIFEST
# =====================================================

manifest_path = (
    QC / "Leakage_Resolution_RESTRICTED"
    / "training_exclusion_manifest.csv"
)

mapping_path = (
    QC / "RESTRICTED_provenance"
    / "six_class_v1_file_mapping.csv"
)

assert manifest_path.exists(), "Exclusion manifest missing."
assert mapping_path.exists(), "Provenance mapping missing."

manifest = pd.read_csv(manifest_path, dtype=str).fillna("")
mapping = pd.read_csv(mapping_path, dtype=str).fillna("")

excluded = set(manifest["excluded_image_relpath"])
included_sources = set(mapping["source_image"])

if len(excluded) != 3:
    issues.append("Expected exactly three excluded images")

if excluded & included_sources:
    issues.append("Excluded source images found in prepared data")

if len(mapping) != sum(
    summary[s]["images"] for s in EXPECTED
):
    issues.append("Provenance mapping count mismatch")

if mapping["prepared_image"].duplicated().any():
    issues.append("Duplicate prepared-image provenance entries")

# =====================================================
# 3. PRINT DATASET SUMMARY
# =====================================================

print("\n===== SPLIT SUMMARY =====")
display(pd.DataFrame(summary).T)

class_df = pd.DataFrame(class_rows)

print("\n===== CLASS OBJECT COUNTS =====")
display(
    class_df.pivot(
        index="class",
        columns="split",
        values="objects"
    )[["train", "valid", "test"]]
)

print("\n===== POSITIVE IMAGES PER CLASS =====")
display(
    class_df.pivot(
        index="class",
        columns="split",
        values="positive_images"
    )[["train", "valid", "test"]]
)

for name in CLASSES:
    for split in EXPECTED:
        row = class_df[
            (class_df["class"] == name)
            & (class_df["split"] == split)
        ]

        if int(row["positive_images"].iloc[0]) == 0:
            issues.append(
                f"No positive images for {name} in {split}"
            )

# =====================================================
# 4. SAVE AUDIT REPORTS
# =====================================================

QC.mkdir(parents=True, exist_ok=True)

class_df.to_csv(
    QC / "final_six_class_distribution.csv",
    index=False
)

audit_report = {
    "dataset": "v1_cleaned",
    "summary": summary,
    "excluded_training_images": len(excluded),
    "issues": issues,
    "status": "PASS" if not issues else "FAIL"
}

(QC / "final_dataset_integrity_report.json").write_text(
    json.dumps(audit_report, indent=2),
    encoding="utf-8"
)

# =====================================================
# 5. FINAL STATUS
# =====================================================

print("\n===== FINAL QUALITY CONTROL STATUS =====")

print("Total images:", sum(
    summary[s]["images"] for s in EXPECTED
))

print("Excluded duplicate training images:", len(excluded))
print("Integrity issues:", len(issues))

if issues:
    print("\nDATASET QC: FAILED")
    for issue in issues[:20]:
        print("-", issue)
    raise RuntimeError(
        "Dataset integrity check failed. "
        "Resolve issues before training."
    )

print("\nDATASET QC: PASSED")
print("Six-class dataset structure: VALID")
print("Annotation format: VALID")
print("Known training exclusions: VERIFIED")
print("Original dataset modified: NO")
print("Ready for visual annotation review: YES")


### 03 — Visual annotation audit (researcher required)
Display random examples from the **training split only** to inspect bounding-box placement, image characteristics, and objects omitted from the six target classes. Do not use test images to decide hyperparameters. Radiographs may contain patient-identifying text burned directly into pixels; check before sharing images. **Do not share screenshots with visible personal information.**


In [ ]:

from pathlib import Path
from PIL import Image, ImageOps, ImageDraw, ImageFont
from IPython.display import display
import pandas as pd
import random

# ==================================================
# VISUAL ANNOTATION QUALITY CONTROL
# ==================================================

ROOT = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026"
)

DATA = ROOT / "01_Datasets/Prepared_6_Classes/v1_cleaned"
QC_DIR = ROOT / "01_Datasets/Dataset_QC/Visual_QC_RESTRICTED"
QC_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = [
    "Caries",
    "Missing Teeth",
    "Root Canal Treatment",
    "Filling",
    "Crown",
    "Unerupted Wisdom Teeth"
]

SPLITS = ["train", "valid", "test"]
rng = random.Random(42)

try:
    FONT = ImageFont.truetype(
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 17
    )
    TITLE_FONT = ImageFont.truetype(
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 22
    )
except OSError:
    FONT = ImageFont.load_default()
    TITLE_FONT = FONT

def load_boxes(label_path):
    boxes = []
    for line in label_path.read_text(
        encoding="utf-8"
    ).splitlines():
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError("Invalid detection label format")
        cid = int(parts[0])
        coords = list(map(float, parts[1:]))
        boxes.append((cid, *coords))
    return boxes

def draw_annotations(image_path, boxes, focus_class):
    with Image.open(image_path) as source:
        im = ImageOps.exif_transpose(source).convert("RGB")

    width, height = im.size
    draw = ImageDraw.Draw(im)

    for cid, x, y, w, h in boxes:
        x1 = max(0, round((x - w / 2) * width))
        y1 = max(0, round((y - h / 2) * height))
        x2 = min(width - 1, round((x + w / 2) * width))
        y2 = min(height - 1, round((y + h / 2) * height))

        color = "yellow" if cid == focus_class else "lime"

        draw.rectangle(
            [x1, y1, x2, y2],
            outline=color,
            width=3
        )

        label = f"{cid}: {CLASSES[cid]}"
        text_y = max(0, y1 - 20)

        draw.text(
            (x1 + 1, text_y),
            label,
            fill=color,
            stroke_width=2,
            stroke_fill="black",
            font=FONT
        )

    return im

review_records = []
sheet_paths = []

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

for split in SPLITS:
    image_dir = DATA / split / "images"
    label_dir = DATA / split / "labels"

    records = []

    for path in sorted(image_dir.iterdir()):
        if path.suffix.lower() not in IMAGE_EXT:
            continue

        label_path = label_dir / f"{path.stem}.txt"
        assert label_path.exists()

        boxes = load_boxes(label_path)

        records.append({
            "image": path,
            "boxes": boxes,
            "classes": {b[0] for b in boxes}
        })

    selected = []
    used_paths = set()

    # One image for each research class
    for cid, class_name in enumerate(CLASSES):
        candidates = [
            r for r in records
            if cid in r["classes"]
            and r["image"] not in used_paths
        ]

        if not candidates:
            raise RuntimeError(
                f"No available unique sample: {split} / {class_name}"
            )

        item = rng.choice(candidates)
        used_paths.add(item["image"])

        selected.append((class_name, cid, item))

    # One image without annotated target classes
    empty_candidates = [
        r for r in records
        if len(r["boxes"]) == 0
        and r["image"] not in used_paths
    ]

    if not empty_candidates:
        raise RuntimeError(f"No empty-label image in {split}")

    selected.append((
        "NO ANNOTATED TARGET",
        None,
        rng.choice(empty_candidates)
    ))

    # Create contact sheet
    TILE_W, TILE_H = 780, 570
    COLUMNS = 2
    ROWS = 4

    sheet = Image.new(
        "RGB",
        (COLUMNS * TILE_W, ROWS * TILE_H),
        "white"
    )

    canvas = ImageDraw.Draw(sheet)

    for index, (class_name, cid, item) in enumerate(selected):
        col = index % COLUMNS
        row = index // COLUMNS

        x0 = col * TILE_W
        y0 = row * TILE_H

        canvas.text(
            (x0 + 15, y0 + 12),
            f"{split.upper()} | {class_name}",
            fill="black",
            font=TITLE_FONT
        )

        annotated = draw_annotations(
            item["image"],
            item["boxes"],
            cid
        )

        annotated.thumbnail((740, 515))

        sheet.paste(
            annotated,
            (
                x0 + (TILE_W - annotated.width) // 2,
                y0 + 47
            )
        )

        review_records.append({
            "split": split,
            "sample_id": item["image"].stem,
            "focus_class": class_name,
            "total_boxes": len(item["boxes"])
        })

    output = QC_DIR / f"visual_qc_{split}.png"
    sheet.save(output)
    sheet_paths.append(output)

# Save a reproducible sample record
pd.DataFrame(review_records).to_csv(
    QC_DIR / "visual_qc_sample_index.csv",
    index=False
)

print("=" * 60)
print("VISUAL ANNOTATION QC")
print("=" * 60)

print("Samples generated:", len(review_records))
print("Contact sheets:", len(sheet_paths))

for path in sheet_paths:
    print(path.name)

print("\nOutput folder:", QC_DIR)
print("Original dataset modified: NO")

print("\nTRAIN REVIEW SHEET:")
display(Image.open(sheet_paths[0]))


In [ ]:

# ============================================
# STEP 1 - TRAINING ENVIRONMENT CHECK
# ============================================

import torch
import platform
import importlib.util

print("=" * 55)
print("TRAINING ENVIRONMENT CHECK")
print("=" * 55)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. Enable GPU in Colab Runtime."
    )

print("GPU:", torch.cuda.get_device_name(0))

gpu_memory = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print("GPU Memory:", round(gpu_memory, 2), "GB")

if importlib.util.find_spec("ultralytics") is None:
    print("Ultralytics: NOT INSTALLED")
else:
    import ultralytics
    print("Ultralytics:", ultralytics.__version__)

print("=" * 55)
print("ENVIRONMENT CHECK: PASSED")


**Freeze Dataset**

In [ ]:

# =====================================================
# STEP 2 - FREEZE RESEARCH DATASET
# =====================================================

from pathlib import Path
from hashlib import sha256
from tqdm.auto import tqdm
import pandas as pd
import json
import io

ROOT = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026"
)

DATA = ROOT / "01_Datasets/Prepared_6_Classes/v1_cleaned"
QC = ROOT / "01_Datasets/Dataset_QC"

QC.mkdir(parents=True, exist_ok=True)

MANIFEST = QC / "frozen_dataset_v1_sha256.csv"
SUMMARY = QC / "frozen_dataset_v1_summary.json"

EXPECTED = {
    "train": 1065,
    "valid": 303,
    "test": 154
}

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

def file_hash(path):
    h = sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

files_to_hash = []

for split, expected_count in EXPECTED.items():
    image_dir = DATA / split / "images"
    label_dir = DATA / split / "labels"

    images = sorted(
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXT
    )

    labels = sorted(label_dir.glob("*.txt"))

    assert len(images) == expected_count
    assert len(labels) == expected_count

    assert {p.stem for p in images} == {
        p.stem for p in labels
    }

    files_to_hash.extend(images)
    files_to_hash.extend(labels)

for filename in ["data.yaml", "build_summary.json"]:
    path = DATA / filename
    assert path.is_file(), f"Missing: {filename}"
    files_to_hash.append(path)

records = []

print("Calculating SHA256 hashes...")

for path in tqdm(files_to_hash, desc="Freezing dataset"):
    records.append({
        "relative_path": path.relative_to(DATA).as_posix(),
        "sha256": file_hash(path)
    })

df = pd.DataFrame(records).sort_values(
    "relative_path"
).reset_index(drop=True)

# Generate a stable fingerprint of the manifest.
csv_text = df.to_csv(index=False, lineterminator="\n")
fingerprint = sha256(csv_text.encode("utf-8")).hexdigest()

# Do not silently overwrite an existing baseline.
if MANIFEST.exists():
    existing = MANIFEST.read_text(encoding="utf-8")
    if existing != csv_text:
        raise RuntimeError(
            "STOP: Dataset contents differ from "
            "the previously frozen manifest."
        )
    print("Existing frozen manifest verified.")
else:
    MANIFEST.write_text(csv_text, encoding="utf-8")
    print("New frozen manifest created.")

result = {
    "dataset_version": "v1_cleaned",
    "train_images": 1065,
    "valid_images": 303,
    "test_images": 154,
    "total_images": 1522,
    "hashed_files": len(df),
    "manifest_sha256": fingerprint
}

if SUMMARY.exists():
    previous = json.loads(SUMMARY.read_text(encoding="utf-8"))
    if previous != result:
        raise RuntimeError("Frozen summary mismatch.")
else:
    SUMMARY.write_text(
        json.dumps(result, indent=2),
        encoding="utf-8"
    )

print("\n" + "=" * 55)
print("DATASET FREEZE SUMMARY")
print("=" * 55)

for key, value in result.items():
    print(f"{key}: {value}")

print("\nDATASET FREEZE: PASSED")
print("Original dataset modified: NO")
print("Ready for local training copy: YES")


In [ ]:

# =====================================================
# STEP 3 - COPY AND VERIFY LOCAL TRAINING DATASET
# =====================================================

from pathlib import Path
from hashlib import sha256
from tqdm.auto import tqdm
import pandas as pd
import shutil
import yaml

ROOT = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026"
)

DRIVE_DATA = (
    ROOT / "01_Datasets/Prepared_6_Classes/v1_cleaned"
)

MANIFEST = (
    ROOT / "01_Datasets/Dataset_QC"
    / "frozen_dataset_v1_sha256.csv"
)

LOCAL_DATA = Path("/content/dentalopg_v1_cleaned")
LOCAL_YAML = Path("/content/dentalopg_v1_local.yaml")

assert DRIVE_DATA.is_dir(), "Drive dataset not found."
assert MANIFEST.is_file(), "Frozen manifest not found."

# 1. Copy without changing the frozen source.
if not LOCAL_DATA.exists():
    print("Copying dataset from Google Drive...")
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)
else:
    print("Local dataset exists. Verifying existing copy.")

# 2. Verify every copied file against frozen SHA256.
manifest = pd.read_csv(MANIFEST, dtype=str)

expected_paths = set(manifest["relative_path"])
actual_paths = {
    p.relative_to(LOCAL_DATA).as_posix()
    for p in LOCAL_DATA.rglob("*")
    if p.is_file()
}

assert expected_paths == actual_paths, (
    "Local file list does not match frozen manifest."
)

def calculate_sha256(path):
    h = sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()

mismatches = []

for row in tqdm(
    manifest.itertuples(index=False),
    total=len(manifest),
    desc="Verifying local dataset"
):
    path = LOCAL_DATA / row.relative_path
    if calculate_sha256(path) != row.sha256:
        mismatches.append(row.relative_path)

if mismatches:
    raise RuntimeError(
        f"Local copy failed integrity verification: "
        f"{len(mismatches)} mismatched files."
    )

# 3. Create a separate Colab-local data.yaml.
# The frozen data.yaml remains unchanged.
config = yaml.safe_load(
    (LOCAL_DATA / "data.yaml").read_text(
        encoding="utf-8"
    )
)

config["path"] = str(LOCAL_DATA)
config["train"] = "train/images"
config["val"] = "valid/images"
config["test"] = "test/images"

assert len(config["names"]) == 6

LOCAL_YAML.write_text(
    yaml.safe_dump(
        config,
        sort_keys=False,
        allow_unicode=True
    ),
    encoding="utf-8"
)

# 4. Confirm dataset split sizes.
expected = {
    "train": 1065,
    "valid": 303,
    "test": 154
}

print("\n===== LOCAL DATASET COUNTS =====")

for split, count in expected.items():
    image_dir = LOCAL_DATA / split / "images"
    label_dir = LOCAL_DATA / split / "labels"

    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in
        {".jpg", ".jpeg", ".png", ".bmp"}
    ]
    labels = list(label_dir.glob("*.txt"))

    assert len(images) == count
    assert len(labels) == count

    print(
        f"{split.upper()}: "
        f"{len(images)} images, "
        f"{len(labels)} labels"
    )

print("\n===== FINAL STATUS =====")
print("Verified files:", len(manifest))
print("SHA256 mismatches:", len(mismatches))
print("Classes:", len(config["names"]))
print("Local YAML:", LOCAL_YAML)
print("Original dataset modified: NO")
print("LOCAL DATASET: READY")


## 07–09 — Train the three AI models
**Prespecified protocol:** Use the same training/validation images, random seed, maximum epoch count, and input resolution. Record the actual batch size available to each architecture. Training one model per session may reduce interruptions on shared Colab GPUs. Set `MODEL_TO_TRAIN`, change `RUN_TRAIN=True`, and execute the cell. Checkpoints and logs are written directly to Drive. **The held-out test split must not influence model selection or hyperparameter tuning.**

| Model | Pretrained weights | Initial batch size |
|---|---|---:|
| YOLOv8s | `yolov8s.pt` | 8 |
| YOLO11s | `yolo11s.pt` | 8 |
| RT-DETR-L | `rtdetr-l.pt` | 2 (reduce to 1 or use a larger GPU if required) |

**RT-DETR note:** Ultralytics recommends `deterministic=False` for RT-DETR because PyTorch's `grid_sample` operation may not be deterministic. Record that exception. Different batch sizes resulting from GPU constraints do **not** constitute identical computational budgets; report the actual training settings for each model.


In [ ]:

# ==========================================================
# STEP 4 - YOLOv8s MODEL TRAINING
# ==========================================================

from ultralytics import YOLO
from pathlib import Path
import torch
import ultralytics
import json
import yaml

# ----------------------------------------------------------
# 1. Research paths
# ----------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026"
)

DATA_YAML = Path("/content/dentalopg_v1_local.yaml")

PROJECT = ROOT / "03_Trained_Models/YOLOv8s"
RUN_NAME = "YOLOv8s_seed42"
RUN_DIR = PROJECT / RUN_NAME

PROTOCOL = ROOT / "00_Research_Protocol"

PROJECT.mkdir(parents=True, exist_ok=True)
PROTOCOL.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# 2. Pre-training checks
# ----------------------------------------------------------

assert torch.cuda.is_available(), "GPU is unavailable."
assert DATA_YAML.exists(), "Local dataset YAML not found."

config = yaml.safe_load(
    DATA_YAML.read_text(encoding="utf-8")
)

assert len(config["names"]) == 6, "Expected 6 classes."

assert Path(config["path"]).is_dir(), (
    "Local dataset path does not exist."
)

if RUN_DIR.exists():
    raise RuntimeError(
        "STOP: A training directory already exists. "
        "Check for an existing checkpoint before rerunning."
    )

# ----------------------------------------------------------
# 3. Training configuration
# ----------------------------------------------------------

TRAIN_CONFIG = {
    "data": str(DATA_YAML),
    "epochs": 100,
    "imgsz": 640,
    "batch": 8,
    "patience": 25,
    "seed": 42,
    "deterministic": True,
    "device": 0,
    "workers": 2,
    "optimizer": "auto",
    "amp": True,
    "pretrained": True,
    "save": True,
    "save_period": 10,
    "plots": True,
    "val": True,
    "cache": False,
    "project": str(PROJECT),
    "name": RUN_NAME,
    "exist_ok": False,
    "verbose": True
}

# ----------------------------------------------------------
# 4. Save experiment metadata
# ----------------------------------------------------------

freeze_summary = (
    ROOT / "01_Datasets/Dataset_QC"
    / "frozen_dataset_v1_summary.json"
)

assert freeze_summary.exists(), (
    "Frozen dataset summary is missing."
)

dataset_metadata = json.loads(
    freeze_summary.read_text(encoding="utf-8")
)

metadata = {
    "model": "YOLOv8s",
    "weights": "yolov8s.pt",
    "dataset_version": "v1_cleaned",
    "dataset_fingerprint": dataset_metadata["manifest_sha256"],
    "dataset_counts": {
        "train": 1065,
        "valid": 303,
        "test": 154
    },
    "training_configuration": TRAIN_CONFIG,
    "pytorch_version": torch.__version__,
    "ultralytics_version": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0)
}

metadata_path = PROTOCOL / "YOLOv8s_training_config.json"

metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

print("=" * 60)
print("YOLOv8s TRAINING CONFIGURATION")
print("=" * 60)

print("Model: YOLOv8s")
print("Dataset:", DATA_YAML)
print("Classes:", len(config["names"]))
print("Epochs:", TRAIN_CONFIG["epochs"])
print("Batch:", TRAIN_CONFIG["batch"])
print("Image size:", TRAIN_CONFIG["imgsz"])
print("GPU:", torch.cuda.get_device_name(0))
print("Results:", RUN_DIR)

# ----------------------------------------------------------
# 5. Load pretrained model
# ----------------------------------------------------------

print("\nLoading pretrained YOLOv8s...")

model = YOLO("yolov8s.pt")

# ----------------------------------------------------------
# 6. Start training
# ----------------------------------------------------------

print("\n" + "=" * 60)
print("STARTING YOLOv8s TRAINING")
print("=" * 60)

results = model.train(**TRAIN_CONFIG)

# ----------------------------------------------------------
# 7. Verify saved outputs
# ----------------------------------------------------------

BEST = RUN_DIR / "weights/best.pt"
LAST = RUN_DIR / "weights/last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

print("\n" + "=" * 60)
print("YOLOv8s TRAINING FINISHED")
print("=" * 60)

print("Best weights:", BEST.exists())
print("Last weights:", LAST.exists())
print("Training log:", RESULTS_CSV.exists())
print("Results directory:", RUN_DIR)

if not BEST.exists():
    raise RuntimeError("Best model checkpoint not found.")

print("\nTRAINING STATUS: COMPLETED")


In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

RUN = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026/"
    "03_Trained_Models/YOLOv8s/YOLOv8s_seed42"
)

for filename in [
    "weights/best.pt",
    "weights/last.pt",
    "results.csv"
]:
    path = RUN / filename
    print(filename, "FOUND" if path.is_file() else "MISSING")

print("\nYOLOv8s checkpoint verification complete.")


In [ ]:

# STEP 5A - PREPARE YOLO11s ENVIRONMENT

from pathlib import Path
from hashlib import sha256
from google.colab import drive
from tqdm.auto import tqdm
import pandas as pd
import torch
import shutil
import yaml

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")
DRIVE_DATA = ROOT / "01_Datasets/Prepared_6_Classes/v1_cleaned"
LOCAL_DATA = Path("/content/dentalopg_v1_cleaned")
LOCAL_YAML = Path("/content/dentalopg_v1_local.yaml")
MANIFEST = ROOT / "01_Datasets/Dataset_QC/frozen_dataset_v1_sha256.csv"

assert torch.cuda.is_available(), "GPU is not enabled."
assert DRIVE_DATA.is_dir(), "Drive dataset missing."
assert MANIFEST.is_file(), "Dataset manifest missing."

if not LOCAL_DATA.exists():
    print("Restoring dataset to Colab...")
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)

def hash_file(path):
    h = sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = pd.read_csv(MANIFEST, dtype=str)
errors = []

for row in tqdm(
    manifest.itertuples(index=False),
    total=len(manifest),
    desc="Verifying local dataset"
):
    path = LOCAL_DATA / row.relative_path
    if not path.is_file() or hash_file(path) != row.sha256:
        errors.append(row.relative_path)

assert not errors, f"Dataset verification failed: {len(errors)} files"

config = yaml.safe_load(
    (LOCAL_DATA / "data.yaml").read_text(encoding="utf-8")
)
config["path"] = str(LOCAL_DATA)
config["train"] = "train/images"
config["val"] = "valid/images"
config["test"] = "test/images"

LOCAL_YAML.write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8"
)

assert len(config["names"]) == 6

RUN_DIR = ROOT / "03_Trained_Models/YOLO11s/YOLO11s_seed42"

print("\n===== YOLO11s PRE-FLIGHT CHECK =====")
print("GPU:", torch.cuda.get_device_name(0))
print("Verified files:", len(manifest))
print("SHA256 mismatches:", len(errors))
print("Classes:", len(config["names"]))
print("Dataset:", LOCAL_YAML)
print("YOLOv8s saved:",
      (ROOT / "03_Trained_Models/YOLOv8s/YOLOv8s_seed42/weights/best.pt").exists())
print("YOLO11s run already exists:", RUN_DIR.exists())
print("STATUS: READY FOR TRAINING" if not RUN_DIR.exists()
      else "STATUS: CHECK EXISTING RUN BEFORE TRAINING")



In [ ]:

# STEP 5B - TRAIN YOLO11s

from ultralytics import YOLO
import ultralytics
import json

PROJECT = ROOT / "03_Trained_Models/YOLO11s"
RUN_NAME = "YOLO11s_seed42"
RUN_DIR = PROJECT / RUN_NAME

if RUN_DIR.exists():
    raise RuntimeError(
        "Existing YOLO11s run detected. "
        "Do not overwrite it. Check checkpoints first."
    )

TRAIN_CONFIG = {
    "data": str(LOCAL_YAML),
    "epochs": 100,
    "imgsz": 640,
    "batch": 8,
    "patience": 25,
    "seed": 42,
    "deterministic": True,
    "device": 0,
    "workers": 2,
    "optimizer": "auto",
    "amp": True,
    "pretrained": True,
    "save": True,
    "save_period": 10,
    "plots": True,
    "val": True,
    "cache": False,
    "project": str(PROJECT),
    "name": RUN_NAME,
    "exist_ok": False
}

# Save experiment metadata
PROTOCOL = ROOT / "00_Research_Protocol"
PROTOCOL.mkdir(parents=True, exist_ok=True)

freeze = json.loads(
    (ROOT / "01_Datasets/Dataset_QC/frozen_dataset_v1_summary.json")
    .read_text(encoding="utf-8")
)

metadata = {
    "model": "YOLO11s",
    "pretrained_weights": "yolo11s.pt",
    "dataset_version": "v1_cleaned",
    "dataset_fingerprint": freeze["manifest_sha256"],
    "training_configuration": TRAIN_CONFIG,
    "pytorch_version": torch.__version__,
    "ultralytics_version": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0)
}

(PROTOCOL / "YOLO11s_training_config.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

print("Starting YOLO11s training...")

model = YOLO("yolo11s.pt")
results = model.train(**TRAIN_CONFIG)

print("\n===== YOLO11s TRAINING COMPLETED =====")
print("Best weights:", (RUN_DIR / "weights/best.pt").exists())
print("Last weights:", (RUN_DIR / "weights/last.pt").exists())
print("Results CSV:", (RUN_DIR / "results.csv").exists())


In [ ]:

from pathlib import Path
import pandas as pd

RUN = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026/"
    "03_Trained_Models/YOLO11s/YOLO11s_seed42"
)

print("===== YOLO11s CHECKPOINT CHECK =====")

for file in [
    "weights/best.pt",
    "weights/last.pt",
    "results.csv"
]:
    path = RUN / file
    print(
        file,
        "FOUND" if path.is_file() else "MISSING"
    )

csv_path = RUN / "results.csv"

if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    print("\nLast completed epoch:")
    print(df["epoch"].iloc[-1])

    print("\nLatest training results:")
    display(df.tail(5))

print("\nCHECKPOINT INSPECTION COMPLETED")


In [ ]:

# ==================================================
# YOLO11s CHECKPOINT RECOVERY DIAGNOSIS
# READ-ONLY: NO TRAINING OR FILE MODIFICATION
# ==================================================

from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import re

RUN = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026/"
    "03_Trained_Models/YOLO11s/YOLO11s_seed42"
)

WEIGHTS = RUN / "weights"
CSV = RUN / "results.csv"

print("=" * 60)
print("YOLO11s TRAINING RECOVERY DIAGNOSIS")
print("=" * 60)

# 1. Check training history
assert CSV.exists(), "Training results.csv not found."

df = pd.read_csv(CSV)
df.columns = df.columns.str.strip()

epochs = pd.to_numeric(df["epoch"], errors="coerce").dropna()
assert not epochs.empty, "No completed epochs found."

last_epoch = int(epochs.iloc[-1])

print("\n===== TRAINING HISTORY =====")
print("Last recorded epoch:", last_epoch)
print("Total logged epochs:", len(epochs))

metric = "metrics/mAP50-95(B)"

if metric in df.columns:
    values = pd.to_numeric(df[metric], errors="coerce")
    if values.notna().any():
        idx = values.idxmax()
        print("Best logged mAP50-95:", round(float(values.loc[idx]), 5))
        print("Best metric epoch:", int(df.loc[idx, "epoch"]))

# 2. Inspect checkpoint metadata
print("\n===== CHECKPOINT STATUS =====")

def inspect_checkpoint(path):
    if not path.exists():
        print(path.name, ": MISSING")
        return

    try:
        model = YOLO(str(path))
        ckpt = model.ckpt or {}

        epoch = ckpt.get("epoch", None)
        optimizer = ckpt.get("optimizer", None)

        print(f"\nFile: {path.name}")
        print("Size MB:", round(path.stat().st_size / 1048576, 2))
        print("Stored epoch:", epoch)
        print("Optimizer available:", optimizer is not None)
        print("Resumable:", (
            isinstance(epoch, int)
            and epoch >= 0
            and optimizer is not None
        ))

        del model

    except Exception as e:
        print(path.name, "LOAD ERROR:", str(e))

inspect_checkpoint(WEIGHTS / "last.pt")
inspect_checkpoint(WEIGHTS / "best.pt")

# 3. Locate periodic checkpoints
periodic = []

for p in WEIGHTS.glob("epoch*.pt"):
    match = re.fullmatch(r"epoch(\d+)\.pt", p.name)
    if match:
        periodic.append((int(match.group(1)), p))

periodic.sort(key=lambda x: x[0])

print("\n===== PERIODIC CHECKPOINTS =====")
print("Available periodic checkpoints:",
      [n for n, _ in periodic])

for _, path in periodic[-3:]:
    inspect_checkpoint(path)

print("\n===== DIAGNOSIS COMPLETE =====")
print("No training was started.")
print("No files were modified.")


In [ ]:

from pathlib import Path
import pandas as pd
import yaml

RUN = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026/"
    "03_Trained_Models/YOLO11s/YOLO11s_seed42"
)

df = pd.read_csv(RUN / "results.csv")
df.columns = df.columns.str.strip()

args = yaml.safe_load(
    (RUN / "args.yaml").read_text(encoding="utf-8")
)

metric = "metrics/mAP50-95(B)"
best_index = df[metric].idxmax()

best_epoch = int(df.loc[best_index, "epoch"])
last_epoch = int(df["epoch"].iloc[-1])
patience = int(args["patience"])
max_epochs = int(args["epochs"])

print("===== EARLY STOPPING VERIFICATION =====")
print("Best epoch:", best_epoch)
print("Last epoch:", last_epoch)
print("Patience:", patience)
print("Maximum epochs:", max_epochs)
print("Epochs without improvement:",
      last_epoch - best_epoch)
print("Best checkpoint exists:",
      (RUN / "weights/best.pt").is_file())

if (
    last_epoch < max_epochs
    and last_epoch - best_epoch >= patience
):
    print("\nSTATUS: CONSISTENT WITH EARLY STOPPING")
    print("No resume recommended.")
else:
    print("\nSTATUS: REQUIRES FURTHER INVESTIGATION")


In [ ]:

# =====================================================
# STEP 6A - RT-DETR-L PRE-FLIGHT CHECK
# =====================================================

from pathlib import Path
from hashlib import sha256
from google.colab import drive
from ultralytics import RTDETR
from tqdm.auto import tqdm
import torch
import ultralytics
import pandas as pd
import shutil
import yaml

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")

DRIVE_DATA = ROOT / "01_Datasets/Prepared_6_Classes/v1_cleaned"
LOCAL_DATA = Path("/content/dentalopg_v1_cleaned")
LOCAL_YAML = Path("/content/dentalopg_v1_local.yaml")

MANIFEST = ROOT / "01_Datasets/Dataset_QC/frozen_dataset_v1_sha256.csv"

RUN_DIR = ROOT / "03_Trained_Models/RT_DETR/RTDETR_L_seed42"

assert torch.cuda.is_available(), "GPU not detected."
assert DRIVE_DATA.is_dir(), "Frozen dataset missing."
assert MANIFEST.is_file(), "Frozen manifest missing."

if not LOCAL_DATA.exists():
    print("Restoring frozen dataset...")
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)

def hash_file(path):
    h = sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = pd.read_csv(MANIFEST, dtype=str)

mismatches = []

for row in tqdm(
    manifest.itertuples(index=False),
    total=len(manifest),
    desc="Verifying dataset"
):
    p = LOCAL_DATA / row.relative_path
    if not p.is_file() or hash_file(p) != row.sha256:
        mismatches.append(row.relative_path)

assert not mismatches, (
    f"Dataset verification failed: {len(mismatches)}"
)

config = yaml.safe_load(
    (LOCAL_DATA / "data.yaml").read_text(encoding="utf-8")
)

config["path"] = str(LOCAL_DATA)
config["train"] = "train/images"
config["val"] = "valid/images"
config["test"] = "test/images"

LOCAL_YAML.write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8"
)

assert len(config["names"]) == 6

previous_models = {
    "YOLOv8s": ROOT / "03_Trained_Models/YOLOv8s/YOLOv8s_seed42/weights/best.pt",
    "YOLO11s": ROOT / "03_Trained_Models/YOLO11s/YOLO11s_seed42/weights/best.pt"
}

print("\n===== RT-DETR-L PRE-FLIGHT CHECK =====")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU Memory (GB):",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("Ultralytics:", ultralytics.__version__)
print("Verified files:", len(manifest))
print("SHA256 mismatches:", len(mismatches))
print("Classes:", len(config["names"]))
print("Dataset:", LOCAL_YAML)

for name, path in previous_models.items():
    print(f"{name} best.pt:", "FOUND" if path.is_file() else "MISSING")

print("RT-DETR-L existing run:", RUN_DIR.exists())

if RUN_DIR.exists():
    print("STATUS: STOP - Inspect existing run")
elif all(p.is_file() for p in previous_models.values()):
    print("STATUS: READY FOR RT-DETR-L TRAINING")
else:
    print("STATUS: CHECK PREVIOUS MODEL FILES")


In [ ]:

# ==========================================================
# STEP 6B - RT-DETR-L TRAINING
# ==========================================================

from pathlib import Path
from ultralytics import RTDETR
import torch
import ultralytics
import yaml
import json
import pandas as pd

# ----------------------------------------------------------
# 1. Research paths
# ----------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026"
)

DATA_YAML = Path("/content/dentalopg_v1_local.yaml")

PROJECT = ROOT / "03_Trained_Models/RT_DETR"
RUN_NAME = "RTDETR_L_seed42"
RUN_DIR = PROJECT / RUN_NAME

PROTOCOL = ROOT / "00_Research_Protocol"

BEST = RUN_DIR / "weights/best.pt"
LAST = RUN_DIR / "weights/last.pt"

PROJECT.mkdir(parents=True, exist_ok=True)
PROTOCOL.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# 2. Pre-training safety checks
# ----------------------------------------------------------

assert torch.cuda.is_available(), (
    "GPU is not available."
)

assert DATA_YAML.is_file(), (
    "Local dataset YAML not found."
)

data_config = yaml.safe_load(
    DATA_YAML.read_text(encoding="utf-8")
)

assert len(data_config["names"]) == 6
assert Path(data_config["path"]).is_dir()

if RUN_DIR.exists():
    raise RuntimeError(
        "STOP: RT-DETR-L run already exists. "
        "Inspect checkpoints before restarting."
    )

# ----------------------------------------------------------
# 3. Training configuration
# ----------------------------------------------------------

TRAIN_CONFIG = {
    "data": str(DATA_YAML),
    "epochs": 100,
    "imgsz": 640,
    "batch": 2,
    "patience": 25,
    "seed": 42,
    "deterministic": False,
    "device": 0,
    "workers": 2,
    "optimizer": "auto",
    "amp": True,
    "pretrained": True,
    "save": True,
    "save_period": 10,
    "plots": True,
    "val": True,
    "cache": False,
    "project": str(PROJECT),
    "name": RUN_NAME,
    "exist_ok": False,
    "verbose": True
}

# ----------------------------------------------------------
# 4. Research metadata
# ----------------------------------------------------------

freeze_path = (
    ROOT / "01_Datasets/Dataset_QC"
    / "frozen_dataset_v1_summary.json"
)

assert freeze_path.is_file()

freeze_info = json.loads(
    freeze_path.read_text(encoding="utf-8")
)

metadata = {
    "model": "RT-DETR-L",
    "pretrained_weights": "rtdetr-l.pt",
    "dataset_version": "v1_cleaned",
    "dataset_fingerprint": freeze_info["manifest_sha256"],
    "dataset_counts": {
        "train": 1065,
        "valid": 303,
        "test": 154
    },
    "training_configuration": TRAIN_CONFIG,
    "pytorch_version": torch.__version__,
    "ultralytics_version": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "research_note": (
        "RT-DETR-L uses batch=2 and deterministic=False "
        "because of architecture and hardware constraints."
    )
}

metadata_path = (
    PROTOCOL / "RTDETR_L_training_config.json"
)

if metadata_path.exists():
    raise RuntimeError(
        "Existing RT-DETR-L protocol found. "
        "Inspect it before starting a new run."
    )

metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

# ----------------------------------------------------------
# 5. Display experiment configuration
# ----------------------------------------------------------

print("=" * 60)
print("RT-DETR-L TRAINING CONFIGURATION")
print("=" * 60)

print("Model: RT-DETR-L")
print("Pretrained weights: rtdetr-l.pt")
print("Dataset:", DATA_YAML)
print("Classes:", len(data_config["names"]))
print("Epochs:", TRAIN_CONFIG["epochs"])
print("Batch size:", TRAIN_CONFIG["batch"])
print("Image size:", TRAIN_CONFIG["imgsz"])
print("Seed:", TRAIN_CONFIG["seed"])
print("GPU:", torch.cuda.get_device_name(0))
print("Output:", RUN_DIR)

# ----------------------------------------------------------
# 6. Load pretrained RT-DETR-L
# ----------------------------------------------------------

print("\nLoading pretrained RT-DETR-L...")

model = RTDETR("rtdetr-l.pt")

# ----------------------------------------------------------
# 7. Start training
# ----------------------------------------------------------

print("\n" + "=" * 60)
print("STARTING RT-DETR-L TRAINING")
print("=" * 60)

results = model.train(**TRAIN_CONFIG)

# ----------------------------------------------------------
# 8. Check saved results
# ----------------------------------------------------------

print("\n" + "=" * 60)
print("RT-DETR-L TRAINING FINISHED")
print("=" * 60)

print("Best checkpoint:", BEST.is_file())
print("Last checkpoint:", LAST.is_file())
print(
    "Training CSV:",
    (RUN_DIR / "results.csv").is_file()
)

if (RUN_DIR / "results.csv").exists():
    history = pd.read_csv(RUN_DIR / "results.csv")
    history.columns = history.columns.str.strip()

    print("Last recorded epoch:",
          int(history["epoch"].iloc[-1]))

if not BEST.is_file():
    raise RuntimeError(
        "Best RT-DETR-L checkpoint not found."
    )

print("\nTRAINING STATUS: COMPLETED")


In [ ]:
# STEP 1 — Restore frozen input exports to the research Drive folder
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
from hashlib import sha256
from google.colab import drive, files
import json

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/DentalOPG_Research_2026/05_Human_Observer_Study/Final_Comparative_Analysis')
INPUTS = PROJECT / 'Inputs'
INPUTS.mkdir(parents=True, exist_ok=True)

INPUT_FILENAMES = [
    'Dentist01_CVAT_Annotations.zip',
    'Dentist02_CVAT_Annotations.zip.zip',
    'reference_annotations.csv'
]
ENGINE = 'DentalOPG_Reader_Analysis_Engine.py'

need = any(not (INPUTS / p).is_file() for p in INPUT_FILENAMES) or not (PROJECT / ENGINE).is_file()
if need:
    print('Upload DentalOPG_Readers_Colab_Inputs_and_Code.zip from this conversation.')
    upload = files.upload()
    assert 'DentalOPG_Readers_Colab_Inputs_and_Code.zip' in upload
    with ZipFile(BytesIO(upload['DentalOPG_Readers_Colab_Inputs_and_Code.zip'])) as z:
        for name in INPUT_FILENAMES + [ENGINE]:
            content = z.read(name)
            dest = PROJECT / ENGINE if name == ENGINE else INPUTS / name
            if dest.is_file():
                assert dest.read_bytes() == content, f'Existing file mismatch: {dest}'
            else:
                dest.write_bytes(content)

manifest = {name: sha256((INPUTS/name).read_bytes()).hexdigest() for name in INPUT_FILENAMES}
manifest[ENGINE] = sha256((PROJECT/ENGINE).read_bytes()).hexdigest()
manifest_path = PROJECT / 'input_manifest_sha256.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Frozen input fingerprints differ. Stop.'
else:
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('STATUS: INPUT FILES VERIFIED')
for name in manifest: print(name, manifest[name][:16] + '...')
print('Project:', PROJECT)

In [ ]:
# STEP 2 — Reproduce matching, QC, and paired-image bootstrap
import subprocess, sys
subprocess.run([sys.executable, str(PROJECT / ENGINE)], check=True)
print('STATUS: READER ANALYSIS COMPLETED')

In [ ]:
# STEP 3 — Inspect the study outputs
import pandas as pd
RESULTS=PROJECT/'Results'
for name in ['QC_Summary','Summary_All_Thresholds','Per_Class_All_Thresholds','Bootstrap_F1_Differences']:
    print('\n===== ' + name + ' =====')
    display(pd.read_csv(RESULTS / (name + '.csv')))
print('\nAll CSV reports:', RESULTS)

In [ ]:

# ==========================================================
# STEP 11-0 - INSTALL REQUIRED LIBRARIES
# ==========================================================

%pip install -q ultralytics==8.4.175 scipy pyyaml

print("STATUS: INSTALLATION COMPLETED")


In [ ]:

# ==========================================================
# VERIFY ULTRALYTICS INSTALLATION
# ==========================================================

import ultralytics
import torch
from ultralytics import YOLO, RTDETR
from scipy.optimize import linear_sum_assignment

print("=" * 55)
print("RESEARCH ENVIRONMENT CHECK")
print("=" * 55)

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert ultralytics.__version__ == "8.4.175"

print("\nSTATUS: ENVIRONMENT READY")


In [ ]:

# ==========================================================
# STEP 11A - SETUP AND VERIFY RESEARCH FILES
# ==========================================================

from pathlib import Path
from hashlib import sha256
from scipy.optimize import linear_sum_assignment
from google.colab import drive
from ultralytics import YOLO, RTDETR

import numpy as np
import pandas as pd
import torch
import yaml
import json
import shutil

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026"
)

HUMAN = (
    ROOT /
    "05_Human_Observer_Study/Final_Comparative_Analysis"
)

HUMAN_RESULTS = HUMAN / "Results"
AI_OUT = HUMAN / "AI_vs_Human_v1"

DRIVE_DATA = (
    ROOT / "01_Datasets/Prepared_6_Classes/v1_cleaned"
)

LOCAL_DATA = Path("/content/dentalopg_v1_cleaned")

MODEL_SPECS = [
    (
        "YOLOv8s",
        YOLO,
        ROOT / "03_Trained_Models/YOLOv8s/"
        "YOLOv8s_seed42/weights/best.pt"
    ),
    (
        "YOLO11s",
        YOLO,
        ROOT / "03_Trained_Models/YOLO11s/"
        "YOLO11s_seed42/weights/best.pt"
    ),
    (
        "RT-DETR-L",
        RTDETR,
        ROOT / "03_Trained_Models/RT_DETR/"
        "RTDETR_L_seed42/weights/best.pt"
    )
]

CLASS_NAMES = [
    "Caries",
    "Missing Teeth",
    "Root Canal Treatment",
    "Filling",
    "Crown",
    "Unerupted Wisdom Teeth"
]

CONF_GRID = [
    0.05, 0.10, 0.15, 0.20,
    0.25, 0.30, 0.35, 0.40,
    0.45, 0.50, 0.55, 0.60,
    0.65, 0.70, 0.75, 0.80
]

IOU_LEVELS = (0.50, 0.25)

def sha_file(path):
    h = sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

# ----------------------------------------------------------
# Safety checks
# ----------------------------------------------------------

assert torch.cuda.is_available(), (
    "Select Runtime > Change runtime type > GPU"
)

assert DRIVE_DATA.is_dir(), (
    "Frozen dataset not found in Google Drive."
)

assert (
    HUMAN_RESULTS / "Per_Image_All_Thresholds.csv"
).is_file(), "Reader analysis results not found."

assert (
    HUMAN_RESULTS / "Per_Class_All_Thresholds.csv"
).is_file(), "Reader per-class results not found."

assert (
    HUMAN / "Inputs/reference_annotations.csv"
).is_file(), "Reference annotations not found."

if not LOCAL_DATA.is_dir():

    print("Restoring frozen dataset locally...")

    shutil.copytree(
        DRIVE_DATA,
        LOCAL_DATA
    )

for split, expected in [
    ("valid", 303),
    ("test", 154)
]:

    images = list(
        (LOCAL_DATA / split / "images").glob("*.*")
    )

    labels = list(
        (LOCAL_DATA / split / "labels").glob("*.txt")
    )

    assert len(images) == expected
    assert len(labels) == expected

WEIGHT_HASHES = {}

for name, loader, weights in MODEL_SPECS:

    assert weights.is_file(), (
        f"Missing checkpoint: {weights}"
    )

    WEIGHT_HASHES[name] = sha_file(weights)

assert len(WEIGHT_HASHES) == 3
assert len(CLASS_NAMES) == 6

AI_OUT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 60)
print("STEP 11A - RESEARCH VERIFICATION")
print("=" * 60)

print("GPU:", torch.cuda.get_device_name(0))
print("Validation images:", 303)
print("Test images:", 154)
print("Models:", len(MODEL_SPECS))
print("Classes:", len(CLASS_NAMES))

for name, _, weights in MODEL_SPECS:
    print(name, "VERIFIED")

print("Reader analysis: VERIFIED")
print("Output directory:", AI_OUT)

print("\nSTATUS: STEP 11A READY")


In [ ]:

# ==================================================
# CHECK REQUIRED VARIABLES BEFORE STEP 11C
# ==================================================

required = [
    "AI_OUT",
    "WEIGHT_HASHES",
    "CONF_GRID",
    "MODEL_SPECS",
    "CLASS_NAMES",
    "LOCAL_DATA",
    "load_gt",
    "predict_split",
    "evaluate_rows",
    "score_counts"
]

missing = []

print("=" * 55)
print("STEP 11 - ENVIRONMENT DIAGNOSIS")
print("=" * 55)

for name in required:
    if name in globals():
        print(f"{name}: READY")
    else:
        print(f"{name}: MISSING")
        missing.append(name)

print("\n===== STATUS =====")

if not missing:
    print("STATUS: READY FOR STEP 11C")
else:
    print("STATUS: ENVIRONMENT INCOMPLETE")
    print("Missing variables:", missing)
    print("Run STEP 11A and STEP 11B first.")


In [ ]:


# ==========================================================
# STEP 11B - SHARED MATCHING AND INFERENCE FUNCTIONS
# ==========================================================

def list_split_images(split):

    images = sorted(
        p for p in (
            LOCAL_DATA / split / "images"
        ).iterdir()
        if p.suffix.lower() in {
            ".jpg", ".jpeg", ".png"
        }
    )

    expected = 303 if split == "valid" else 154

    assert len(images) == expected

    return images


def load_gt(split):

    result = {}

    for path in list_split_images(split):

        entries = []

        txt = (
            LOCAL_DATA / split / "labels"
            / f"{path.stem}.txt"
        )

        assert txt.is_file()

        for line in txt.read_text().splitlines():

            if not line.strip():
                continue

            parts = line.split()

            assert len(parts) == 5

            cls = int(parts[0])

            xc, yc, w, h = map(
                float, parts[1:]
            )

            assert 0 <= cls < 6

            entries.append((
                cls,
                xc - w / 2,
                yc - h / 2,
                xc + w / 2,
                yc + h / 2
            ))

        result[path.name] = entries

    return result


def box_iou(a, b):

    xx1 = max(a[0], b[0])
    yy1 = max(a[1], b[1])

    xx2 = min(a[2], b[2])
    yy2 = min(a[3], b[3])

    inter = (
        max(0.0, xx2 - xx1)
        * max(0.0, yy2 - yy1)
    )

    if inter <= 0:
        return 0.0

    area_a = (
        max(0.0, a[2] - a[0])
        * max(0.0, a[3] - a[1])
    )

    area_b = (
        max(0.0, b[2] - b[0])
        * max(0.0, b[3] - b[1])
    )

    return inter / (
        area_a + area_b - inter
    )


def pair_match_count(aa, bb, threshold):

    n = len(aa)
    m = len(bb)

    if not n or not m:
        return 0

    matrix = np.array([
        [box_iou(a, b) for b in bb]
        for a in aa
    ], dtype=float)

    weights = np.zeros(
        (n + m, n + m),
        dtype=float
    )

    weights[:n, :m] = np.where(
        matrix + 1e-12 >= threshold,
        2.0 + matrix,
        0.0
    )

    ii, jj = linear_sum_assignment(
        weights,
        maximize=True
    )

    return sum(
        int(
            i < n
            and j < m
            and matrix[i, j] + 1e-12 >= threshold
        )
        for i, j in zip(ii, jj)
    )


def predict_split(name, loader, weights, split):

    model = loader(str(weights))

    rows = []

    images = list_split_images(split)

    seen = []

    results = model.predict(
        source=[str(p) for p in images],
        imgsz=640,
        conf=0.01,
        iou=0.70,
        max_det=300,
        agnostic_nms=False,
        device=0,
        save=False,
        stream=True,
        verbose=False
    )

    for res in results:

        imname = Path(res.path).name
        seen.append(imname)

        height, width = res.orig_shape

        if res.boxes is None:
            continue

        boxes = res.boxes.xyxy.cpu().numpy()

        classes = (
            res.boxes.cls.cpu().numpy().astype(int)
        )

        scores = (
            res.boxes.conf.cpu().numpy()
        )

        for cls, confidence, box in zip(
            classes, scores, boxes
        ):

            assert 0 <= cls < 6

            rows.append({
                "model": name,
                "filename": imname,
                "class_id": int(cls),
                "confidence": float(confidence),
                "x1": float(box[0] / width),
                "y1": float(box[1] / height),
                "x2": float(box[2] / width),
                "y2": float(box[3] / height)
            })

    assert len(seen) == len(images)

    assert set(seen) == {
        p.name for p in images
    }

    del model

    torch.cuda.empty_cache()

    return pd.DataFrame(
        rows,
        columns=[
            "model", "filename", "class_id",
            "confidence", "x1", "y1", "x2", "y2"
        ]
    )


def evaluate_rows(
    preds,
    gt,
    threshold,
    confidence,
    case_map=None
):

    selected = preds[
        preds["confidence"] >= confidence
    ]

    pred_groups = {
        key: group[
            ["x1", "y1", "x2", "y2"]
        ].to_numpy(dtype=float).tolist()
        for key, group in selected.groupby(
            ["filename", "class_id"]
        )
    }

    rows = []

    for filename, reference in gt.items():

        for cls in range(6):

            predicted_boxes = pred_groups.get(
                (filename, cls),
                []
            )

            reference_boxes = [
                list(b[1:])
                for b in reference
                if b[0] == cls
            ]

            tp = pair_match_count(
                predicted_boxes,
                reference_boxes,
                threshold
            )

            rows.append({
                "filename": filename,
                "case_id": (
                    case_map.get(filename)
                    if case_map else ""
                ),
                "class_id": cls,
                "class_name": CLASS_NAMES[cls],
                "TP": tp,
                "FP": len(predicted_boxes) - tp,
                "FN": len(reference_boxes) - tp,
                "detections": len(predicted_boxes),
                "reference_boxes": len(reference_boxes)
            })

    return pd.DataFrame(rows)


def score_counts(df):

    tp = int(df["TP"].sum())
    fp = int(df["FP"].sum())
    fn = int(df["FN"].sum())

    precision = (
        tp / (tp + fp)
        if tp + fp else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn else 0.0
    )

    f1 = (
        2 * tp / (2 * tp + fp + fn)
        if (2 * tp + fp + fn) else 0.0
    )

    return tp, fp, fn, precision, recall, f1


print("=" * 60)
print("MATCHING ALGORITHM INITIALIZED")
print("=" * 60)

print("One-to-one matching: ENABLED")
print("Class matching: REQUIRED")
print("Primary IoU: 0.50")
print("Exploratory IoU: 0.25")

print("\nSTATUS: STEP 11B READY")


In [ ]:

# ==========================================================
# STEP 11B FIX - MEMORY-SAFE MODEL INFERENCE
# PROCESS ONE IMAGE AT A TIME
# ==========================================================

import gc
import torch
import pandas as pd
from pathlib import Path

def predict_split(name, loader, weights, split):

    image_paths = list_split_images(split)

    expected_names = {
        path.name for path in image_paths
    }

    source_dir = (
        LOCAL_DATA / split / "images"
    )

    print("=" * 60)
    print("MEMORY-SAFE INFERENCE")
    print("Model:", name)
    print("Split:", split)
    print("Images:", len(image_paths))
    print("Batch size: 1")
    print("=" * 60)

    # Clear memory from previous inference runs
    gc.collect()
    torch.cuda.empty_cache()

    rows = []
    seen = []

    model = None
    results = None
    res = None

    try:

        model = loader(str(weights))

        with torch.inference_mode():

            # IMPORTANT:
            # Pass the directory, NOT a list of 303 files.
            # batch=1 limits the inference batch size.
            # stream=True avoids storing all results.

            results = model.predict(
                source=str(source_dir),
                imgsz=640,
                conf=0.01,
                iou=0.70,
                batch=1,
                max_det=300,
                agnostic_nms=False,
                device=0,
                half=False,
                save=False,
                stream=True,
                verbose=False
            )

            for res in results:

                filename = Path(res.path).name

                assert filename in expected_names, (
                    f"Unexpected image: {filename}"
                )

                seen.append(filename)

                height, width = res.orig_shape

                if res.boxes is None:
                    continue

                boxes = (
                    res.boxes.xyxy
                    .detach()
                    .cpu()
                    .numpy()
                )

                classes = (
                    res.boxes.cls
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(int)
                )

                confidences = (
                    res.boxes.conf
                    .detach()
                    .cpu()
                    .numpy()
                )

                for cls, confidence, box in zip(
                    classes,
                    confidences,
                    boxes
                ):

                    assert 0 <= cls < 6

                    rows.append({
                        "model": name,
                        "filename": filename,
                        "class_id": int(cls),
                        "confidence": float(confidence),
                        "x1": float(box[0] / width),
                        "y1": float(box[1] / height),
                        "x2": float(box[2] / width),
                        "y2": float(box[3] / height)
                    })

        # Confirm every image was processed exactly once
        assert len(seen) == len(image_paths), (
            "Incomplete inference."
        )

        assert len(set(seen)) == len(seen), (
            "Duplicate image predictions detected."
        )

        assert set(seen) == expected_names, (
            "Image set mismatch."
        )

        df = pd.DataFrame(
            rows,
            columns=[
                "model",
                "filename",
                "class_id",
                "confidence",
                "x1",
                "y1",
                "x2",
                "y2"
            ]
        )

        print("\nInference completed.")
        print("Images processed:", len(seen))
        print("Predictions:", len(df))

        return df

    finally:

        # Release generator and model references
        res = None

        if results is not None:
            if hasattr(results, "close"):
                results.close()

        results = None
        model = None

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

            allocated = (
                torch.cuda.memory_allocated()
                / (1024 ** 3)
            )

            print(
                "GPU allocated after cleanup:",
                round(allocated, 3),
                "GB"
            )


print("\nSTATUS: MEMORY-SAFE INFERENCE READY")


In [ ]:

# ==========================================================
# STEP 11C - VALIDATION CONFIDENCE CALIBRATION
# NO TEST SET - NO TRAINING
# ==========================================================

lock_path = (
    AI_OUT / "AI_Validation_Confidence_Lock.json"
)

validation_summary = (
    AI_OUT / "AI_Validation_Confidence_Grid.csv"
)

# ----------------------------------------------------------
# 1. Check whether confidence is already locked
# ----------------------------------------------------------

if lock_path.exists():

    lock = json.loads(
        lock_path.read_text(encoding="utf-8")
    )

    assert lock["weights_sha256"] == WEIGHT_HASHES, (
        "Model checkpoints changed since calibration."
    )

    assert lock["confidence_candidates"] == CONF_GRID

    assert validation_summary.is_file()

    print("Existing confidence lock verified.")
    print("No recalibration performed.")

else:

    assert not validation_summary.exists(), (
        "Partial validation results exist. "
        "Inspect before running."
    )

    # ------------------------------------------------------
    # 2. Load validation ground truth
    # ------------------------------------------------------

    gt_valid = load_gt("valid")

    grid = []
    selected = {}

    # ------------------------------------------------------
    # 3. Calibrate each model
    # ------------------------------------------------------

    for name, loader, weights in MODEL_SPECS:

        print("\n" + "=" * 60)
        print("VALIDATION MODEL:", name)
        print("=" * 60)

        predictions = predict_split(
            name,
            loader,
            weights,
            "valid"
        )

        for confidence in CONF_GRID:

            evaluated = evaluate_rows(
                preds=predictions,
                gt=gt_valid,
                threshold=0.50,
                confidence=confidence
            )

            tp, fp, fn, p, r, f1 = score_counts(
                evaluated
            )

            grid.append({
                "model": name,
                "confidence": confidence,
                "TP": tp,
                "FP": fp,
                "FN": fn,
                "precision": p,
                "recall": r,
                "F1": f1
            })

        candidates = pd.DataFrame(
            row for row in grid
            if row["model"] == name
        )

        best = candidates.sort_values(
            ["F1", "confidence"],
            ascending=[False, False]
        ).iloc[0]

        selected[name] = float(
            best["confidence"]
        )

        print(
            "Selected confidence:",
            selected[name]
        )

        print(
            "Validation F1:",
            round(float(best["F1"]), 4)
        )

    # ------------------------------------------------------
    # 4. Lock confidence thresholds
    # ------------------------------------------------------

    lock = {
        "selection_method": (
            "Highest validation micro F1 at IoU=0.50; "
            "ties resolved using higher confidence."
        ),
        "confidence_candidates": CONF_GRID,
        "selected_confidence": selected,
        "weights_sha256": WEIGHT_HASHES,
        "evaluation_iou_thresholds": [
            0.50,
            0.25
        ],
        "note": (
            "Post-hoc comparative calibration after "
            "earlier test mAP reporting. "
            "No confidence threshold tuned on test."
        )
    }

    pd.DataFrame(grid).to_csv(
        validation_summary,
        index=False
    )

    with open(lock_path, "x") as f:
        json.dump(
            lock,
            f,
            indent=2
        )

# ----------------------------------------------------------
# 5. Final confidence lock report
# ----------------------------------------------------------

print("\n" + "=" * 60)
print("VALIDATION CONFIDENCE LOCK")
print("=" * 60)

confidence_table = pd.DataFrame([
    {
        "model": name,
        "locked_confidence": (
            lock["selected_confidence"][name]
        )
    }
    for name, _, _ in MODEL_SPECS
])

display(confidence_table)

print("\nLock file:", lock_path)
print("Validation grid:", validation_summary)

print("\nSTATUS: AI VALIDATION CONFIDENCE LOCKED")


In [ ]:

# ==========================================================
# STEP 11D-1 - VERIFY THE 154 TEST IMAGE IDENTITIES
# ==========================================================

from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
from PIL import Image
from hashlib import sha256
import pandas as pd
import numpy as np
import json
import re

assert lock_path.is_file(), "Validation confidence lock missing."

LOCK = json.loads(lock_path.read_text())

assert LOCK["weights_sha256"] == WEIGHT_HASHES
assert LOCK["selected_confidence"] == {
    "YOLOv8s": 0.4,
    "YOLO11s": 0.3,
    "RT-DETR-L": 0.5
}

TEST_IMAGES = list_split_images("test")
GT_TEST = load_gt("test")

assert len(TEST_IMAGES) == 154
assert sum(len(v) for v in GT_TEST.values()) == 588

READER_2_ZIP = (
    ROOT / "05_Human_Observer_Study"
    / "CVAT_Reader_Uploads_v1"
    / "Reader_2_CVAT_IMAGES_ONLY.zip"
)

assert READER_2_ZIP.is_file(), (
    "Reader 2 blinded image ZIP not found in Drive."
)

def decoded_pixel_hash(image):
    rgb = image.convert("RGB")
    h = sha256()
    h.update(str(rgb.size).encode())
    h.update(rgb.tobytes())
    return h.hexdigest()

source_hashes = {}

for path in TEST_IMAGES:
    with Image.open(path) as im:
        key = decoded_pixel_hash(im)

    assert key not in source_hashes, (
        "Duplicate test image pixels require manual review."
    )

    source_hashes[key] = path.name

mapping = []

with ZipFile(READER_2_ZIP) as z:

    names = [
        n for n in z.namelist()
        if n.lower().endswith(".png")
    ]

    assert len(names) == 154

    for name in names:

        match = re.search(r"CASE_\d{4}", Path(name).name)

        assert match is not None

        with Image.open(BytesIO(z.read(name))) as im:
            key = decoded_pixel_hash(im)

        assert key in source_hashes, (
            f"Image mismatch: {name}"
        )

        mapping.append({
            "case_id": match.group(),
            "original_filename": source_hashes[key],
            "reader_2_filename": Path(name).name
        })

MAP_DF = pd.DataFrame(mapping)

assert len(MAP_DF) == 154
assert MAP_DF["case_id"].nunique() == 154
assert MAP_DF["original_filename"].nunique() == 154

assert set(MAP_DF["case_id"]) == {
    f"CASE_{i:04d}" for i in range(1, 155)
}

assert set(MAP_DF["original_filename"]) == {
    p.name for p in TEST_IMAGES
}

CASE_MAP = dict(zip(
    MAP_DF["original_filename"],
    MAP_DF["case_id"]
))

# Check original dataset labels against the reader reference.
reference = pd.read_csv(
    HUMAN / "Inputs/reference_annotations.csv"
)

assert len(reference) == 588

reference["case_id"] = reference[
    "cvat_filename"
].str.extract(r"(CASE_\d{4})")[0]

assert reference["case_id"].notna().all()

for row in reference.itertuples():
    assert CASE_MAP[row.original_filename] == row.case_id

for original, group in reference.groupby("original_filename"):

    expected = sorted([
        (
            int(r.class_id),
            round(float(r.x_center-r.width/2), 6),
            round(float(r.y_center-r.height/2), 6),
            round(float(r.x_center+r.width/2), 6),
            round(float(r.y_center+r.height/2), 6)
        )
        for r in group.itertuples()
    ])

    actual = sorted([
        (
            int(box[0]),
            *(round(float(v), 6) for v in box[1:])
        )
        for box in GT_TEST[original]
    ])

    assert expected == actual, (
        f"Reference mismatch: {original}"
    )

MAP_PATH = AI_OUT / "Verified_Test_Case_Mapping.csv"

MAP_DF = MAP_DF.sort_values("case_id").reset_index(drop=True)

if MAP_PATH.exists():
    old = pd.read_csv(MAP_PATH, dtype=str)
    assert old.equals(MAP_DF.astype(str))
else:
    MAP_DF.to_csv(MAP_PATH, index=False)

print("=" * 60)
print("STEP 11D-1 - TEST IMAGE VERIFICATION")
print("=" * 60)
print("Images:", len(TEST_IMAGES))
print("Reference boxes:", sum(map(len, GT_TEST.values())))
print("Case mapping: VERIFIED")
print("Reference labels: VERIFIED")
print("Confidence thresholds: LOCKED")
print("Mapping file:", MAP_PATH)

print("\nSTATUS: TEST IDENTITIES VERIFIED")


In [ ]:

# ==========================================================
# STEP 11D-2 - RUN LOCKED TEST INFERENCE
# MEMORY-SAFE, WITH RESTARTABLE SAVED PREDICTIONS
# ==========================================================

import gc
import torch
import os
import json

RESULTS_DIR = AI_OUT / "Results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for name, loader, weights in MODEL_SPECS:

    prefix = name.replace("-", "_")

    prediction_path = (
        RESULTS_DIR / f"{prefix}_Test_Predictions.csv"
    )

    manifest_path = (
        RESULTS_DIR / f"{prefix}_Test_Manifest.json"
    )

    if prediction_path.exists():

        assert manifest_path.is_file(), (
            f"Missing manifest for existing {name} predictions."
        )

        metadata = json.loads(
            manifest_path.read_text()
        )

        assert metadata["weights_sha256"] == WEIGHT_HASHES[name]
        assert metadata["predictions_sha256"] == sha_file(prediction_path)
        assert metadata["images_processed"] == 154
        assert metadata["inference_confidence_floor"] == 0.01

        print(f"{name}: VERIFIED SAVED PREDICTIONS")
        continue

    assert not manifest_path.exists(), (
        f"Partial manifest detected for {name}."
    )

    print("\n" + "=" * 60)
    print("TEST MODEL:", name)
    print("Locked confidence:", LOCK["selected_confidence"][name])
    print("=" * 60)

    predictions = predict_split(
        name=name,
        loader=loader,
        weights=weights,
        split="test"
    )

    assert set(predictions["filename"]).issubset(
        set(CASE_MAP)
    )

    assert predictions["confidence"].between(
        0, 1
    ).all()

    predictions["case_id"] = (
        predictions["filename"].map(CASE_MAP)
    )

    assert predictions["case_id"].notna().all()

    temporary_path = prediction_path.with_suffix(
        ".partial.csv"
    )

    assert not temporary_path.exists(), (
        f"Inspect partial file: {temporary_path}"
    )

    predictions.to_csv(
        temporary_path,
        index=False
    )

    os.replace(
        temporary_path,
        prediction_path
    )

    metadata = {
        "model": name,
        "weights_sha256": WEIGHT_HASHES[name],
        "predictions_sha256": sha_file(prediction_path),
        "images_processed": 154,
        "inference_confidence_floor": 0.01,
        "locked_confidence": LOCK[
            "selected_confidence"
        ][name],
        "imgsz": 640,
        "nms_iou": 0.70,
        "max_det": 300
    }

    with open(manifest_path, "x") as f:
        json.dump(metadata, f, indent=2)

    print(name, "SAVED")
    print("Predictions:", len(predictions))

    del predictions
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("ALL THREE MODELS TESTED")
print("=" * 60)
print("Output:", RESULTS_DIR)

print("\nSTATUS: AI TEST PREDICTIONS SAVED")


In [ ]:

# ==========================================================
# STEP 11D-3 - FIVE-WAY COMPARATIVE ANALYSIS
# PRIMARY: IoU 0.50
# EXPLORATORY: IoU 0.25
# ==========================================================

from scipy.optimize import linear_sum_assignment
import numpy as np
import pandas as pd

FINAL_SUMMARY = (
    RESULTS_DIR / "AI_vs_Human_Combined_Summary.csv"
)

assert not FINAL_SUMMARY.exists(), (
    "Final test analysis already exists. "
    "Inspect saved results rather than overwriting."
)

AI_ROWS = []
SUMMARY = []
PER_CLASS = []
PER_IMAGE = []

for name, _, _ in MODEL_SPECS:

    pred_path = (
        RESULTS_DIR /
        f"{name.replace('-', '_')}_Test_Predictions.csv"
    )

    assert pred_path.is_file()

    predictions = pd.read_csv(pred_path)

    confidence = float(
        LOCK["selected_confidence"][name]
    )

    for iou_threshold in [0.50, 0.25]:

        evaluated = evaluate_rows(
            preds=predictions,
            gt=GT_TEST,
            threshold=iou_threshold,
            confidence=confidence,
            case_map=CASE_MAP
        )

        assert len(evaluated) == 154 * 6

        evaluated["participant"] = name
        evaluated["threshold"] = iou_threshold
        evaluated["locked_confidence"] = confidence

        AI_ROWS.append(evaluated)

        tp, fp, fn, p, r, f1 = score_counts(
            evaluated
        )

        SUMMARY.append({
            "participant": name,
            "threshold": iou_threshold,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision": p,
            "recall": r,
            "F1": f1,
            "confidence": confidence
        })

        for cls, grp in evaluated.groupby("class_id"):

            ct, cf, cn, cp, cr, cF = score_counts(grp)

            PER_CLASS.append({
                "participant": name,
                "threshold": iou_threshold,
                "class_id": int(cls),
                "class_name": CLASS_NAMES[int(cls)],
                "TP": ct,
                "FP": cf,
                "FN": cn,
                "precision": cp,
                "recall": cr,
                "F1": cF
            })

        per_image = evaluated.groupby(
            "case_id",
            as_index=False
        )[["TP", "FP", "FN"]].sum()

        per_image["participant"] = name
        per_image["threshold"] = iou_threshold

        PER_IMAGE.append(per_image)

# ----------------------------------------------------------
# Load the two dentists' previously verified results
# ----------------------------------------------------------

human_images = pd.read_csv(
    HUMAN_RESULTS / "Per_Image_All_Thresholds.csv"
)

human_classes = pd.read_csv(
    HUMAN_RESULTS / "Per_Class_All_Thresholds.csv"
)

for dentist in ["Dentist 01", "Dentist 02"]:

    comparison = f"{dentist} vs Dataset reference"

    for iou_threshold in [0.50, 0.25]:

        image_rows = human_images[
            (human_images["comparison"] == comparison)
            & np.isclose(
                human_images["threshold"],
                iou_threshold
            )
        ].copy()

        assert len(image_rows) == 154
        assert set(image_rows["case_id"]) == set(
            MAP_DF["case_id"]
        )

        tp, fp, fn, p, r, f1 = score_counts(
            image_rows
        )

        SUMMARY.append({
            "participant": dentist,
            "threshold": iou_threshold,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision": p,
            "recall": r,
            "F1": f1,
            "confidence": np.nan
        })

        image_rows = image_rows[
            ["case_id", "TP", "FP", "FN"]
        ]

        image_rows["participant"] = dentist
        image_rows["threshold"] = iou_threshold

        PER_IMAGE.append(image_rows)

        class_rows = human_classes[
            (human_classes["comparison"] == comparison)
            & np.isclose(
                human_classes["threshold"],
                iou_threshold
            )
        ]

        assert len(class_rows) == 6

        for row in class_rows.itertuples():

            PER_CLASS.append({
                "participant": dentist,
                "threshold": iou_threshold,
                "class_id": int(row.class_id),
                "class_name": row.class_name,
                "TP": int(row.TP),
                "FP": int(row.FP),
                "FN": int(row.FN),
                "precision": float(row.precision),
                "recall": float(row.recall),
                "F1": float(row.F1)
            })

# ----------------------------------------------------------
# Assemble all five participants
# ----------------------------------------------------------

summary_df = pd.DataFrame(SUMMARY)
per_class_df = pd.DataFrame(PER_CLASS)
per_image_df = pd.concat(PER_IMAGE, ignore_index=True)
ai_rows_df = pd.concat(AI_ROWS, ignore_index=True)

assert len(summary_df) == 10
assert len(per_class_df) == 60
assert len(per_image_df) == 154 * 10

for threshold in [0.50, 0.25]:

    sub = summary_df[
        np.isclose(summary_df["threshold"], threshold)
    ]

    assert len(sub) == 5
    assert (sub["TP"] + sub["FN"] == 588).all()

# Verify unchanged dentists' primary results
for dentist, expected_tp in [
    ("Dentist 01", 321),
    ("Dentist 02", 364)
]:

    sub = summary_df[
        (summary_df["participant"] == dentist)
        & np.isclose(summary_df["threshold"], 0.50)
    ]

    assert int(sub.iloc[0]["TP"]) == expected_tp

# ----------------------------------------------------------
# Paired image-level bootstrap
# ----------------------------------------------------------

participants = [
    "YOLOv8s",
    "YOLO11s",
    "RT-DETR-L",
    "Dentist 01",
    "Dentist 02"
]

all_cases = sorted(MAP_DF["case_id"])

rng = np.random.default_rng(20261014)
n_boot = 4000

sample_indices = rng.integers(
    0, 154, size=(n_boot, 154)
)

def micro_f1(values):

    tp = values[..., 0]
    fp = values[..., 1]
    fn = values[..., 2]

    return 2 * tp / np.maximum(
        1.0, 2 * tp + fp + fn
    )

BOOTSTRAP = []

for threshold in [0.50, 0.25]:

    vectors = {}

    for participant in participants:

        sub = per_image_df[
            (per_image_df["participant"] == participant)
            & np.isclose(
                per_image_df["threshold"], threshold
            )
        ]

        values = sub.set_index("case_id")[
            ["TP", "FP", "FN"]
        ].reindex(all_cases).to_numpy(dtype=float)

        assert values.shape == (154, 3)
        assert not np.isnan(values).any()

        vectors[participant] = values

    for i, first in enumerate(participants):

        for second in participants[i + 1:]:

            a = vectors[first]
            b = vectors[second]

            observed = float(
                micro_f1(b.sum(axis=0))
                - micro_f1(a.sum(axis=0))
            )

            differences = (
                micro_f1(b[sample_indices].sum(axis=1))
                - micro_f1(a[sample_indices].sum(axis=1))
            )

            BOOTSTRAP.append({
                "threshold": threshold,
                "A": first,
                "B": second,
                "B_minus_A_F1": observed,
                "ci_low": float(
                    np.quantile(differences, 0.025)
                ),
                "ci_high": float(
                    np.quantile(differences, 0.975)
                ),
                "bootstrap_images": 154,
                "bootstrap_replicates": n_boot
            })

# ----------------------------------------------------------
# Save official results
# ----------------------------------------------------------

ai_rows_df.to_csv(
    RESULTS_DIR / "AI_Per_Image_Class.csv",
    index=False
)

per_class_df.to_csv(
    RESULTS_DIR / "AI_vs_Human_Combined_Per_Class.csv",
    index=False
)

per_image_df.to_csv(
    RESULTS_DIR / "AI_vs_Human_Combined_Per_Image.csv",
    index=False
)

pd.DataFrame(BOOTSTRAP).to_csv(
    RESULTS_DIR / "AI_vs_Human_Paired_Bootstrap.csv",
    index=False
)

summary_df.to_csv(
    FINAL_SUMMARY,
    index=False
)

# ----------------------------------------------------------
# Display final tables
# ----------------------------------------------------------

print("\n" + "=" * 65)
print("FIVE-WAY COMPARISON - PRIMARY IoU 0.50")
print("=" * 65)

display(
    summary_df[
        np.isclose(summary_df["threshold"], 0.50)
    ].sort_values("F1", ascending=False)
)

print("\n" + "=" * 65)
print("FIVE-WAY COMPARISON - EXPLORATORY IoU 0.25")
print("=" * 65)

display(
    summary_df[
        np.isclose(summary_df["threshold"], 0.25)
    ].sort_values("F1", ascending=False)
)

print("\nResults directory:", RESULTS_DIR)

print("\nSTATUS: FIVE-WAY TEST COMPARISON COMPLETED")


In [ ]:

# ==========================================================
# STEP 11E-1 - AI VS DENTISTS: PAIRED BOOTSTRAP
# Reads existing results only - no GPU required
# ==========================================================

from pathlib import Path
import pandas as pd
import numpy as np

RESULTS_DIR = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026/"
    "05_Human_Observer_Study/"
    "Final_Comparative_Analysis/"
    "AI_vs_Human_v1/Results"
)

bootstrap = pd.read_csv(
    RESULTS_DIR / "AI_vs_Human_Paired_Bootstrap.csv"
)

models = ["YOLOv8s", "YOLO11s", "RT-DETR-L"]
dentists = ["Dentist 01", "Dentist 02"]

rows = []

for threshold in [0.50, 0.25]:

    for model in models:
        for dentist in dentists:

            subset = bootstrap[
                np.isclose(bootstrap["threshold"], threshold)
                & (
                    (
                        (bootstrap["A"] == model)
                        & (bootstrap["B"] == dentist)
                    )
                    |
                    (
                        (bootstrap["A"] == dentist)
                        & (bootstrap["B"] == model)
                    )
                )
            ]

            assert len(subset) == 1

            result = subset.iloc[0]

            delta = float(result["B_minus_A_F1"])
            low = float(result["ci_low"])
            high = float(result["ci_high"])

            # Always report AI minus Dentist
            if result["A"] == model:
                delta, low, high = (
                    -delta, -high, -low
                )

            rows.append({
                "IoU": threshold,
                "AI_model": model,
                "Dentist": dentist,
                "AI_minus_Dentist_F1_pp": round(delta * 100, 2),
                "CI_95_low_pp": round(low * 100, 2),
                "CI_95_high_pp": round(high * 100, 2),
                "CI_excludes_zero": bool(
                    low > 0 or high < 0
                )
            })

comparison = pd.DataFrame(rows)

display(
    comparison[
        np.isclose(comparison["IoU"], 0.50)
    ]
)

comparison.to_csv(
    RESULTS_DIR / "AI_vs_Dentists_F1_CI_Overview.csv",
    index=False
)

print("\nSTATUS: PAIRED BOOTSTRAP REVIEW COMPLETED")


In [ ]:

# ==========================================================
# STEP 11E-2 - PER-CLASS F1 COMPARISON
# ==========================================================

import pandas as pd
import numpy as np

per_class = pd.read_csv(
    RESULTS_DIR / "AI_vs_Human_Combined_Per_Class.csv"
)

class_order = [
    "Caries",
    "Missing Teeth",
    "Root Canal Treatment",
    "Filling",
    "Crown",
    "Unerupted Wisdom Teeth"
]

participant_order = [
    "Dentist 01",
    "Dentist 02",
    "YOLOv8s",
    "YOLO11s",
    "RT-DETR-L"
]

for threshold in [0.50, 0.25]:

    selected = per_class[
        np.isclose(per_class["threshold"], threshold)
    ]

    table = selected.pivot(
        index="class_name",
        columns="participant",
        values="F1"
    )

    table = table.loc[
        class_order,
        participant_order
    ]

    table = (table * 100).round(2)

    print("\n" + "=" * 65)
    print(f"PER-CLASS F1 (%) - IoU {threshold:.2f}")
    print("=" * 65)

    display(table)

    filename = (
        "Per_Class_F1_IoU_050.csv"
        if threshold == 0.50
        else "Per_Class_F1_IoU_025.csv"
    )

    table.to_csv(RESULTS_DIR / filename)

print("\nSTATUS: PER-CLASS ANALYSIS COMPLETED")


In [ ]:

# ==========================================================
# STEP 11F - DISAGREEMENT AUDIT & BLINDED REVIEW REGISTRY
# NO RETRAINING - NO MODIFICATION OF ORIGINAL ANNOTATIONS
# ==========================================================

from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026/"
    "05_Human_Observer_Study/"
    "Final_Comparative_Analysis/AI_vs_Human_v1"
)

RESULTS = BASE / "Results"

source = pd.read_csv(
    RESULTS / "AI_vs_Human_Combined_Per_Image.csv"
)

mapping = pd.read_csv(
    BASE / "Verified_Test_Case_Mapping.csv"
)

participants = [
    "Dentist 01",
    "Dentist 02",
    "YOLOv8s",
    "YOLO11s",
    "RT-DETR-L"
]

# Use the locked primary evaluation only
df = source[
    np.isclose(source["threshold"], 0.50)
].copy()

assert len(mapping) == 154
assert mapping["case_id"].is_unique
assert len(df) == 154 * 5
assert set(df["participant"]) == set(participants)
assert set(df["case_id"]) == set(mapping["case_id"])
assert not df.duplicated(
    ["case_id", "participant"]
).any()

# Check all participants use the same reference counts
df["reference_boxes"] = df["TP"] + df["FN"]

assert (
    df.groupby("case_id")["reference_boxes"]
    .nunique()
    .eq(1)
    .all()
)

# Image-level F1 is undefined when both reference
# and predictions contain zero boxes.
denominator = (
    2 * df["TP"] + df["FP"] + df["FN"]
)

df["image_F1"] = np.where(
    denominator > 0,
    2 * df["TP"] / denominator.replace(0, np.nan),
    np.nan
)

scores = df.pivot(
    index="case_id",
    columns="participant",
    values="image_F1"
)

scores = scores.reindex(columns=participants)

# Descriptive disagreement indicators
audit = pd.DataFrame(index=scores.index)

audit["F1_spread_pp"] = (
    scores.max(axis=1) - scores.min(axis=1)
) * 100

audit["F1_spread_pp"] = (
    audit["F1_spread_pp"].fillna(0)
)

audit["Dentist02_minus_RTDETR_F1_pp"] = (
    (scores["Dentist 02"] - scores["RT-DETR-L"])
    * 100
)

audit["total_FP_across_evaluators"] = (
    df.groupby("case_id")["FP"].sum()
)

audit["total_FN_across_evaluators"] = (
    df.groupby("case_id")["FN"].sum()
)

audit["reference_boxes"] = (
    df.groupby("case_id")["reference_boxes"].first()
)

audit["reference_empty_with_FP"] = (
    (audit["reference_boxes"] == 0)
    & (audit["total_FP_across_evaluators"] > 0)
)

audit = audit.reset_index()

audit = audit.merge(
    mapping[
        [
            "case_id",
            "original_filename",
            "reader_2_filename"
        ]
    ],
    on="case_id",
    validate="one_to_one"
)

# Rank for the researcher's quality-control review
audit = audit.sort_values(
    [
        "reference_empty_with_FP",
        "F1_spread_pp",
        "total_FN_across_evaluators",
        "total_FP_across_evaluators"
    ],
    ascending=[False, False, False, False]
).reset_index(drop=True)

audit.insert(
    0,
    "review_priority",
    np.arange(1, len(audit) + 1)
)

assert len(audit) == 154

audit_path = (
    RESULTS / "Researcher_Disagreement_Priority.csv"
)

audit.to_csv(audit_path, index=False)

# Create a fully blinded randomized list of all 154 cases
blinded = mapping[
    ["case_id", "reader_2_filename"]
].sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

blinded.insert(
    0,
    "review_order",
    np.arange(1, 155)
)

blinded = blinded.rename(
    columns={"reader_2_filename": "image_filename"}
)

blinded_path = (
    RESULTS / "Blinded_Expert_Review_List.csv"
)

blinded.to_csv(blinded_path, index=False)

print("=" * 65)
print("STEP 11F - DISAGREEMENT AUDIT COMPLETED")
print("=" * 65)

print("Total cases:", len(audit))
print("Primary IoU:", 0.50)

print(
    "Reference-empty cases:",
    int((audit["reference_boxes"] == 0).sum())
)

print(
    "Reference-empty cases with detections:",
    int(audit["reference_empty_with_FP"].sum())
)

print("\nTOP 10 CASES FOR RESEARCHER REVIEW")

display(
    audit[
        [
            "case_id",
            "reference_boxes",
            "F1_spread_pp",
            "total_FP_across_evaluators",
            "total_FN_across_evaluators"
        ]
    ].head(10)
)

print("\nResearcher report:", audit_path)
print("Blinded expert registry:", blinded_path)

print("\nSTATUS: DISAGREEMENT AUDIT READY")


In [ ]:

# ==========================================================
# STEP 11G - RESEARCHER VISUAL DISAGREEMENT AUDIT
# 10 PRIORITY CASES - READ ONLY
# ==========================================================

from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026"
)

BASE = (
    ROOT / "05_Human_Observer_Study"
    / "Final_Comparative_Analysis/AI_vs_Human_v1"
)

RESULTS = BASE / "Results"

IMAGE_DIR = (
    ROOT / "01_Datasets"
    / "Prepared_6_Classes/v1_cleaned"
    / "test/images"
)

priority = pd.read_csv(
    RESULTS / "Researcher_Disagreement_Priority.csv"
)

mapping = pd.read_csv(
    BASE / "Verified_Test_Case_Mapping.csv"
)

evaluations = pd.read_csv(
    RESULTS / "AI_vs_Human_Combined_Per_Image.csv"
)

# ----------------------------------------------------------
# 1. Select the 10 highest-priority cases
# ----------------------------------------------------------

cases = priority.head(10).copy()

assert len(cases) == 10
assert cases["case_id"].is_unique

primary = evaluations[
    np.isclose(evaluations["threshold"], 0.50)
    & evaluations["case_id"].isin(cases["case_id"])
].copy()

participants = [
    "Dentist 01",
    "Dentist 02",
    "YOLOv8s",
    "YOLO11s",
    "RT-DETR-L"
]

assert len(primary) == 10 * 5

# ----------------------------------------------------------
# 2. Display per-evaluator false-positive counts
# ----------------------------------------------------------

fp_table = primary.pivot(
    index="case_id",
    columns="participant",
    values="FP"
)

fp_table = fp_table.reindex(
    index=cases["case_id"],
    columns=participants
)

fp_table["Total FP"] = fp_table.sum(axis=1)

print("=" * 65)
print("FALSE POSITIVE COUNTS BY EVALUATOR")
print("Dataset reference; primary IoU 0.50")
print("=" * 65)

display(fp_table)

# ----------------------------------------------------------
# 3. Display original radiographs without annotations
# ----------------------------------------------------------

case_to_filename = dict(zip(
    mapping["case_id"],
    mapping["original_filename"]
))

fig, axes = plt.subplots(
    5, 2,
    figsize=(13, 25)
)

for ax, row in zip(axes.ravel(), cases.itertuples()):

    case_id = row.case_id
    filename = case_to_filename[case_id]

    image_path = IMAGE_DIR / filename

    assert image_path.is_file(), image_path

    with Image.open(image_path) as image:
        ax.imshow(image.convert("RGB"))

    ax.set_title(
        f"{case_id} | Reference boxes: "
        f"{int(row.reference_boxes)} | "
        f"Total FP: {int(row.total_FP_across_evaluators)}",
        fontsize=11
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

# ----------------------------------------------------------
# 4. Create a researcher-only clinical audit tracker
# ----------------------------------------------------------

tracker_path = (
    RESULTS / "Researcher_Clinical_Audit_Tracker.csv"
)

if not tracker_path.exists():

    tracker = priority[
        [
            "case_id",
            "original_filename",
            "reference_boxes"
        ]
    ].copy()

    tracker["review_status"] = "Pending"
    tracker["possible_missing_annotation"] = ""
    tracker["possible_box_boundary_issue"] = ""
    tracker["clinical_review_notes"] = ""
    tracker["adjudication_required"] = ""

    tracker.to_csv(
        tracker_path,
        index=False
    )

print("\nAudit tracker:", tracker_path)
print("Original annotations modified: NO")
print("Test predictions modified: NO")
print("\nSTATUS: STEP 11G VISUAL AUDIT READY")


In [ ]:

# ==========================================================
# STEP 11G - RESEARCHER VISUAL DISAGREEMENT AUDIT
# 10 PRIORITY CASES - READ ONLY
# ==========================================================

from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026"
)

BASE = (
    ROOT / "05_Human_Observer_Study"
    / "Final_Comparative_Analysis/AI_vs_Human_v1"
)

RESULTS = BASE / "Results"

IMAGE_DIR = (
    ROOT / "01_Datasets"
    / "Prepared_6_Classes/v1_cleaned"
    / "test/images"
)

priority = pd.read_csv(
    RESULTS / "Researcher_Disagreement_Priority.csv"
)

mapping = pd.read_csv(
    BASE / "Verified_Test_Case_Mapping.csv"
)

evaluations = pd.read_csv(
    RESULTS / "AI_vs_Human_Combined_Per_Image.csv"
)

# ----------------------------------------------------------
# 1. Select the 10 highest-priority cases
# ----------------------------------------------------------

cases = priority.head(10).copy()

assert len(cases) == 10
assert cases["case_id"].is_unique

primary = evaluations[
    np.isclose(evaluations["threshold"], 0.50)
    & evaluations["case_id"].isin(cases["case_id"])
].copy()

participants = [
    "Dentist 01",
    "Dentist 02",
    "YOLOv8s",
    "YOLO11s",
    "RT-DETR-L"
]

assert len(primary) == 10 * 5

# ----------------------------------------------------------
# 2. Display per-evaluator false-positive counts
# ----------------------------------------------------------

fp_table = primary.pivot(
    index="case_id",
    columns="participant",
    values="FP"
)

fp_table = fp_table.reindex(
    index=cases["case_id"],
    columns=participants
)

fp_table["Total FP"] = fp_table.sum(axis=1)

print("=" * 65)
print("FALSE POSITIVE COUNTS BY EVALUATOR")
print("Dataset reference; primary IoU 0.50")
print("=" * 65)

display(fp_table)

# ----------------------------------------------------------
# 3. Display original radiographs without annotations
# ----------------------------------------------------------

case_to_filename = dict(zip(
    mapping["case_id"],
    mapping["original_filename"]
))

fig, axes = plt.subplots(
    5, 2,
    figsize=(13, 25)
)

for ax, row in zip(axes.ravel(), cases.itertuples()):

    case_id = row.case_id
    filename = case_to_filename[case_id]

    image_path = IMAGE_DIR / filename

    assert image_path.is_file(), image_path

    with Image.open(image_path) as image:
        ax.imshow(image.convert("RGB"))

    ax.set_title(
        f"{case_id} | Reference boxes: "
        f"{int(row.reference_boxes)} | "
        f"Total FP: {int(row.total_FP_across_evaluators)}",
        fontsize=11
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

# ----------------------------------------------------------
# 4. Create a researcher-only clinical audit tracker
# ----------------------------------------------------------

tracker_path = (
    RESULTS / "Researcher_Clinical_Audit_Tracker.csv"
)

if not tracker_path.exists():

    tracker = priority[
        [
            "case_id",
            "original_filename",
            "reference_boxes"
        ]
    ].copy()

    tracker["review_status"] = "Pending"
    tracker["possible_missing_annotation"] = ""
    tracker["possible_box_boundary_issue"] = ""
    tracker["clinical_review_notes"] = ""
    tracker["adjudication_required"] = ""

    tracker.to_csv(
        tracker_path,
        index=False
    )

print("\nAudit tracker:", tracker_path)
print("Original annotations modified: NO")
print("Test predictions modified: NO")
print("\nSTATUS: STEP 11G VISUAL AUDIT READY")


In [ ]:

# ==========================================================
# STEP 11H - VISUALIZE ALL FIVE EVALUATORS PER CASE
# RESEARCHER-ONLY AUDIT - READ ONLY
# ==========================================================

from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as ET
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import pandas as pd
import json
import re

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")
HUMAN = ROOT / "05_Human_Observer_Study/Final_Comparative_Analysis"
BASE = HUMAN / "AI_vs_Human_v1"
RESULTS = BASE / "Results"
IMG_DIR = ROOT / "01_Datasets/Prepared_6_Classes/v1_cleaned/test/images"
LBL_DIR = ROOT / "01_Datasets/Prepared_6_Classes/v1_cleaned/test/labels"

CLASSES = [
    "Caries", "Missing Teeth", "Root Canal Treatment",
    "Filling", "Crown", "Unerupted Wisdom Teeth"
]
COLORS = [
    "#FA3232", "#FFBE28", "#32C8FF",
    "#32DC6E", "#C864FF", "#FF6EC8"
]

mapping = pd.read_csv(BASE / "Verified_Test_Case_Mapping.csv")
lock = json.loads(
    (BASE / "AI_Validation_Confidence_Lock.json").read_text()
)

def read_cvat_boxes(zip_path):
    result = {}

    with ZipFile(zip_path) as archive:
        xml_files = [
            n for n in archive.namelist()
            if n.endswith("annotations.xml")
        ]
        assert len(xml_files) == 1, zip_path

        root = ET.fromstring(archive.read(xml_files[0]))

        for image in root.findall(".//image"):
            match = re.search(
                r"CASE_\d{4}", image.attrib["name"]
            )
            assert match is not None

            case_id = match.group()
            w = float(image.attrib["width"])
            h = float(image.attrib["height"])

            boxes = []
            for box in image.findall("box"):
                name = box.attrib["label"]
                assert name in CLASSES, name

                boxes.append((
                    CLASSES.index(name),
                    float(box.attrib["xtl"]) / w,
                    float(box.attrib["ytl"]) / h,
                    float(box.attrib["xbr"]) / w,
                    float(box.attrib["ybr"]) / h,
                    None
                ))

            assert case_id not in result
            result[case_id] = boxes

    assert len(result) == 154
    return result


READERS = {
    "Dentist 01": read_cvat_boxes(
        HUMAN / "Inputs/Dentist01_CVAT_Annotations.zip"
    ),
    "Dentist 02": read_cvat_boxes(
        HUMAN / "Inputs/Dentist02_CVAT_Annotations.zip.zip"
    )
}

AI = {}
for name in ["YOLOv8s", "YOLO11s", "RT-DETR-L"]:
    path = RESULTS / (
        name.replace("-", "_") + "_Test_Predictions.csv"
    )
    AI[name] = pd.read_csv(path)


def get_reference_boxes(filename):
    path = LBL_DIR / (Path(filename).stem + ".txt")
    boxes = []

    for line in path.read_text().splitlines():
        if not line.strip():
            continue

        cls, xc, yc, bw, bh = map(float, line.split())

        boxes.append((
            int(cls),
            xc - bw / 2,
            yc - bh / 2,
            xc + bw / 2,
            yc + bh / 2,
            None
        ))

    return boxes


def render_boxes(original, boxes):
    image = original.copy().convert("RGB")
    draw = ImageDraw.Draw(image)
    w, h = image.size

    for cls, x1, y1, x2, y2, confidence in boxes:
        cls = int(cls)

        xy = (
            int(max(0, x1 * w)),
            int(max(0, y1 * h)),
            int(min(w-1, x2 * w)),
            int(min(h-1, y2 * h))
        )

        draw.rectangle(xy, outline=COLORS[cls], width=3)

        label = CLASSES[cls]
        if confidence is not None:
            label += f" {confidence:.2f}"

        draw.text(
            (xy[0], max(0, xy[1]-13)),
            label,
            fill=COLORS[cls],
            stroke_width=1,
            stroke_fill="black"
        )

    return image


def review_case(case_id):
    row = mapping.loc[mapping.case_id == case_id]
    assert len(row) == 1, "Case not found"

    filename = row.iloc[0]["original_filename"]
    with Image.open(IMG_DIR / filename) as im:
        original = im.convert("RGB").copy()

    panels = [
        ("Original image", original),
        (
            "Dataset reference",
            render_boxes(original, get_reference_boxes(filename))
        )
    ]

    for dentist in ["Dentist 01", "Dentist 02"]:
        panels.append((
            dentist,
            render_boxes(original, READERS[dentist][case_id])
        ))

    for model in ["YOLOv8s", "YOLO11s", "RT-DETR-L"]:
        threshold = lock["selected_confidence"][model]
        df = AI[model]

        df = df[
            (df["case_id"] == case_id)
            & (df["confidence"] >= threshold)
        ]

        boxes = [
            (
                int(r.class_id),
                float(r.x1), float(r.y1),
                float(r.x2), float(r.y2),
                float(r.confidence)
            )
            for r in df.itertuples()
        ]

        panels.append((
            f"{model} | {len(boxes)} detections",
            render_boxes(original, boxes)
        ))

    fig, axes = plt.subplots(2, 4, figsize=(19, 11))

    for ax, (title, image) in zip(axes.flat, panels):
        ax.imshow(image)
        ax.set_title(title, fontsize=11)
        ax.axis("off")

    last = axes.flat[-1]
    last.axis("off")
    last.set_title("Class legend")

    for idx, name in enumerate(CLASSES):
        last.text(
            0.05, 0.92 - idx * 0.13,
            f"{idx} - {name}",
            color=COLORS[idx],
            fontsize=12,
            transform=last.transAxes
        )

    fig.suptitle(
        f"{case_id} | Researcher-only comparison",
        fontsize=17
    )
    plt.tight_layout()
    plt.show()


# Display one case at a time
review_case("CASE_0082")


In [ ]:

# ==========================================================
# STEP 12A - PREPARE 8 BLINDED IMAGES FOR EXPERT 03
# Creates a CVAT ZIP containing PNG images ONLY.
# Does not modify the original test dataset.
# ==========================================================

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from PIL import Image
from io import BytesIO
from hashlib import sha256
from google.colab import drive
import pandas as pd
import random
import json

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")

BASE = (
    ROOT / "05_Human_Observer_Study"
    / "Final_Comparative_Analysis/AI_vs_Human_v1"
)

RESULTS = BASE / "Results"
IMAGE_DIR = (
    ROOT / "01_Datasets/Prepared_6_Classes/"
    "v1_cleaned/test/images"
)
LABEL_DIR = (
    ROOT / "01_Datasets/Prepared_6_Classes/"
    "v1_cleaned/test/labels"
)

EXPERT_ROOT = (
    ROOT / "05_Human_Observer_Study"
    / "Final_Comparative_Analysis"
    / "Expert_8_Case_Adjudication"
)

ROUND1 = EXPERT_ROOT / "01_Expert_Round1_Blinded"
PRIVATE = EXPERT_ROOT / "02_Researcher_ONLY"
ROUND2 = EXPERT_ROOT / "03_Expert_Round2_After_Export"

for folder in [ROUND1, PRIVATE, ROUND2]:
    folder.mkdir(parents=True, exist_ok=True)

CLASSES = [
    "Caries",
    "Missing Teeth",
    "Root Canal Treatment",
    "Filling",
    "Crown",
    "Unerupted Wisdom Teeth"
]

COLORS = [
    "#FA3232",
    "#FFBE28",
    "#32C8FF",
    "#32DC6E",
    "#C864FF",
    "#FF6EC8"
]

audit = pd.read_csv(
    RESULTS / "Researcher_Disagreement_Priority.csv"
)

mapping = pd.read_csv(
    BASE / "Verified_Test_Case_Mapping.csv"
)

# Select all reference-empty images with >= 1 detection
flag = (
    audit["reference_empty_with_FP"]
    .astype(str)
    .str.lower()
    .eq("true")
)

selected = audit[
    flag & audit["reference_boxes"].eq(0)
].copy()

selected = selected.sort_values("case_id")

EXPECTED_CASES = {
    "CASE_0082",
    "CASE_0053",
    "CASE_0049",
    "CASE_0091",
    "CASE_0115",
    "CASE_0008",
    "CASE_0071",
    "CASE_0132"
}

assert len(selected) == 8
assert set(selected["case_id"]) == EXPECTED_CASES
assert selected["total_FP_across_evaluators"].sum() == 25

case_files = mapping.set_index(
    "case_id"
)["original_filename"].to_dict()

# Randomize presentation order, reproducibly
case_order = sorted(EXPECTED_CASES)

rng = random.Random(20261014)
rng.shuffle(case_order)

image_payloads = {}
private_map_rows = []

for position, case_id in enumerate(case_order, 1):

    original_filename = case_files[case_id]
    original_path = IMAGE_DIR / original_filename

    assert original_path.is_file(), original_path

    label_path = (
        LABEL_DIR
        / f"{Path(original_filename).stem}.txt"
    )

    assert label_path.is_file()

    # These eight cases must have zero target boxes
    assert not label_path.read_text().strip()

    cvat_name = f"E3_{position:04d}_{case_id}.png"

    with Image.open(original_path) as im:
        original = im.convert("RGB")

        assert original.size == (640, 640)

        memory = BytesIO()
        original.save(
            memory,
            format="PNG",
            compress_level=6
        )

    image_payloads[cvat_name] = memory.getvalue()

    private_map_rows.append({
        "case_id": case_id,
        "expert_filename": cvat_name,
        "original_filename": original_filename
    })

# ZIP files must contain only the eight PNG images.
def write_zip_without_overwrite(path, content):

    if path.exists():

        with ZipFile(path) as z:
            assert set(z.namelist()) == set(content)

            for filename, expected_bytes in content.items():
                assert (
                    sha256(z.read(filename)).hexdigest()
                    == sha256(expected_bytes).hexdigest()
                ), f"Existing file differs: {filename}"

        print("Existing verified ZIP retained:", path.name)

    else:

        with ZipFile(
            path,
            mode="x",
            compression=ZIP_DEFLATED
        ) as z:

            for filename, data in content.items():
                z.writestr(filename, data)

        print("Created ZIP:", path.name)

    with ZipFile(path) as z:
        assert z.testzip() is None

CVAT_ZIP = (
    ROUND1 / "Expert03_Round1_CVAT_8_Images_ONLY.zip"
)

write_zip_without_overwrite(
    CVAT_ZIP,
    image_payloads
)

# Private mapping - do not share with expert
PRIVATE_MAPPING = (
    PRIVATE / "Expert03_Private_Image_Mapping.csv"
)

private_mapping_df = pd.DataFrame(private_map_rows)

if PRIVATE_MAPPING.exists():
    old = pd.read_csv(PRIVATE_MAPPING)
    assert old.equals(private_mapping_df)
else:
    private_mapping_df.to_csv(
        PRIVATE_MAPPING,
        index=False
    )

# Case-level completion log, shareable with expert
CASE_LOG = ROUND1 / "Expert03_Round1_Case_Log.csv"

case_log = private_mapping_df[
    ["case_id", "expert_filename"]
].copy()

case_log["image_reviewed"] = ""
case_log["any_target_finding"] = ""
case_log["image_adequacy"] = ""
case_log["clinical_notes"] = ""

if not CASE_LOG.exists():
    case_log.to_csv(CASE_LOG, index=False)

print("\n" + "=" * 65)
print("STEP 12A - BLINDED CVAT PACKAGE REPORT")
print("=" * 65)

print("Images:", len(image_payloads))
print("Reference boxes:", 0)
print("Images with annotation overlays:", 0)
print("Original image dimensions:", "640 x 640")
print("Total flagged detections:", 25)
print("CVAT ZIP:", CVAT_ZIP)
print("Completion log:", CASE_LOG)
print("Private mapping:", PRIVATE_MAPPING)

print("\nSTATUS: EXPERT 03 ROUND 1 PACKAGE READY")


In [ ]:

# ==========================================================
# STEP 12B - EXPERT ADJUDICATION MATERIALS
# 1. Private five-evaluator comparison panels
# 2. Source-blinded individual candidate images
# 3. Expert decision CSV
# NO RETRAINING - NO CHANGES TO ORIGINAL ANNOTATIONS
# ==========================================================

from xml.etree import ElementTree as ET
from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
from zipfile import ZipFile
import pandas as pd
import random
import json
import re

INPUTS = (
    ROOT / "05_Human_Observer_Study"
    / "Final_Comparative_Analysis/Inputs"
)

READER_FILES = {
    "Dentist 01": (
        INPUTS / "Dentist01_CVAT_Annotations.zip"
    ),
    "Dentist 02": (
        INPUTS / "Dentist02_CVAT_Annotations.zip.zip"
    )
}

LOCK = json.loads(
    (
        BASE / "AI_Validation_Confidence_Lock.json"
    ).read_text()
)

MODEL_NAMES = [
    "YOLOv8s",
    "YOLO11s",
    "RT-DETR-L"
]

try:
    FONT = ImageFont.truetype(
        "/usr/share/fonts/truetype/dejavu/"
        "DejaVuSans-Bold.ttf",
        16
    )
except OSError:
    FONT = ImageFont.load_default()


# ----------------------------------------------------------
# 1. Read original doctor annotations
# ----------------------------------------------------------

def read_reader_annotations(zip_path):

    result = {}

    with ZipFile(zip_path) as archive:

        xml_names = [
            n for n in archive.namelist()
            if Path(n).name == "annotations.xml"
        ]

        assert len(xml_names) == 1, zip_path

        root = ET.fromstring(
            archive.read(xml_names[0])
        )

        image_nodes = root.findall(".//image")

        assert len(image_nodes) == 154

        for image in image_nodes:

            match = re.search(
                r"CASE_\d{4}",
                image.attrib["name"]
            )

            assert match is not None

            case_id = match.group()

            width = float(image.attrib["width"])
            height = float(image.attrib["height"])

            boxes = []

            for child in image:

                if child.tag in {
                    "polygon", "mask", "polyline"
                }:
                    raise ValueError(
                        f"Unexpected shape in {case_id}"
                    )

                if child.tag != "box":
                    continue

                label = child.attrib["label"]

                assert label in CLASSES, label

                boxes.append({
                    "class_id": CLASSES.index(label),
                    "x1": float(child.attrib["xtl"]) / width,
                    "y1": float(child.attrib["ytl"]) / height,
                    "x2": float(child.attrib["xbr"]) / width,
                    "y2": float(child.attrib["ybr"]) / height,
                    "confidence": None
                })

            assert case_id not in result
            result[case_id] = boxes

    assert len(result) == 154

    return result


doctor_annotations = {
    name: read_reader_annotations(path)
    for name, path in READER_FILES.items()
}

# ----------------------------------------------------------
# 2. Load locked model detections
# ----------------------------------------------------------

model_annotations = {}

for name in MODEL_NAMES:

    path = RESULTS / (
        name.replace("-", "_")
        + "_Test_Predictions.csv"
    )

    assert path.is_file(), path

    df = pd.read_csv(path)

    confidence_threshold = float(
        LOCK["selected_confidence"][name]
    )

    df = df[
        df["confidence"] >= confidence_threshold
    ].copy()

    model_annotations[name] = {}

    for case_id, group in df.groupby("case_id"):

        model_annotations[name][case_id] = [
            {
                "class_id": int(r.class_id),
                "x1": float(r.x1),
                "y1": float(r.y1),
                "x2": float(r.x2),
                "y2": float(r.y2),
                "confidence": float(r.confidence)
            }
            for r in group.itertuples()
        ]


# ----------------------------------------------------------
# 3. Drawing functions
# ----------------------------------------------------------

def annotated_image(original, boxes, show_scores=False):

    image = original.copy().convert("RGB")
    draw = ImageDraw.Draw(image)

    width, height = image.size

    for box in boxes:

        cls = int(box["class_id"])

        xy = [
            max(0, int(box["x1"] * width)),
            max(0, int(box["y1"] * height)),
            min(width - 1, int(box["x2"] * width)),
            min(height - 1, int(box["y2"] * height))
        ]

        draw.rectangle(
            xy,
            outline=COLORS[cls],
            width=3
        )

        label = CLASSES[cls]

        if (
            show_scores
            and box.get("confidence") is not None
        ):
            label += (
                f" {box['confidence']:.2f}"
            )

        ty = max(0, xy[1] - 19)

        draw.text(
            (xy[0], ty),
            label,
            font=FONT,
            fill=COLORS[cls],
            stroke_width=2,
            stroke_fill="black"
        )

    return image


def png_bytes(image):

    buffer = BytesIO()
    image.save(
        buffer,
        format="PNG",
        compress_level=6
    )

    return buffer.getvalue()


def create_contact_sheet(panels):

    tile_w = 360
    tile_h = 390

    board = Image.new(
        "RGB",
        (tile_w * 4, tile_h * 2),
        "white"
    )

    draw = ImageDraw.Draw(board)

    for idx, (title, image) in enumerate(panels):

        x = (idx % 4) * tile_w
        y = (idx // 4) * tile_h

        thumbnail = image.resize(
            (360, 360),
            Image.Resampling.LANCZOS
        )

        board.paste(
            thumbnail,
            (x, y + 28)
        )

        draw.text(
            (x + 8, y + 5),
            title,
            fill="black",
            font=FONT
        )

    return board


# ----------------------------------------------------------
# 4. Build private comparison images
# ----------------------------------------------------------

private_zip_contents = {}
round2_zip_contents = {}

candidate_rows = []
private_candidate_rows = []

expected_fp = selected.set_index(
    "case_id"
)["total_FP_across_evaluators"].to_dict()

for case_id in sorted(EXPECTED_CASES):

    filename = case_files[case_id]

    with Image.open(IMAGE_DIR / filename) as im:
        original = im.convert("RGB").copy()

    sources = {
        "Dentist 01": doctor_annotations[
            "Dentist 01"
        ].get(case_id, []),

        "Dentist 02": doctor_annotations[
            "Dentist 02"
        ].get(case_id, []),

        "YOLOv8s": model_annotations[
            "YOLOv8s"
        ].get(case_id, []),

        "YOLO11s": model_annotations[
            "YOLO11s"
        ].get(case_id, []),

        "RT-DETR-L": model_annotations[
            "RT-DETR-L"
        ].get(case_id, [])
    }

    # All source boxes are "FP" against the empty reference
    total_detections = sum(
        len(boxes) for boxes in sources.values()
    )

    assert total_detections == int(
        expected_fp[case_id]
    ), f"Detection count mismatch: {case_id}"

    # Create private side-by-side panels
    panels = [
        ("Original", original),
        (
            "Dataset reference - 0 boxes",
            original.copy()
        )
    ]

    for source_name, boxes in sources.items():

        marked = annotated_image(
            original,
            boxes,
            show_scores=True
        )

        panels.append((
            f"{source_name} - {len(boxes)} boxes",
            marked
        ))

        private_zip_contents[
            f"{case_id}/{source_name.replace(' ', '_')}.png"
        ] = png_bytes(marked)

    contact_sheet = create_contact_sheet(panels)

    private_zip_contents[
        f"COMPARISON_{case_id}.png"
    ] = png_bytes(contact_sheet)

    private_zip_contents[
        f"{case_id}/Original.png"
    ] = png_bytes(original)

    private_zip_contents[
        f"{case_id}/Dataset_Reference_Empty.png"
    ] = png_bytes(original)

    # Collect all detection proposals, then shuffle their sources
    proposals = []

    for source_name, boxes in sources.items():

        for box in boxes:
            proposals.append((
                source_name,
                box
            ))

    random.Random(
        20261014 + int(case_id[-4:])
    ).shuffle(proposals)

    for proposal_number, (source_name, box) in enumerate(
        proposals, 1
    ):

        candidate_id = (
            f"{case_id}_P{proposal_number:02d}"
        )

        cls = int(box["class_id"])

        # Expert receives one anonymous proposal per image
        single_marked = annotated_image(
            original,
            [box],
            show_scores=False
        )

        preview = Image.new(
            "RGB",
            (1280, 680),
            "white"
        )

        preview_draw = ImageDraw.Draw(preview)

        preview.paste(original, (0, 40))
        preview.paste(single_marked, (640, 40))

        preview_draw.text(
            (12, 8),
            f"{case_id} | ORIGINAL",
            font=FONT,
            fill="black"
        )

        preview_draw.text(
            (652, 8),
            f"{candidate_id} | {CLASSES[cls]}",
            font=FONT,
            fill="black"
        )

        round2_zip_contents[
            f"Candidate_Images/{candidate_id}.png"
        ] = png_bytes(preview)

        # Expert-visible decision form: NO source identity
        candidate_rows.append({
            "case_id": case_id,
            "candidate_id": candidate_id,
            "proposed_class": CLASSES[cls],
            "decision": "",
            "correct_class_if_changed": "",
            "box_adjustment_needed": "",
            "duplicate_of_candidate_id": "",
            "already_marked_in_round1": "",
            "clinical_reason": ""
        })

        # Researcher-only source mapping
        private_candidate_rows.append({
            "case_id": case_id,
            "candidate_id": candidate_id,
            "source": source_name,
            "class_id": cls,
            "class_name": CLASSES[cls],
            "confidence": box.get("confidence"),
            "x1": box["x1"],
            "y1": box["y1"],
            "x2": box["x2"],
            "y2": box["y2"]
        })

assert len(candidate_rows) == 25
assert len(private_candidate_rows) == 25

# ----------------------------------------------------------
# 5. Create decision forms and confidential source key
# ----------------------------------------------------------

DECISIONS_CSV = (
    ROUND2 / "Expert03_Round2_Decisions.csv"
)

decisions_df = pd.DataFrame(candidate_rows)

if DECISIONS_CSV.exists():
    # Preserve the expert's answers if file was edited
    print("Existing decision form retained.")
else:
    decisions_df.to_csv(
        DECISIONS_CSV,
        index=False,
        encoding="utf-8-sig"
    )

SOURCE_KEY = (
    PRIVATE / "Private_Candidate_Source_Key.csv"
)

source_key_df = pd.DataFrame(
    private_candidate_rows
)

if SOURCE_KEY.exists():

    old = pd.read_csv(SOURCE_KEY)

    assert old["candidate_id"].tolist() == (
        source_key_df["candidate_id"].tolist()
    )

else:

    source_key_df.to_csv(
        SOURCE_KEY,
        index=False,
        encoding="utf-8-sig"
    )

# ----------------------------------------------------------
# 6. Round 2 expert instructions
# ----------------------------------------------------------

expert_instructions = """
EXPERT 03 - ROUND 2 ANONYMOUS CANDIDATE REVIEW

IMPORTANT:
Complete and export your independent CVAT annotation
of all eight images BEFORE opening these proposal images.

The proposals are anonymous.
They may originate from different observers or systems.
Do not infer the source of any proposal.

For each candidate, fill Expert03_Round2_Decisions.csv.

DECISION OPTIONS:
Accept     = Finding is supported by the radiograph.
Reject     = Finding is not supported by the radiograph.
Uncertain  = Cannot determine reliably.
Duplicate  = Same finding as another candidate.

If Accept:
- Confirm or correct the proposed class.
- State whether the box needs adjustment.
- Indicate if the finding was already marked in Round 1.

If Duplicate:
- Enter the ID of the matching candidate in
  duplicate_of_candidate_id.

If Uncertain:
- Explain why the radiograph is insufficient.

Do not force a positive diagnosis.
Findings outside the six study classes are out of scope.

Finally, complete a SEPARATE FINAL CVAT task
using the eight original unmarked images, reflecting
the final adjudicated annotations. Do not overwrite
the saved Round 1 export.

This review covers eight selected cases only.
It is not independent verification of all 154 test images.
""".strip()

round2_zip_contents[
    "README_Expert_Round2.txt"
] = expert_instructions.encode("utf-8")

round2_zip_contents[
    "Expert03_Round2_Decisions.csv"
] = decisions_df.to_csv(
    index=False
).encode("utf-8-sig")

# Do not share this key with the expert
private_zip_contents[
    "PRIVATE_Candidate_Source_Key.csv"
] = source_key_df.to_csv(
    index=False
).encode("utf-8-sig")

# ----------------------------------------------------------
# 7. Save ZIPs without overwriting existing outputs
# ----------------------------------------------------------

PRIVATE_ZIP = (
    PRIVATE / "Researcher_ONLY_All_Comparisons.zip"
)

ROUND2_ZIP = (
    ROUND2 / "Expert03_Round2_Anonymous_Proposals.zip"
)

write_zip_without_overwrite(
    PRIVATE_ZIP,
    private_zip_contents
)

write_zip_without_overwrite(
    ROUND2_ZIP,
    round2_zip_contents
)

# ----------------------------------------------------------
# 8. Final QC
# ----------------------------------------------------------

with ZipFile(CVAT_ZIP) as z:
    assert len(z.namelist()) == 8
    assert all(
        n.endswith(".png") for n in z.namelist()
    )

with ZipFile(ROUND2_ZIP) as z:
    proposal_images = [
        n for n in z.namelist()
        if n.startswith("Candidate_Images/")
        and n.endswith(".png")
    ]
    assert len(proposal_images) == 25

print("\n" + "=" * 65)
print("STEP 12B - EXPERT MATERIALS COMPLETED")
print("=" * 65)

print("Cases:", len(EXPECTED_CASES))
print("Anonymous detection proposals:", len(candidate_rows))
print("Private researcher panels:", 8)
print("CVAT ZIP:", CVAT_ZIP)
print("Researcher comparison ZIP:", PRIVATE_ZIP)
print("Anonymous proposal ZIP:", ROUND2_ZIP)
print("Expert decision form:", DECISIONS_CSV)
print("Confidential source key:", SOURCE_KEY)

print("\nSTATUS: EXPERT 03 ALL PACKAGES READY")


In [ ]:

# ==========================================================
# STEP 13A - PREPARE MANUSCRIPT TABLES AND FIGURES
# Existing results only - No GPU or model inference
# ==========================================================

from pathlib import Path
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026"
)

RESULTS = (
    ROOT / "05_Human_Observer_Study"
    / "Final_Comparative_Analysis"
    / "AI_vs_Human_v1/Results"
)

OUTPUT = ROOT / "06_Manuscript"
TABLES = OUTPUT / "Tables"
FIGURES = OUTPUT / "Figures"

TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

summary = pd.read_csv(
    RESULTS / "AI_vs_Human_Combined_Summary.csv"
)

per_class = pd.read_csv(
    RESULTS / "AI_vs_Human_Combined_Per_Class.csv"
)

bootstrap = pd.read_csv(
    RESULTS / "AI_vs_Human_Paired_Bootstrap.csv"
)

ORDER = [
    "Dentist 01",
    "Dentist 02",
    "YOLOv8s",
    "YOLO11s",
    "RT-DETR-L"
]

CLASS_ORDER = [
    "Caries",
    "Missing Teeth",
    "Root Canal Treatment",
    "Filling",
    "Crown",
    "Unerupted Wisdom Teeth"
]

assert len(summary) == 10
assert len(per_class) == 60
assert set(summary["participant"]) == set(ORDER)

# ----------------------------------------------------------
# TABLE 1 - Primary comparative performance
# ----------------------------------------------------------

primary = summary[
    np.isclose(summary["threshold"], 0.50)
].copy()

assert len(primary) == 5
assert (primary["TP"] + primary["FN"] == 588).all()

primary["Evaluator"] = primary["participant"]
primary["Confidence threshold"] = primary["confidence"]

for metric in ["precision", "recall", "F1"]:
    primary[metric] = (
        primary[metric] * 100
    ).round(2)

table1 = primary[
    [
        "Evaluator",
        "Confidence threshold",
        "TP", "FP", "FN",
        "precision", "recall", "F1"
    ]
].rename(columns={
    "precision": "Precision (%)",
    "recall": "Recall (%)",
    "F1": "Micro F1 (%)"
})

table1["Evaluator"] = pd.Categorical(
    table1["Evaluator"],
    categories=ORDER,
    ordered=True
)

table1 = table1.sort_values("Evaluator")

table1.to_csv(
    TABLES / "Table_1_Primary_Performance.csv",
    index=False
)

# ----------------------------------------------------------
# TABLE 2 - Per-class performance at IoU 0.50
# ----------------------------------------------------------

pc = per_class[
    np.isclose(per_class["threshold"], 0.50)
].copy()

table2 = pc.pivot(
    index="class_name",
    columns="participant",
    values="F1"
)

table2 = (
    table2.loc[CLASS_ORDER, ORDER] * 100
).round(2)

table2.to_csv(
    TABLES / "Table_2_Per_Class_F1.csv"
)

# ----------------------------------------------------------
# TABLE 3 - Sensitivity analysis (IoU 0.25)
# ----------------------------------------------------------

exploratory = summary[
    np.isclose(summary["threshold"], 0.25)
].copy()

assert len(exploratory) == 5

table3 = exploratory[
    ["participant", "precision", "recall", "F1"]
].copy()

for metric in ["precision", "recall", "F1"]:
    table3[metric] = (
        table3[metric] * 100
    ).round(2)

table3.columns = [
    "Evaluator",
    "Precision (%)",
    "Recall (%)",
    "Micro F1 (%)"
]

table3["Evaluator"] = pd.Categorical(
    table3["Evaluator"],
    categories=ORDER,
    ordered=True
)

table3 = table3.sort_values("Evaluator")

table3.to_csv(
    TABLES / "Table_3_Exploratory_IoU025.csv",
    index=False
)

# ----------------------------------------------------------
# TABLE 4 - Paired bootstrap confidence intervals
# ----------------------------------------------------------

boot_primary = bootstrap[
    np.isclose(bootstrap["threshold"], 0.50)
].copy()

for metric in [
    "B_minus_A_F1",
    "ci_low",
    "ci_high"
]:
    boot_primary[metric] = (
        boot_primary[metric] * 100
    ).round(2)

boot_primary.to_csv(
    TABLES / "Table_4_Paired_Bootstrap.csv",
    index=False
)

# ----------------------------------------------------------
# FIGURE 1 - Main F1 comparison
# ----------------------------------------------------------

plot_df = summary.pivot(
    index="participant",
    columns="threshold",
    values="F1"
).loc[ORDER]

x = np.arange(len(ORDER))
width = 0.36

fig, ax = plt.subplots(figsize=(11, 6))

bars1 = ax.bar(
    x - width / 2,
    plot_df[0.50] * 100,
    width,
    label="Primary: IoU 0.50"
)

bars2 = ax.bar(
    x + width / 2,
    plot_df[0.25] * 100,
    width,
    label="Exploratory: IoU 0.25"
)

ax.bar_label(bars1, fmt="%.1f", padding=3)
ax.bar_label(bars2, fmt="%.1f", padding=3)

ax.set_xticks(x, ORDER)
ax.set_ylabel("Micro F1-score (%)")
ax.set_ylim(0, 105)
ax.set_title(
    "AI Models vs Human Observers\n"
    "Comparison Against Dataset Annotations"
)

ax.legend()
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

fig.savefig(
    FIGURES / "Figure_1_Five_Way_F1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ----------------------------------------------------------
# DOCUMENT STUDY LIMITATIONS
# ----------------------------------------------------------

notes = """
MANUSCRIPT ANALYSIS NOTES

Primary analysis:
- 154 internal test radiographs
- Six dental finding categories
- 588 existing dataset reference boxes
- Matching IoU >= 0.50
- Locked model confidence thresholds from validation

Exploratory analysis:
- Matching IoU >= 0.25
- Post-hoc sensitivity analysis

Statistical comparison:
- Paired image-level bootstrap
- 4000 bootstrap replicates
- Nominal 95% confidence intervals
- No multiple-comparison adjustment

Reference limitation:
- Dataset annotations are not an independently
  established clinical diagnostic ground truth.
- Eight reference-empty images underwent targeted
  expert review.
- This review does not validate all 154 images.
- Primary dataset-reference results remain unchanged.
- Patient-level independence has not been fully verified.

Interpret results as annotation-based detection
performance, not definitive clinical diagnostic accuracy.
""".strip()

(OUTPUT / "Analysis_Notes.txt").write_text(
    notes,
    encoding="utf-8"
)

print("\n" + "=" * 65)
print("STEP 13A - MANUSCRIPT MATERIALS")
print("=" * 65)

print("\nTABLE 1 - PRIMARY PERFORMANCE")
display(table1)

print("\nTABLE 2 - PER-CLASS F1")
display(table2)

print("\nTABLE 3 - EXPLORATORY PERFORMANCE")
display(table3)

print("\nFiles saved to:", OUTPUT)

print("\nSTATUS: MANUSCRIPT TABLES READY")


In [ ]:

# ==========================================================
# STEP 13B - ADDITIONAL MANUSCRIPT TABLES
# Existing study data only; no model inference
# ==========================================================

from pathlib import Path
from google.colab import drive
import numpy as np
import pandas as pd
import json

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")
HUMAN = ROOT / "05_Human_Observer_Study/Final_Comparative_Analysis"
AI_BASE = HUMAN / "AI_vs_Human_v1"
RESULTS = AI_BASE / "Results"
TABLES = ROOT / "06_Manuscript/Tables"
FIGURES = ROOT / "06_Manuscript/Figures"

TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

NAMES = [
    "Caries",
    "Missing Teeth",
    "Root Canal Treatment",
    "Filling",
    "Crown",
    "Unerupted Wisdom Teeth"
]

PARTICIPANTS = [
    "Dentist 01", "Dentist 02",
    "YOLOv8s", "YOLO11s", "RT-DETR-L"
]

summary = pd.read_csv(
    RESULTS / "AI_vs_Human_Combined_Summary.csv"
)
per_class = pd.read_csv(
    RESULTS / "AI_vs_Human_Combined_Per_Class.csv"
)
bootstrap = pd.read_csv(
    RESULTS / "AI_vs_Human_Paired_Bootstrap.csv"
)

assert len(summary) == 10
assert len(per_class) == 60
assert len(bootstrap) == 20

# ----------------------------------------------------------
# TABLE 5 - Test set distribution
# ----------------------------------------------------------

label_dir = (
    ROOT / "01_Datasets/Prepared_6_Classes"
    / "v1_cleaned/test/labels"
)

label_files = sorted(label_dir.glob("*.txt"))
assert len(label_files) == 154

instances = np.zeros(6, dtype=int)
image_counts = np.zeros(6, dtype=int)
empty_images = 0

for path in label_files:

    class_ids = []

    for line in path.read_text().splitlines():
        if line.strip():
            parts = line.split()
            assert len(parts) == 5
            cls = int(parts[0])
            assert 0 <= cls < 6
            class_ids.append(cls)
            instances[cls] += 1

    if not class_ids:
        empty_images += 1

    for cls in set(class_ids):
        image_counts[cls] += 1

assert int(instances.sum()) == 588
assert empty_images == 23

table5 = pd.DataFrame({
    "Dental finding": NAMES,
    "Annotated instances": instances,
    "Images containing finding": image_counts,
    "Percentage of all 588 boxes": np.round(
        instances / 588 * 100, 2
    )
})

table5.to_csv(
    TABLES / "Table_5_Test_Class_Distribution.csv",
    index=False
)

# ----------------------------------------------------------
# TABLE 6 - Reader annotation quality control
# ----------------------------------------------------------

table6 = pd.read_csv(
    HUMAN / "Results/QC_Summary.csv"
)

assert set(table6["reader"]) == {
    "Dentist 01", "Dentist 02"
}

table6.to_csv(
    TABLES / "Table_6_Reader_Annotation_QC.csv",
    index=False
)

# ----------------------------------------------------------
# TABLE 7 - AI versus dentists, primary IoU 0.50
# Report AI minus dentist consistently
# ----------------------------------------------------------

comparisons = []

for model in ["YOLOv8s", "YOLO11s", "RT-DETR-L"]:
    for dentist in ["Dentist 01", "Dentist 02"]:

        hit = bootstrap[
            np.isclose(bootstrap["threshold"], 0.50)
            & (
                (
                    (bootstrap["A"] == model)
                    & (bootstrap["B"] == dentist)
                )
                |
                (
                    (bootstrap["A"] == dentist)
                    & (bootstrap["B"] == model)
                )
            )
        ]

        assert len(hit) == 1
        item = hit.iloc[0]

        difference = float(item["B_minus_A_F1"])
        low = float(item["ci_low"])
        high = float(item["ci_high"])

        if item["B"] == dentist:
            difference = -difference
            low, high = -high, -low

        comparisons.append({
            "AI model": model,
            "Human observer": dentist,
            "AI minus human F1 (pp)": round(
                100 * difference, 2
            ),
            "95% CI lower (pp)": round(100 * low, 2),
            "95% CI upper (pp)": round(100 * high, 2),
            "CI excludes zero": bool(
                low > 0 or high < 0
            )
        })

table7 = pd.DataFrame(comparisons)

table7.to_csv(
    TABLES / "Table_7_AI_vs_Human_F1_Differences.csv",
    index=False
)

# ----------------------------------------------------------
# TABLE 8 - F1 sensitivity to localization threshold
# ----------------------------------------------------------

pivot = summary.pivot(
    index="participant",
    columns="threshold",
    values="F1"
).loc[PARTICIPANTS]

table8 = pd.DataFrame({
    "Evaluator": PARTICIPANTS,
    "F1 IoU 0.50 (%)": np.round(
        pivot[0.50].values * 100, 2
    ),
    "F1 IoU 0.25 (%)": np.round(
        pivot[0.25].values * 100, 2
    )
})

table8["F1 change (pp)"] = (
    table8["F1 IoU 0.25 (%)"]
    - table8["F1 IoU 0.50 (%)"]
).round(2)

table8.to_csv(
    TABLES / "Table_8_IoU_Sensitivity.csv",
    index=False
)

# ----------------------------------------------------------
# TABLE 9 - Validation confidence calibration
# ----------------------------------------------------------

validation = pd.read_csv(
    AI_BASE / "AI_Validation_Confidence_Grid.csv"
)

lock = json.loads(
    (
        AI_BASE / "AI_Validation_Confidence_Lock.json"
    ).read_text()
)

rows = []

for model in ["YOLOv8s", "YOLO11s", "RT-DETR-L"]:

    conf = float(
        lock["selected_confidence"][model]
    )

    selected = validation[
        (validation["model"] == model)
        & np.isclose(validation["confidence"], conf)
    ]

    assert len(selected) == 1

    rows.append({
        "Model": model,
        "Locked confidence": conf,
        "Validation F1 (%)": round(
            100 * selected.iloc[0]["F1"], 2
        )
    })

table9 = pd.DataFrame(rows)

table9.to_csv(
    TABLES / "Table_9_Validation_Confidence_Lock.csv",
    index=False
)

print("=" * 65)
print("STEP 13B - ADDITIONAL TABLES COMPLETED")
print("=" * 65)

for number, table in [
    (5, table5),
    (6, table6),
    (7, table7),
    (8, table8),
    (9, table9)
]:
    print(f"\nTABLE {number}")
    display(table)

print("\nNew tables saved:", TABLES)
print("\nSTATUS: ADDITIONAL MANUSCRIPT TABLES READY")


In [ ]:

# ==========================================================
# STEP 13C - PUBLICATION-QUALITY STATISTICAL FIGURES
# Outputs: PNG at 300 DPI + scalable SVG
# No training, no test-set recalibration
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "svg.fonttype": "none"
})

def save_figure(fig, filename):

    for ext in ("png", "svg"):
        target = FIGURES / f"{filename}.{ext}"

        fig.savefig(
            target,
            dpi=300,
            bbox_inches="tight",
            facecolor="white"
        )

        print("Saved:", target.name)

    plt.show()
    plt.close(fig)


# ----------------------------------------------------------
# FIGURE 2 - Per-class F1 heatmap
# ----------------------------------------------------------

heatmap = per_class[
    np.isclose(per_class["threshold"], 0.50)
].pivot(
    index="class_name",
    columns="participant",
    values="F1"
)

heatmap = (
    heatmap.loc[NAMES, PARTICIPANTS] * 100
)

fig, ax = plt.subplots(figsize=(11, 6.5))

im = ax.imshow(
    heatmap.values,
    cmap="viridis",
    vmin=0,
    vmax=100,
    aspect="auto"
)

ax.set_xticks(
    np.arange(len(PARTICIPANTS)),
    PARTICIPANTS
)

ax.set_yticks(
    np.arange(len(NAMES)),
    NAMES
)

plt.setp(
    ax.get_xticklabels(),
    rotation=20,
    ha="right"
)

for i in range(len(NAMES)):
    for j in range(len(PARTICIPANTS)):

        value = heatmap.iloc[i, j]

        ax.text(
            j, i,
            f"{value:.1f}",
            ha="center",
            va="center",
            color="white" if value < 55 else "black",
            fontsize=10,
            fontweight="bold"
        )

fig.colorbar(
    im,
    ax=ax,
    label="Micro F1-score within class (%)"
)

ax.set_title(
    "Per-Class Detection Performance\n"
    "Primary Matching Criterion: IoU >= 0.50"
)

fig.tight_layout()

save_figure(
    fig,
    "Figure_2_Per_Class_F1_Heatmap"
)


# ----------------------------------------------------------
# FIGURE 3 - Paired bootstrap forest plot
# ----------------------------------------------------------

labels = [
    f"{r['AI model']} vs {r['Human observer']}"
    for _, r in table7.iterrows()
]

estimate = table7[
    "AI minus human F1 (pp)"
].to_numpy()

low = table7[
    "95% CI lower (pp)"
].to_numpy()

high = table7[
    "95% CI upper (pp)"
].to_numpy()

y = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(11, 6))

ax.errorbar(
    estimate,
    y,
    xerr=[
        estimate - low,
        high - estimate
    ],
    fmt="o",
    capsize=5,
    markersize=7,
    linewidth=1.8
)

ax.axvline(
    0,
    linestyle="--",
    linewidth=1.2
)

ax.set_yticks(y, labels)
ax.invert_yaxis()

ax.set_xlabel(
    "F1 difference: AI minus dentist (percentage points)"
)

ax.set_title(
    "Paired Image-Level Bootstrap Comparisons\n"
    "Primary IoU >= 0.50 | 4,000 Replicates"
)

ax.grid(axis="x", alpha=0.2)
fig.tight_layout()

save_figure(
    fig,
    "Figure_3_AI_Human_Bootstrap_Forest"
)


# ----------------------------------------------------------
# FIGURE 4 - Precision-recall operating points
# ----------------------------------------------------------

primary = summary[
    np.isclose(summary["threshold"], 0.50)
].set_index("participant").loc[PARTICIPANTS]

fig, ax = plt.subplots(figsize=(9.5, 7))

for i, participant in enumerate(PARTICIPANTS):

    x = primary.loc[participant, "precision"] * 100
    y = primary.loc[participant, "recall"] * 100

    marker = "o" if participant.startswith(
        "Dentist"
    ) else "s"

    ax.scatter(
        x, y,
        s=125,
        marker=marker,
        label=participant
    )

    ax.annotate(
        f"{participant}\n({x:.1f}, {y:.1f})",
        (x, y),
        xytext=(7, 8 + (i % 2) * 14),
        textcoords="offset points",
        fontsize=9
    )

ax.set_xlim(50, 82)
ax.set_ylim(48, 73)

ax.set_xlabel("Precision (%)")
ax.set_ylabel("Recall (%)")

ax.set_title(
    "Precision–Recall Operating Points\n"
    "At Locked Confidence and IoU >= 0.50"
)

ax.grid(alpha=0.25)
fig.tight_layout()

save_figure(
    fig,
    "Figure_4_Precision_Recall_Operating_Points"
)


# ----------------------------------------------------------
# FIGURE 5 - True positives, false positives, false negatives
# ----------------------------------------------------------

counts = primary[
    ["TP", "FP", "FN"]
]

x = np.arange(len(PARTICIPANTS))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))

for idx, metric in enumerate(["TP", "FP", "FN"]):

    bars = ax.bar(
        x + (idx - 1) * width,
        counts[metric].values,
        width,
        label=metric
    )

    ax.bar_label(
        bars,
        fontsize=9,
        padding=2
    )

ax.set_xticks(x, PARTICIPANTS)
ax.set_ylabel("Number of bounding boxes")
ax.set_ylim(
    0,
    counts.to_numpy().max() * 1.2
)

ax.set_title(
    "Detection Outcomes Across Five Evaluators\n"
    "Dataset Reference | IoU >= 0.50"
)

ax.legend()
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()

save_figure(
    fig,
    "Figure_5_TP_FP_FN_Comparison"
)

print("\nSTATUS: STATISTICAL FIGURES READY")


In [ ]:

# ==========================================================
# STEP 13D - STUDY FLOW, DATA DISTRIBUTION,
# AND REAL RADIOGRAPHIC EXAMPLES
# ==========================================================

from matplotlib.patches import FancyBboxPatch
from PIL import Image
from io import BytesIO
from zipfile import ZipFile
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------------------------------------
# FIGURE 6 - Study workflow
# ----------------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 10)
ax.axis("off")

def box(x, y, width, height, content):

    patch = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle="round,pad=0.09",
        linewidth=1.5,
        edgecolor="#30425B",
        facecolor="#EAF0F7"
    )

    ax.add_patch(patch)

    ax.text(
        x + width / 2,
        y + height / 2,
        content,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="medium"
    )


def arrow(x1, y1, x2, y2):

    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(
            arrowstyle="->",
            linewidth=1.6,
            color="#30425B"
        )
    )


box(
    4.0, 8.9, 4.0, 0.75,
    "Frozen Dental OPG Dataset\n1,522 radiographs"
)

box(0.5, 6.7, 3.0, 0.9, "Training set\n1,065 images")
box(4.5, 6.7, 3.0, 0.9, "Validation set\n303 images")
box(8.5, 6.7, 3.0, 0.9, "Internal test set\n154 images; 588 boxes")

arrow(6, 8.9, 2, 7.6)
arrow(6, 8.9, 6, 7.6)
arrow(6, 8.9, 10, 7.6)

box(
    0.5, 4.8, 3.0, 1.0,
    "Model training\nYOLOv8s, YOLO11s, RT-DETR-L"
)

box(
    4.5, 4.8, 3.0, 1.0,
    "Validation calibration\nConfidence thresholds locked"
)

box(
    8.5, 4.8, 3.0, 1.0,
    "Blinded human evaluation\nTwo dentists"
)

arrow(2, 6.7, 2, 5.8)
arrow(6, 6.7, 6, 5.8)
arrow(10, 6.7, 10, 5.8)

box(
    3.7, 2.65, 4.6, 0.95,
    "Five-way test comparison\nIoU 0.50 / 0.25; micro F1"
)

arrow(2, 4.8, 4.5, 3.6)
arrow(6, 4.8, 6, 3.6)
arrow(10, 4.8, 7.5, 3.6)

box(
    2.3, 0.8, 4.1, 0.9,
    "Paired image-level bootstrap\n4,000 replicates"
)

box(
    7.4, 0.8, 3.9, 0.9,
    "Targeted expert review\n8 selected cases"
)

arrow(5.4, 2.65, 4.35, 1.7)
arrow(6.6, 2.65, 9.3, 1.7)

ax.set_title(
    "Study Design and Evaluation Workflow",
    fontsize=17,
    pad=18
)

save_figure(
    fig,
    "Figure_6_Study_Workflow"
)


# ----------------------------------------------------------
# FIGURE 7 - Number of annotated findings by category
# ----------------------------------------------------------

data = table5.copy()

fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.barh(
    data["Dental finding"],
    data["Annotated instances"]
)

ax.bar_label(bars, padding=4)

ax.invert_yaxis()
ax.set_xlabel("Number of dataset-annotated instances")

ax.set_title(
    "Distribution of the Six Dental Finding Classes\n"
    "Internal Test Set: 154 Radiographs"
)

ax.set_xlim(
    0,
    data["Annotated instances"].max() * 1.2
)

ax.grid(axis="x", alpha=0.2)

fig.tight_layout()

save_figure(
    fig,
    "Figure_7_Test_Class_Distribution"
)


# ----------------------------------------------------------
# FIGURE 8 - Real qualitative examples
# Uses PRIVATE comparison images prepared in STEP 12B.
# Researcher audit figures; NOT verified diagnoses.
# ----------------------------------------------------------

PRIVATE_ZIP = (
    HUMAN / "Expert_8_Case_Adjudication"
    / "02_Researcher_ONLY"
    / "Researcher_ONLY_All_Comparisons.zip"
)

case_examples = ["CASE_0082", "CASE_0053"]

if PRIVATE_ZIP.is_file():

    fig, axes = plt.subplots(
        2, 1,
        figsize=(16, 12)
    )

    with ZipFile(PRIVATE_ZIP) as z:

        for ax, case_id in zip(
            axes, case_examples
        ):

            target = (
                f"COMPARISON_{case_id}.png"
            )

            assert target in z.namelist()

            with Image.open(
                BytesIO(z.read(target))
            ) as im:
                picture = im.convert("RGB").copy()

            ax.imshow(picture)
            ax.set_title(
                f"{case_id} - Reference-negative "
                "disagreement example",
                fontsize=13
            )
            ax.axis("off")

    fig.suptitle(
        "Qualitative Examples of Annotation Discordance",
        fontsize=17
    )

    fig.tight_layout()

    save_figure(
        fig,
        "Figure_8_Qualitative_Radiographic_Examples"
    )

else:

    print(
        "Figure 8 skipped: private comparison ZIP "
        "was not found."
    )

    print(
        "Figures 6 and 7 were still generated."
    )

print("\nSTATUS: MANUSCRIPT FIGURES COMPLETED")
print("Output:", FIGURES)


In [ ]:

# ==========================================================
# STEP 13E-1 - CONSOLIDATE MANUSCRIPT TABLES
# Final selection: 4 main tables
# Existing results only - no retraining
# ==========================================================

from pathlib import Path
from google.colab import drive
import numpy as np
import pandas as pd

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")

SOURCE = ROOT / "06_Manuscript"
AI_RESULTS = (
    ROOT / "05_Human_Observer_Study"
    / "Final_Comparative_Analysis"
    / "AI_vs_Human_v1/Results"
)

FINAL = SOURCE / "Final_Selected_Materials"
TABLES = FINAL / "Tables"
FIGURES = FINAL / "Figures"

TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

ORDER = [
    "Dentist 01",
    "Dentist 02",
    "YOLOv8s",
    "YOLO11s",
    "RT-DETR-L"
]

CLASSES = [
    "Caries",
    "Missing Teeth",
    "Root Canal Treatment",
    "Filling",
    "Crown",
    "Unerupted Wisdom Teeth"
]

summary = pd.read_csv(
    AI_RESULTS / "AI_vs_Human_Combined_Summary.csv"
)

per_class = pd.read_csv(
    AI_RESULTS / "AI_vs_Human_Combined_Per_Class.csv"
)

bootstrap = pd.read_csv(
    AI_RESULTS / "AI_vs_Human_Paired_Bootstrap.csv"
)

assert len(summary) == 10
assert len(per_class) == 60

# ----------------------------------------------------------
# TABLE 1 - Test dataset characteristics
# ----------------------------------------------------------

table1 = pd.read_csv(
    SOURCE / "Tables/Table_5_Test_Class_Distribution.csv"
)

assert len(table1) == 6
assert table1["Annotated instances"].sum() == 588

table1.to_csv(
    TABLES / "Table_1_Test_Dataset.csv",
    index=False
)

# ----------------------------------------------------------
# TABLE 2 - Merge primary and sensitivity performance
# ----------------------------------------------------------

primary = summary[
    np.isclose(summary["threshold"], 0.50)
].set_index("participant").loc[ORDER]

secondary = summary[
    np.isclose(summary["threshold"], 0.25)
].set_index("participant").loc[ORDER]

table2 = pd.DataFrame({
    "Evaluator": ORDER,
    "Confidence": [
        "-" if pd.isna(v) else f"{v:.2f}"
        for v in primary["confidence"]
    ],
    "TP": primary["TP"].astype(int).values,
    "FP": primary["FP"].astype(int).values,
    "FN": primary["FN"].astype(int).values,
    "Precision (%)": (
        primary["precision"].values * 100
    ).round(2),
    "Recall (%)": (
        primary["recall"].values * 100
    ).round(2),
    "F1 IoU 0.50 (%)": (
        primary["F1"].values * 100
    ).round(2),
    "F1 IoU 0.25 (%)": (
        secondary["F1"].values * 100
    ).round(2)
})

table2.to_csv(
    TABLES / "Table_2_Combined_Performance.csv",
    index=False
)

# ----------------------------------------------------------
# TABLE 3 - Per-class F1 at primary IoU 0.50
# ----------------------------------------------------------

pc = per_class[
    np.isclose(per_class["threshold"], 0.50)
]

table3 = pc.pivot(
    index="class_name",
    columns="participant",
    values="F1"
)

table3 = (
    table3.loc[CLASSES, ORDER] * 100
).round(2)

table3.index.name = "Dental finding"

table3.to_csv(
    TABLES / "Table_3_Per_Class_F1.csv"
)

# ----------------------------------------------------------
# TABLE 4 - Six AI vs dentist paired comparisons
# ----------------------------------------------------------

comparison_rows = []

for model in ["YOLOv8s", "YOLO11s", "RT-DETR-L"]:
    for dentist in ["Dentist 01", "Dentist 02"]:

        hit = bootstrap[
            np.isclose(bootstrap["threshold"], 0.50)
            & (
                (
                    (bootstrap["A"] == model)
                    & (bootstrap["B"] == dentist)
                )
                |
                (
                    (bootstrap["A"] == dentist)
                    & (bootstrap["B"] == model)
                )
            )
        ]

        assert len(hit) == 1

        r = hit.iloc[0]

        diff = float(r["B_minus_A_F1"])
        low = float(r["ci_low"])
        high = float(r["ci_high"])

        # Consistent direction: AI minus dentist
        if r["B"] == dentist:
            diff, low, high = -diff, -high, -low

        comparison_rows.append({
            "Comparison": f"{model} vs {dentist}",
            "F1 difference (pp)": round(diff * 100, 2),
            "95% CI lower (pp)": round(low * 100, 2),
            "95% CI upper (pp)": round(high * 100, 2)
        })

table4 = pd.DataFrame(comparison_rows)

assert len(table4) == 6

table4.to_csv(
    TABLES / "Table_4_Paired_Bootstrap.csv",
    index=False
)

# ----------------------------------------------------------
# Display selected tables
# ----------------------------------------------------------

for number, df in [
    (1, table1),
    (2, table2),
    (3, table3),
    (4, table4)
]:

    print("\n" + "=" * 60)
    print(f"FINAL TABLE {number}")
    print("=" * 60)

    display(df)

print("\nFinal tables directory:", TABLES)
print("\nSTATUS: FOUR MANUSCRIPT TABLES READY")


In [ ]:

# ==========================================================
# STEP 13E-2 - CONSOLIDATE SCIENTIFIC FIGURES
# Final selection: 3 main figures
# Figure 1: Four statistical panels
# Figure 2: Study workflow
# Figure 3: Four radiographic panels
# ==========================================================

from PIL import Image, ImageDraw, ImageFont, ImageOps
from zipfile import ZipFile
from io import BytesIO
from pathlib import Path
import shutil

OLD_FIGURES = SOURCE / "Figures"

def get_font(size, bold=False):
    base = "/usr/share/fonts/truetype/dejavu/"
    filename = (
        "DejaVuSans-Bold.ttf"
        if bold else "DejaVuSans.ttf"
    )

    try:
        return ImageFont.truetype(
            base + filename,
            size
        )
    except OSError:
        return ImageFont.load_default()


FONT_TITLE = get_font(37, bold=True)
FONT_PANEL = get_font(27, bold=True)
FONT_SMALL = get_font(22)


def make_panel(
    image,
    title,
    width,
    height
):

    panel = Image.new(
        "RGB",
        (width, height),
        "white"
    )

    draw = ImageDraw.Draw(panel)

    draw.rectangle(
        [0, 0, width - 1, height - 1],
        outline="#C8CDD3",
        width=2
    )

    draw.text(
        (28, 22),
        title,
        fill="#182230",
        font=FONT_PANEL
    )

    available_width = width - 44
    available_height = height - 115

    fitted = ImageOps.contain(
        image.convert("RGB"),
        (available_width, available_height),
        method=Image.Resampling.LANCZOS
    )

    x = (width - fitted.width) // 2
    y = 85 + (available_height - fitted.height) // 2

    panel.paste(fitted, (x, y))

    return panel


def build_grid(
    panels,
    rows,
    columns,
    title,
    output_path,
    tile_width=1800,
    tile_height=1330
):

    assert len(panels) == rows * columns

    margin = 35
    gap = 24
    header = 105

    total_width = (
        columns * tile_width
        + (columns - 1) * gap
        + 2 * margin
    )

    total_height = (
        rows * tile_height
        + (rows - 1) * gap
        + 2 * margin
        + header
    )

    canvas = Image.new(
        "RGB",
        (total_width, total_height),
        "#F5F6F8"
    )

    draw = ImageDraw.Draw(canvas)

    draw.text(
        (margin, 38),
        title,
        font=FONT_TITLE,
        fill="#14202E"
    )

    for idx, (label, picture) in enumerate(panels):

        row = idx // columns
        column = idx % columns

        x = margin + column * (tile_width + gap)
        y = (
            margin + header
            + row * (tile_height + gap)
        )

        panel = make_panel(
            picture,
            label,
            tile_width,
            tile_height
        )

        canvas.paste(panel, (x, y))

    canvas.save(
        output_path,
        format="PNG",
        dpi=(300, 300),
        optimize=True
    )

    print("Saved:", output_path.name)
    print("Resolution:", canvas.size)


# ----------------------------------------------------------
# FIGURE 1 - Four statistical panels
# ----------------------------------------------------------

statistical_sources = [
    (
        "A | Overall F1 comparison",
        "Figure_1_Five_Way_F1.png"
    ),
    (
        "B | Per-class F1 heatmap",
        "Figure_2_Per_Class_F1_Heatmap.png"
    ),
    (
        "C | Paired bootstrap confidence intervals",
        "Figure_3_AI_Human_Bootstrap_Forest.png"
    ),
    (
        "D | Precision-recall operating points",
        "Figure_4_Precision_Recall_Operating_Points.png"
    )
]

stat_panels = []

for caption, filename in statistical_sources:

    path = OLD_FIGURES / filename

    assert path.is_file(), (
        f"Missing figure: {path}\n"
        "Run STEP 13A/13C before merging figures."
    )

    with Image.open(path) as img:
        stat_panels.append((
            caption,
            img.convert("RGB").copy()
        ))

build_grid(
    panels=stat_panels,
    rows=2,
    columns=2,
    title="Figure 1. Comparative detection performance",
    output_path=(
        FIGURES / "Figure_1_Combined_Statistics.png"
    )
)


# ----------------------------------------------------------
# FIGURE 2 - Study workflow
# ----------------------------------------------------------

workflow_source = (
    OLD_FIGURES / "Figure_6_Study_Workflow.png"
)

assert workflow_source.is_file(), (
    "Run STEP 13D to generate the workflow figure."
)

workflow_target = (
    FIGURES / "Figure_2_Study_Workflow.png"
)

shutil.copy2(
    workflow_source,
    workflow_target
)

print("Saved:", workflow_target.name)


# ----------------------------------------------------------
# FIGURE 3 - Four real radiographic panels
# ----------------------------------------------------------

private_zip = (
    HUMAN / "Expert_8_Case_Adjudication"
    / "02_Researcher_ONLY"
    / "Researcher_ONLY_All_Comparisons.zip"
)

assert private_zip.is_file(), (
    "Researcher-only comparisons ZIP was not found."
)

CASE_ID = "CASE_0082"

qualitative_sources = [
    ("A | Original radiograph",
     f"{CASE_ID}/Original.png"),
    ("B | Dentist 01 annotation",
     f"{CASE_ID}/Dentist_01.png"),
    ("C | Dentist 02 annotation",
     f"{CASE_ID}/Dentist_02.png"),
    ("D | RT-DETR-L predictions",
     f"{CASE_ID}/RT-DETR-L.png")
]

qualitative_panels = []

with ZipFile(private_zip) as z:

    assert z.testzip() is None

    for caption, filename in qualitative_sources:

        assert filename in z.namelist(), filename

        with Image.open(
            BytesIO(z.read(filename))
        ) as image:

            qualitative_panels.append((
                caption,
                image.convert("RGB").copy()
            ))

build_grid(
    panels=qualitative_panels,
    rows=2,
    columns=2,
    title=(
        "Figure 3. Reference-negative case: "
        + CASE_ID
    ),
    output_path=(
        FIGURES / "Figure_3_Qualitative_Comparison.png"
    ),
    tile_width=1250,
    tile_height=1300
)

print("\n" + "=" * 65)
print("FINAL MANUSCRIPT FIGURES")
print("=" * 65)

for path in sorted(FIGURES.glob("Figure_*.png")):
    print(path.name)

print("\nSTATUS: THREE MANUSCRIPT FIGURES READY")


In [ ]:

# ==========================================================
# STEP 14A - EXACT LEXICOGRAPHIC MATCHING AUDIT
# Cached predictions only - NO training / NO GPU
# Does NOT overwrite published or previous results
# ==========================================================

from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as ET
from scipy.optimize import linear_sum_assignment
from collections import defaultdict
from google.colab import drive
import pandas as pd
import numpy as np
import json
import re

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")
HUMAN = ROOT / "05_Human_Observer_Study/Final_Comparative_Analysis"
BASE = HUMAN / "AI_vs_Human_v1"
RESULTS = BASE / "Results"
AUDIT = ROOT / "06_Manuscript/Methodology_Audit"
AUDIT.mkdir(parents=True, exist_ok=True)

MAPPING = pd.read_csv(
    BASE / "Verified_Test_Case_Mapping.csv"
)
LOCK = json.loads(
    (BASE / "AI_Validation_Confidence_Lock.json").read_text()
)

CASE_TO_IMAGE = dict(zip(
    MAPPING["case_id"], MAPPING["original_filename"]
))

IMAGE_TO_CASE = dict(zip(
    MAPPING["original_filename"], MAPPING["case_id"]
))

assert len(CASE_TO_IMAGE) == 154
assert len(IMAGE_TO_CASE) == 154

CLASSES = [
    "Caries", "Missing Teeth", "Root Canal Treatment",
    "Filling", "Crown", "Unerupted Wisdom Teeth"
]

LABEL_DIR = (
    ROOT / "01_Datasets/Prepared_6_Classes/"
    "v1_cleaned/test/labels"
)

# ----------------------------------------------------------
# 1. Read frozen YOLO reference annotations
# ----------------------------------------------------------

GT = defaultdict(list)

for case_id, filename in CASE_TO_IMAGE.items():

    label_path = LABEL_DIR / (
        Path(filename).stem + ".txt"
    )
    assert label_path.is_file(), label_path

    for line in label_path.read_text().splitlines():

        if not line.strip():
            continue

        cls, xc, yc, w, h = map(
            float, line.split()
        )

        GT[(case_id, int(cls))].append((
            xc-w/2, yc-h/2,
            xc+w/2, yc+h/2
        ))

assert sum(map(len, GT.values())) == 588

# ----------------------------------------------------------
# 2. Read original CVAT annotations
# ----------------------------------------------------------

PRED = {}

reader_files = {
    "Dentist 01": HUMAN / "Inputs/Dentist01_CVAT_Annotations.zip",
    "Dentist 02": HUMAN / "Inputs/Dentist02_CVAT_Annotations.zip.zip"
}

for reader, path in reader_files.items():

    assert path.is_file(), path
    groups = defaultdict(list)

    with ZipFile(path) as z:

        xml_names = [
            n for n in z.namelist()
            if Path(n).name == "annotations.xml"
        ]

        assert len(xml_names) == 1
        root = ET.fromstring(z.read(xml_names[0]))
        images = root.findall(".//image")

        assert len(images) == 154

        for image in images:

            match = re.search(
                r"CASE_\d{4}", image.attrib["name"]
            )
            assert match

            case_id = match.group()
            assert case_id in CASE_TO_IMAGE

            iw = float(image.attrib["width"])
            ih = float(image.attrib["height"])

            for box in image.findall("box"):

                cls = CLASSES.index(box.attrib["label"])

                groups[(case_id, cls)].append((
                    float(box.attrib["xtl"]) / iw,
                    float(box.attrib["ytl"]) / ih,
                    float(box.attrib["xbr"]) / iw,
                    float(box.attrib["ybr"]) / ih
                ))

    PRED[reader] = groups

# ----------------------------------------------------------
# 3. Read existing locked AI predictions
# ----------------------------------------------------------

for model in ["YOLOv8s", "YOLO11s", "RT-DETR-L"]:

    path = RESULTS / (
        model.replace("-", "_") + "_Test_Predictions.csv"
    )

    assert path.is_file(), path

    df = pd.read_csv(path)
    confidence = LOCK["selected_confidence"][model]

    df = df[df["confidence"] >= confidence]

    groups = defaultdict(list)

    for r in df.itertuples():

        assert r.case_id in CASE_TO_IMAGE
        assert 0 <= int(r.class_id) < 6

        groups[
            (r.case_id, int(r.class_id))
        ].append((
            float(r.x1), float(r.y1),
            float(r.x2), float(r.y2)
        ))

    PRED[model] = groups

# ----------------------------------------------------------
# 4. IoU and cardinality-first matching
# ----------------------------------------------------------

def iou(a, b):

    ix1 = max(a[0], b[0])
    iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2])
    iy2 = min(a[3], b[3])

    intersection = (
        max(0, ix2 - ix1)
        * max(0, iy2 - iy1)
    )

    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])

    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


def match_count(pred_boxes, ref_boxes, threshold, strict):

    n = len(pred_boxes)
    m = len(ref_boxes)

    if n == 0 or m == 0:
        return 0

    overlaps = np.array([
        [iou(a, b) for b in ref_boxes]
        for a in pred_boxes
    ])

    # With strict=True, one extra valid match must be
    # worth more than any possible total IoU improvement.
    bonus = min(n, m) + 1 if strict else 2.0

    weights = np.zeros((n+m, n+m), dtype=float)

    eligible = overlaps + 1e-12 >= threshold

    weights[:n, :m] = np.where(
        eligible,
        bonus + overlaps,
        0.0
    )

    rows, cols = linear_sum_assignment(
        weights, maximize=True
    )

    return sum(
        1 for i, j in zip(rows, cols)
        if i < n and j < m and eligible[i, j]
    )

# ----------------------------------------------------------
# 5. Calculate both methods on same frozen predictions
# ----------------------------------------------------------

audit_rows = []

for participant, groups in PRED.items():

    for threshold in [0.50, 0.25]:

        for case_id in sorted(CASE_TO_IMAGE):

            for cls in range(6):

                key = (case_id, cls)

                predictions = groups.get(key, [])
                references = GT.get(key, [])

                old_tp = match_count(
                    predictions, references,
                    threshold, strict=False
                )

                new_tp = match_count(
                    predictions, references,
                    threshold, strict=True
                )

                assert new_tp >= old_tp

                audit_rows.append({
                    "participant": participant,
                    "threshold": threshold,
                    "case_id": case_id,
                    "class_id": cls,
                    "detections": len(predictions),
                    "reference_boxes": len(references),
                    "old_TP": old_tp,
                    "new_TP": new_tp,
                    "old_FP": len(predictions)-old_tp,
                    "new_FP": len(predictions)-new_tp,
                    "old_FN": len(references)-old_tp,
                    "new_FN": len(references)-new_tp
                })

audit_df = pd.DataFrame(audit_rows)
assert len(audit_df) == 5 * 2 * 154 * 6

audit_df.to_csv(
    AUDIT / "Matching_Audit_Per_Image_Class.csv",
    index=False
)

summary_rows = []

for (participant, threshold), group in audit_df.groupby(
    ["participant", "threshold"]
):

    row = {
        "participant": participant,
        "threshold": threshold
    }

    for prefix in ["old", "new"]:

        tp = int(group[f"{prefix}_TP"].sum())
        fp = int(group[f"{prefix}_FP"].sum())
        fn = int(group[f"{prefix}_FN"].sum())

        row[f"{prefix}_TP"] = tp
        row[f"{prefix}_FP"] = fp
        row[f"{prefix}_FN"] = fn
        row[f"{prefix}_F1"] = (
            2*tp/(2*tp+fp+fn)
            if 2*tp+fp+fn else 0
        )

    row["TP_change"] = row["new_TP"] - row["old_TP"]
    summary_rows.append(row)

audit_summary = pd.DataFrame(summary_rows)

# Verify historical method against recorded results
original_summary = pd.read_csv(
    RESULTS / "AI_vs_Human_Combined_Summary.csv"
)

verification = audit_summary.merge(
    original_summary[
        ["participant", "threshold", "TP", "FP", "FN"]
    ],
    on=["participant", "threshold"],
    validate="one_to_one"
)

verification["previous_results_reproduced"] = (
    (verification["old_TP"] == verification["TP"])
    & (verification["old_FP"] == verification["FP"])
    & (verification["old_FN"] == verification["FN"])
)

audit_summary.to_csv(
    AUDIT / "Matching_Audit_Summary.csv",
    index=False
)

print("=" * 65)
print("MATCHING ALGORITHM AUDIT")
print("=" * 65)

display(audit_summary)

print(
    "\nHistorical results reproduced:",
    bool(verification["previous_results_reproduced"].all())
)

print(
    "Any TP changes with strict matching:",
    bool(audit_summary["TP_change"].ne(0).any())
)

print("\nAudit files:", AUDIT)
print("\nSTATUS: MATCHING AUDIT COMPLETED")


In [ ]:

# ==========================================================
# STEP 14B - CROSS-SPLIT DUPLICATE AUDIT
# Exact pixel comparison + perceptual dHash screening
# Does not modify images, labels, or dataset splits
# ==========================================================

from pathlib import Path
from PIL import Image
from hashlib import sha256
from itertools import combinations
from google.colab import drive
import pandas as pd
import numpy as np

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")

DATA = (
    ROOT / "01_Datasets/Prepared_6_Classes/v1_cleaned"
)

OUT = ROOT / "06_Manuscript/Methodology_Audit"
OUT.mkdir(parents=True, exist_ok=True)

SPLITS = {
    "train": 1065,
    "valid": 303,
    "test": 154
}

def make_hashes(path):

    with Image.open(path) as image:

        rgb = image.convert("RGB")

        exact = sha256(
            str(rgb.size).encode() + rgb.tobytes()
        ).hexdigest()

        grey = rgb.convert("L").resize(
            (9, 8),
            Image.Resampling.LANCZOS
        )

        arr = np.asarray(grey, dtype=np.int16)

        bits = (arr[:, 1:] > arr[:, :-1]).flatten()

        perceptual = 0

        for bit in bits:
            perceptual = (
                (perceptual << 1) | int(bit)
            )

    return exact, perceptual


records = []

for split, expected in SPLITS.items():

    image_dir = DATA / split / "images"

    images = sorted(
        p for p in image_dir.iterdir()
        if p.suffix.lower() in {
            ".jpg", ".jpeg", ".png", ".bmp"
        }
    )

    assert len(images) == expected, (
        f"{split}: expected {expected}, found {len(images)}"
    )

    print("Hashing:", split, len(images))

    for path in images:

        exact, perceptual = make_hashes(path)

        records.append({
            "split": split,
            "filename": path.name,
            "exact_hash": exact,
            "dhash64": perceptual
        })

records_df = pd.DataFrame(records)

assert len(records_df) == 1522

# ----------------------------------------------------------
# Exact duplicate screening
# ----------------------------------------------------------

exact_rows = []

for _, group in records_df.groupby("exact_hash"):

    if group["split"].nunique() < 2:
        continue

    for a, b in combinations(
        group.to_dict("records"), 2
    ):

        if a["split"] == b["split"]:
            continue

        exact_rows.append({
            "split_A": a["split"],
            "image_A": a["filename"],
            "split_B": b["split"],
            "image_B": b["filename"],
            "hamming_distance": 0,
            "type": "Exact decoded-pixel duplicate"
        })

# ----------------------------------------------------------
# Perceptual near-duplicate candidate screening
# Different patients can still look similar.
# Candidates MUST be reviewed visually.
# ----------------------------------------------------------

near_rows = []

for i, split_a in enumerate(SPLITS):

    for split_b in list(SPLITS)[i+1:]:

        A = records_df[
            records_df["split"] == split_a
        ].to_dict("records")

        B = records_df[
            records_df["split"] == split_b
        ].to_dict("records")

        for a in A:

            for b in B:

                distance = (
                    int(a["dhash64"])
                    ^ int(b["dhash64"])
                ).bit_count()

                if distance <= 4:

                    near_rows.append({
                        "split_A": split_a,
                        "image_A": a["filename"],
                        "split_B": split_b,
                        "image_B": b["filename"],
                        "hamming_distance": distance,
                        "exact_pixel_match": (
                            a["exact_hash"] == b["exact_hash"]
                        ),
                        "review_decision": "Pending"
                    })

exact_df = pd.DataFrame(exact_rows)
near_df = pd.DataFrame(near_rows)

exact_df.to_csv(
    OUT / "Cross_Split_Exact_Duplicates.csv",
    index=False
)

near_df.to_csv(
    OUT / "Cross_Split_Near_Duplicate_Candidates.csv",
    index=False
)

print("\n" + "=" * 65)
print("DATASET LEAKAGE SCREENING REPORT")
print("=" * 65)

print("Total images:", len(records_df))
print("Exact cross-split pairs:", len(exact_df))
print("Near-duplicate candidate pairs:", len(near_df))

if len(exact_df):
    print("\nEXACT DUPLICATES")
    display(exact_df.head(20))

if len(near_df):
    print("\nNEAR-DUPLICATE CANDIDATES")
    display(
        near_df.sort_values(
            "hamming_distance"
        ).head(20)
    )

print("\nIMPORTANT:")
print("Near-duplicate screening is not patient identification.")
print("Review flagged pairs manually before drawing conclusions.")
print("STATUS: CROSS-SPLIT AUDIT COMPLETED")


In [ ]:

# ==========================================================
# STEP 14C - MULTIPLE-COMPARISON-ADJUSTED CIs
# Six AI-vs-dentist comparisons at primary IoU 0.50
# Paired image-level bootstrap, Bonferroni adjustment
# ==========================================================

import numpy as np
import pandas as pd
from pathlib import Path

AUDIT = (
    Path("/content/drive/MyDrive/DentalOPG_Research_2026")
    / "06_Manuscript/Methodology_Audit"
)

RESULTS = (
    Path("/content/drive/MyDrive/DentalOPG_Research_2026")
    / "05_Human_Observer_Study/"
    "Final_Comparative_Analysis/AI_vs_Human_v1/Results"
)

audit = pd.read_csv(
    AUDIT / "Matching_Audit_Summary.csv"
)

assert audit["TP_change"].eq(0).all(), (
    "Matching results changed. Recompute final results first."
)

image_data = pd.read_csv(
    RESULTS / "AI_vs_Human_Combined_Per_Image.csv"
)

source = image_data[
    np.isclose(image_data["threshold"], 0.50)
]

models = ["YOLOv8s", "YOLO11s", "RT-DETR-L"]
dentists = ["Dentist 01", "Dentist 02"]

cases = sorted(source["case_id"].unique())
assert len(cases) == 154

vectors = {}

for participant in models + dentists:

    group = source[
        source["participant"] == participant
    ]

    matrix = group.set_index("case_id")[
        ["TP", "FP", "FN"]
    ].reindex(cases)

    assert not matrix.isna().any().any()

    vectors[participant] = matrix.to_numpy(
        dtype=np.float64
    )


def micro_f1(counts):

    tp = counts[..., 0]
    fp = counts[..., 1]
    fn = counts[..., 2]

    return 2*tp / np.maximum(
        1, 2*tp + fp + fn
    )


rng = np.random.default_rng(20261014)
n_boot = 20000

indices = rng.integers(
    0, 154,
    size=(n_boot, 154)
)

alpha = 0.05
comparisons_count = len(models) * len(dentists)
adjusted_alpha = alpha / comparisons_count

rows = []

for model in models:
    for dentist in dentists:

        a = vectors[model]
        b = vectors[dentist]

        observed = (
            micro_f1(a.sum(axis=0))
            - micro_f1(b.sum(axis=0))
        )

        samples_a = micro_f1(
            a[indices].sum(axis=1)
        )

        samples_b = micro_f1(
            b[indices].sum(axis=1)
        )

        diff = samples_a - samples_b

        nominal = np.quantile(
            diff,
            [0.025, 0.975]
        )

        adjusted = np.quantile(
            diff,
            [
                adjusted_alpha / 2,
                1 - adjusted_alpha / 2
            ]
        )

        rows.append({
            "AI model": model,
            "Dentist": dentist,
            "Difference (pp)": round(
                100*observed, 2
            ),
            "Nominal 95% lower": round(
                100*nominal[0], 2
            ),
            "Nominal 95% upper": round(
                100*nominal[1], 2
            ),
            "Bonferroni CI lower": round(
                100*adjusted[0], 2
            ),
            "Bonferroni CI upper": round(
                100*adjusted[1], 2
            ),
            "Adjusted CI excludes zero": bool(
                adjusted[0] > 0 or adjusted[1] < 0
            )
        })

report = pd.DataFrame(rows)

report.to_csv(
    AUDIT / "Bonferroni_Adjusted_Bootstrap_CIs.csv",
    index=False
)

print("=" * 65)
print("MULTIPLE COMPARISONS AUDIT")
print("=" * 65)

print("Bootstrap replicates:", n_boot)
print("Comparisons:", comparisons_count)
print(
    "Per-comparison CI coverage:",
    round((1-adjusted_alpha)*100, 4), "%"
)

display(report)

print("\nSTATUS: ADJUSTED BOOTSTRAP ANALYSIS READY")


In [ ]:

# ==========================================================
# STEP 14D - SECOND-STAGE CROSS-SPLIT IMAGE SIMILARITY
# pHash + SSIM + visual review
# Read-only: does not change the dataset
# ==========================================================

from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from google.colab import drive
import pandas as pd
import numpy as np
import cv2
import math

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026"
)

DATA = (
    ROOT / "01_Datasets/Prepared_6_Classes/v1_cleaned"
)

OUT = ROOT / "06_Manuscript/Methodology_Audit"

candidate_file = (
    OUT / "Cross_Split_Near_Duplicate_Candidates.csv"
)

assert candidate_file.is_file()

candidates = pd.read_csv(candidate_file)

print("Candidates to screen:", len(candidates))

# ----------------------------------------------------------
# 1. Load each unique image only once
# ----------------------------------------------------------

image_keys = set()

for row in candidates.itertuples():
    image_keys.add((row.split_A, row.image_A))
    image_keys.add((row.split_B, row.image_B))

cache = {}

for split, filename in sorted(image_keys):

    path = DATA / split / "images" / filename

    assert path.is_file(), path

    image = cv2.imread(
        str(path), cv2.IMREAD_GRAYSCALE
    )

    assert image is not None

    small = cv2.resize(
        image, (128, 128),
        interpolation=cv2.INTER_AREA
    )

    cache[(split, filename)] = small

print("Unique images loaded:", len(cache))


# ----------------------------------------------------------
# 2. Calculate perceptual hash
# ----------------------------------------------------------

def phash(image):

    reduced = cv2.resize(
        image, (32, 32),
        interpolation=cv2.INTER_AREA
    ).astype(np.float32)

    dct = cv2.dct(reduced)

    low = dct[:8, :8].flatten()

    median = np.median(low[1:])

    bits = low[1:] > median

    value = 0

    for bit in bits:
        value = (value << 1) | int(bit)

    return value


hash_cache = {
    key: phash(image)
    for key, image in cache.items()
}


# ----------------------------------------------------------
# 3. Structural similarity calculation
# ----------------------------------------------------------

def structural_similarity(a, b):

    x = a.astype(np.float32)
    y = b.astype(np.float32)

    kernel = (11, 11)
    sigma = 1.5

    mux = cv2.GaussianBlur(x, kernel, sigma)
    muy = cv2.GaussianBlur(y, kernel, sigma)

    sigmax = (
        cv2.GaussianBlur(x*x, kernel, sigma)
        - mux*mux
    )

    sigmay = (
        cv2.GaussianBlur(y*y, kernel, sigma)
        - muy*muy
    )

    sigmaxy = (
        cv2.GaussianBlur(x*y, kernel, sigma)
        - mux*muy
    )

    c1 = (0.01 * 255) ** 2
    c2 = (0.03 * 255) ** 2

    numerator = (
        (2*mux*muy+c1)
        * (2*sigmaxy+c2)
    )

    denominator = (
        (mux*mux+muy*muy+c1)
        * (sigmax+sigmay+c2)
    )

    return float(np.mean(
        numerator / np.maximum(denominator, 1e-8)
    ))


# ----------------------------------------------------------
# 4. Score every candidate pair
# ----------------------------------------------------------

scored_rows = []

for row in candidates.itertuples():

    key_a = (row.split_A, row.image_A)
    key_b = (row.split_B, row.image_B)

    a = cache[key_a]
    b = cache[key_b]

    phash_distance = (
        hash_cache[key_a] ^ hash_cache[key_b]
    ).bit_count()

    ssim = structural_similarity(a, b)

    scored_rows.append({
        "split_A": row.split_A,
        "image_A": row.image_A,
        "split_B": row.split_B,
        "image_B": row.image_B,
        "dhash_distance": int(
            row.hamming_distance
        ),
        "phash_distance": phash_distance,
        "SSIM": ssim,
        "high_similarity_flag": bool(
            ssim >= 0.94
            or (
                phash_distance <= 6
                and ssim >= 0.88
            )
        ),
        "manual_review": "Pending"
    })

report = pd.DataFrame(scored_rows)

report = report.sort_values(
    ["SSIM", "phash_distance"],
    ascending=[False, True]
).reset_index(drop=True)

report_path = (
    OUT / "Cross_Split_Advanced_Similarity_Audit.csv"
)

report.to_csv(report_path, index=False)

# ----------------------------------------------------------
# 5. Create a contact sheet for the 24 highest priority
# visually diverse candidate pairs.
# ----------------------------------------------------------

chosen = []
used_images = set()

for r in report.itertuples():

    key_a = (r.split_A, r.image_A)
    key_b = (r.split_B, r.image_B)

    if key_a in used_images or key_b in used_images:
        continue

    chosen.append(r)
    used_images.update([key_a, key_b])

    if len(chosen) == 24:
        break

if len(chosen) < 24:

    already = {
        (r.split_A, r.image_A, r.split_B, r.image_B)
        for r in chosen
    }

    for r in report.itertuples():

        key = (
            r.split_A, r.image_A,
            r.split_B, r.image_B
        )

        if key not in already:
            chosen.append(r)
            already.add(key)

        if len(chosen) == 24:
            break

try:
    font = ImageFont.truetype(
        "/usr/share/fonts/truetype/dejavu/"
        "DejaVuSans.ttf", 15
    )
except OSError:
    font = ImageFont.load_default()

tile_w = 548
tile_h = 324
columns = 2
rows = math.ceil(len(chosen) / columns)

sheet = Image.new(
    "RGB",
    (columns * tile_w, rows * tile_h),
    "white"
)

draw = ImageDraw.Draw(sheet)

for i, r in enumerate(chosen):

    col = i % columns
    row = i // columns

    x = col * tile_w
    y = row * tile_h

    for side, split, filename in [
        (0, r.split_A, r.image_A),
        (1, r.split_B, r.image_B)
    ]:

        path = DATA / split / "images" / filename

        with Image.open(path) as im:
            picture = im.convert("RGB").resize(
                (256, 256)
            )

        sheet.paste(
            picture,
            (x + 8 + side * 270, y + 24)
        )

        draw.text(
            (x + 8 + side * 270, y + 284),
            f"{split}: {filename}",
            font=font,
            fill="black"
        )

    draw.text(
        (x + 8, y + 3),
        (
            f"Pair {i+1} | SSIM {r.SSIM:.3f}"
            f" | pHash {r.phash_distance}"
        ),
        font=font,
        fill="black"
    )

sheet_path = (
    OUT / "Cross_Split_Top_24_Visual_Check.png"
)

sheet.save(sheet_path)

# ----------------------------------------------------------
# 6. Final report
# ----------------------------------------------------------

print("\n" + "=" * 65)
print("ADVANCED DUPLICATE SCREENING")
print("=" * 65)

print("Candidate pairs:", len(report))

print(
    "High similarity flags:",
    int(report["high_similarity_flag"].sum())
)

print(
    "Pairs with SSIM >= 0.95:",
    int((report["SSIM"] >= 0.95).sum())
)

print(
    "Pairs with SSIM >= 0.98:",
    int((report["SSIM"] >= 0.98).sum())
)

display(report.head(20))

print("\nDetailed report:", report_path)
print("Visual comparison:", sheet_path)

print("\nImportant:")
print("Similarity flags are NOT confirmed duplicates.")
print("Patient independence cannot be inferred from hashes.")
print("Original dataset modified: NO")

print("\nSTATUS: ADVANCED SIMILARITY AUDIT READY")


In [ ]:

# ==========================================================
# EXPORT FINAL TABLES AND FIGURES FOR WORD MANUSCRIPT
# ==========================================================

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import drive, files

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/DentalOPG_Research_2026"
)

FINAL = (
    ROOT / "06_Manuscript/Final_Selected_Materials"
)

TABLES = FINAL / "Tables"
FIGURES = FINAL / "Figures"

expected_tables = [
    "Table_1_Test_Dataset.csv",
    "Table_2_Combined_Performance.csv",
    "Table_3_Per_Class_F1.csv",
    "Table_4_Paired_Bootstrap.csv"
]

expected_figures = [
    "Figure_1_Combined_Statistics.png",
    "Figure_2_Study_Workflow.png",
    "Figure_3_Qualitative_Comparison.png"
]

for filename in expected_tables:
    assert (TABLES / filename).is_file(), filename

for filename in expected_figures:
    assert (FIGURES / filename).is_file(), filename

output = (
    ROOT / "06_Manuscript"
    / "Manuscript_Tables_And_Figures.zip"
)

with ZipFile(output, "w", ZIP_DEFLATED) as z:

    for filename in expected_tables:
        z.write(
            TABLES / filename,
            arcname=f"Tables/{filename}"
        )

    for filename in expected_figures:
        z.write(
            FIGURES / filename,
            arcname=f"Figures/{filename}"
        )

print("=" * 60)
print("MANUSCRIPT ASSETS EXPORT")
print("=" * 60)
print("Tables:", len(expected_tables))
print("Figures:", len(expected_figures))
print("File:", output)
print("STATUS: MANUSCRIPT ASSETS READY")

files.download(str(output))


In [ ]:

# ==========================================================
# FINAL METHODS CHECK - EXTRACT ACTUAL TRAINING SETTINGS
# Read saved training arguments; no model training
# ==========================================================

from pathlib import Path
import yaml
import pandas as pd

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")

RUNS = {
    "YOLOv8s": ROOT / "03_Trained_Models/YOLOv8s/YOLOv8s_seed42",
    "YOLO11s": ROOT / "03_Trained_Models/YOLO11s/YOLO11s_seed42",
    "RT-DETR-L": ROOT / "03_Trained_Models/RT_DETR/RTDETR_L_seed42"
}

KEYS = [
    "optimizer", "lr0", "lrf", "momentum", "weight_decay",
    "warmup_epochs", "batch", "epochs", "patience",
    "imgsz", "seed", "deterministic",
    "hsv_h", "hsv_s", "hsv_v", "degrees",
    "translate", "scale", "shear",
    "fliplr", "flipud", "mosaic", "mixup",
    "close_mosaic"
]

rows = []

for model, folder in RUNS.items():

    config = folder / "args.yaml"

    if not config.exists():
        print(f"WARNING: Missing {config}")
        continue

    with open(config, encoding="utf-8") as f:
        settings = yaml.safe_load(f) or {}

    row = {"Model": model}

    for key in KEYS:
        row[key] = settings.get(key, "Not recorded")

    rows.append(row)

report = pd.DataFrame(rows)

display(report.T)

output = ROOT / "06_Manuscript/Training_Configuration_Extract.csv"
report.to_csv(output, index=False)

print("\nSaved:", output)
print("STATUS: TRAINING CONFIGURATION EXTRACTED")


In [ ]:

# ==========================================================
# STEP 15A - FINAL SUPPLEMENTARY TABLES S1-S4
# Source: frozen research results in Google Drive
# No training, no inference, no modification of source data
# ==========================================================

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import drive
import pandas as pd
import numpy as np

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")

HUMAN = (
    ROOT / "05_Human_Observer_Study"
    / "Final_Comparative_Analysis"
)

AI_RESULTS = HUMAN / "AI_vs_Human_v1/Results"
AUDIT = ROOT / "06_Manuscript/Methodology_Audit"
MANUSCRIPT = ROOT / "06_Manuscript"

SUPP = MANUSCRIPT / "Supplementary_Material"
TABLES = SUPP / "Tables"
TABLES.mkdir(parents=True, exist_ok=True)

MODELS = ["YOLOv8s", "YOLO11s", "RT-DETR-L"]

EVALUATORS = [
    "Dentist 01",
    "Dentist 02",
    "YOLOv8s",
    "YOLO11s",
    "RT-DETR-L"
]

CLASSES = [
    "Caries",
    "Missing Teeth",
    "Root Canal Treatment",
    "Filling",
    "Crown",
    "Unerupted Wisdom Teeth"
]

per_class = pd.read_csv(
    AI_RESULTS / "AI_vs_Human_Combined_Per_Class.csv"
)

matching_audit = pd.read_csv(
    AUDIT / "Matching_Audit_Summary.csv"
)

similarity = pd.read_csv(
    AUDIT / "Cross_Split_Advanced_Similarity_Audit.csv"
)

assert len(per_class) == 60
assert len(matching_audit) == 10
assert len(similarity) == 2294

# ----------------------------------------------------------
# S1 - TRAINING CONFIGURATIONS
# ----------------------------------------------------------

training = pd.read_csv(
    MANUSCRIPT / "Training_Configuration_Extract.csv"
)

assert set(training["Model"]) == set(MODELS)

training = training.set_index("Model")

settings = [
    ("Input size (pixels)", "imgsz"),
    ("Batch size", "batch"),
    ("Maximum epochs", "epochs"),
    ("Early stopping patience", "patience"),
    ("Optimizer configuration", "optimizer"),
    ("Configured lr0", "lr0"),
    ("Configured lrf", "lrf"),
    ("Configured momentum", "momentum"),
    ("Configured weight decay", "weight_decay"),
    ("Warm-up epochs", "warmup_epochs"),
    ("Random seed", "seed"),
    ("Deterministic execution", "deterministic"),
    ("Hue augmentation", "hsv_h"),
    ("Saturation augmentation", "hsv_s"),
    ("Value augmentation", "hsv_v"),
    ("Degrees", "degrees"),
    ("Translation", "translate"),
    ("Scaling", "scale"),
    ("Shearing", "shear"),
    ("Horizontal flipping", "fliplr"),
    ("Vertical flipping", "flipud"),
    ("Mosaic", "mosaic"),
    ("MixUp", "mixup"),
    ("Close mosaic (epochs)", "close_mosaic")
]

completed_epochs = {
    "YOLOv8s": 96,
    "YOLO11s": 60,
    "RT-DETR-L": 77
}

best_epochs = {
    "YOLOv8s": 71,
    "YOLO11s": 35,
    "RT-DETR-L": 52
}

s1_rows = []

for label, key in settings:
    assert key in training.columns

    row = {"Parameter": label}
    for model in MODELS:
        row[model] = training.at[model, key]

    s1_rows.append(row)

for label, values in [
    ("Completed epochs", completed_epochs),
    ("Best checkpoint epoch", best_epochs)
]:
    row = {"Parameter": label}
    row.update(values)
    s1_rows.append(row)

s1 = pd.DataFrame(s1_rows)

s1_path = TABLES / "Supplementary_Table_S1_Training.csv"
s1.to_csv(s1_path, index=False)

# ----------------------------------------------------------
# S2 - COMPLETE PER-CLASS PRIMARY METRICS (30 ROWS)
# ----------------------------------------------------------

primary = per_class[
    np.isclose(per_class["threshold"], 0.50)
].copy()

assert len(primary) == 30

s2 = primary[
    [
        "participant", "class_name",
        "TP", "FP", "FN",
        "precision", "recall", "F1"
    ]
].copy()

s2["Reference instances"] = s2["TP"] + s2["FN"]

for metric in ["precision", "recall", "F1"]:
    s2[metric] = (100 * s2[metric]).round(2)

s2 = s2.rename(columns={
    "participant": "Evaluator",
    "class_name": "Dental finding",
    "precision": "Precision (%)",
    "recall": "Recall (%)",
    "F1": "F1 (%)"
})

s2 = s2[
    [
        "Evaluator", "Dental finding",
        "Reference instances",
        "TP", "FP", "FN",
        "Precision (%)", "Recall (%)", "F1 (%)"
    ]
]

s2["Evaluator"] = pd.Categorical(
    s2["Evaluator"],
    categories=EVALUATORS,
    ordered=True
)

s2["Dental finding"] = pd.Categorical(
    s2["Dental finding"],
    categories=CLASSES,
    ordered=True
)

s2 = s2.sort_values(
    ["Evaluator", "Dental finding"]
).reset_index(drop=True)

assert len(s2) == 30

s2_path = TABLES / "Supplementary_Table_S2_Primary_Per_Class.csv"
s2.to_csv(s2_path, index=False)

# ----------------------------------------------------------
# S3 - EXPLORATORY PER-CLASS RESULTS AND F1 CHANGES
# ----------------------------------------------------------

secondary = per_class[
    np.isclose(per_class["threshold"], 0.25)
].copy()

assert len(secondary) == 30

s3 = secondary.merge(
    primary[
        ["participant", "class_id", "F1"]
    ].rename(columns={"F1": "primary_F1"}),
    on=["participant", "class_id"],
    validate="one_to_one"
)

s3["F1 increase (pp)"] = (
    (s3["F1"] - s3["primary_F1"]) * 100
).round(2)

for metric in ["precision", "recall", "F1", "primary_F1"]:
    s3[metric] = (100 * s3[metric]).round(2)

s3 = s3.rename(columns={
    "participant": "Evaluator",
    "class_name": "Dental finding",
    "precision": "Precision at IoU 0.25 (%)",
    "recall": "Recall at IoU 0.25 (%)",
    "primary_F1": "F1 at IoU 0.50 (%)",
    "F1": "F1 at IoU 0.25 (%)"
})

s3 = s3[
    [
        "Evaluator",
        "Dental finding",
        "TP", "FP", "FN",
        "Precision at IoU 0.25 (%)",
        "Recall at IoU 0.25 (%)",
        "F1 at IoU 0.50 (%)",
        "F1 at IoU 0.25 (%)",
        "F1 increase (pp)"
    ]
]

s3["Evaluator"] = pd.Categorical(
    s3["Evaluator"],
    categories=EVALUATORS,
    ordered=True
)

s3["Dental finding"] = pd.Categorical(
    s3["Dental finding"],
    categories=CLASSES,
    ordered=True
)

s3 = s3.sort_values(
    ["Evaluator", "Dental finding"]
).reset_index(drop=True)

assert len(s3) == 30

s3_path = TABLES / "Supplementary_Table_S3_IoU_Sensitivity.csv"
s3.to_csv(s3_path, index=False)

# ----------------------------------------------------------
# S4 - METHOD AND DATA INTEGRITY AUDIT
# ----------------------------------------------------------

assert matching_audit["TP_change"].eq(0).all()

high_similarity = int(
    similarity["high_similarity_flag"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
    .sum()
)

ssim_095 = int((similarity["SSIM"] >= 0.95).sum())

assert high_similarity == 0
assert ssim_095 == 0

# Exact duplicate CSV can be empty (no header).
exact_path = AUDIT / "Cross_Split_Exact_Duplicates.csv"

assert exact_path.is_file()

try:
    exact_count = len(pd.read_csv(exact_path))
except pd.errors.EmptyDataError:
    exact_count = 0

assert exact_count == 0

s4_data = [
    ("Dataset", "Final radiographs", "1,522"),
    (
        "Dataset",
        "Train / validation / test",
        "1,065 / 303 / 154"
    ),
    ("Reference", "Test reference boxes", "588"),
    (
        "Reference",
        "Images without six-class reference boxes",
        "23"
    ),
    (
        "Reader QC",
        "Dentist 01 boxes / images",
        "565 / 154"
    ),
    (
        "Reader QC",
        "Dentist 02 boxes / images",
        "606 / 154"
    ),
    (
        "Reader QC",
        "Invalid or exactly duplicated boxes",
        "0 for both dentists"
    ),
    (
        "Image integrity",
        "Exact cross-split duplicate pairs",
        str(exact_count)
    ),
    (
        "Image integrity",
        "dHash candidate pairs",
        str(len(similarity))
    ),
    (
        "Image integrity",
        "Advanced high-similarity flags",
        str(high_similarity)
    ),
    (
        "Image integrity",
        "Candidate pairs with SSIM >= 0.95",
        str(ssim_095)
    ),
    (
        "Matching audit",
        "Combinations reproduced",
        f"{len(matching_audit)}/10"
    ),
    (
        "Matching audit",
        "Changes in matched TP counts",
        str(int(matching_audit["TP_change"].sum()))
    ),
    (
        "Expert review",
        "Selected images",
        "8"
    ),
    (
        "Expert review",
        "Independent expert CVAT boxes",
        "14"
    ),
    (
        "Expert review",
        "Accepted / rejected detection proposals",
        "23 / 2"
    ),
    (
        "Study limitation",
        "Patient-level independence",
        "Not established"
    )
]

s4 = pd.DataFrame(
    s4_data,
    columns=[
        "Audit domain", "Verification item", "Result"
    ]
)

s4_path = TABLES / "Supplementary_Table_S4_Methodology_Audit.csv"
s4.to_csv(s4_path, index=False)

# ----------------------------------------------------------
# FINAL VALIDATION AND ZIP EXPORT
# ----------------------------------------------------------

for number, df in [
    ("S1", s1),
    ("S2", s2),
    ("S3", s3),
    ("S4", s4)
]:
    print("\n" + "=" * 65)
    print(f"SUPPLEMENTARY TABLE {number}")
    print("=" * 65)
    display(df)

zip_path = SUPP / "Supplementary_Tables_S1_to_S4.zip"

with ZipFile(zip_path, "w", ZIP_DEFLATED) as z:
    for path in [s1_path, s2_path, s3_path, s4_path]:
        z.write(path, arcname=path.name)

print("\n" + "=" * 65)
print("SUPPLEMENTARY MATERIAL READY")
print("=" * 65)

print("Tables created:", 4)
print("S2 class-specific rows:", len(s2))
print("S3 sensitivity rows:", len(s3))
print("Output:", TABLES)
print("ZIP:", zip_path)

print("\nSTATUS: FOUR SUPPLEMENTARY TABLES READY")


In [ ]:

# ==========================================================
# STEP 15B - UPDATE MAIN TABLE 4 AND FIGURE 2C
# Bonferroni-adjusted bootstrap confidence intervals
# Rebuild combined Figure 2 with corrected panel C
# No training / no inference
# ==========================================================

from pathlib import Path
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/DentalOPG_Research_2026")

MANUSCRIPT = ROOT / "06_Manuscript"

FINAL = MANUSCRIPT / "Final_Selected_Materials"
FINAL_TABLES = FINAL / "Tables"
FINAL_FIGURES = FINAL / "Figures"
ORIGINAL_FIGURES = MANUSCRIPT / "Figures"

AUDIT = MANUSCRIPT / "Methodology_Audit"

for folder in [FINAL_TABLES, FINAL_FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)

source = AUDIT / "Bonferroni_Adjusted_Bootstrap_CIs.csv"

assert source.is_file(), source

df = pd.read_csv(source)

assert len(df) == 6

# ----------------------------------------------------------
# 1. CREATE CORRECTED MAIN TABLE 4
# ----------------------------------------------------------

table4 = pd.DataFrame({
    "Comparison": (
        df["AI model"] + " vs " + df["Dentist"]
    ),
    "AI minus dentist F1 (pp)": df["Difference (pp)"],
    "Bonferroni CI lower (pp)": df["Bonferroni CI lower"],
    "Bonferroni CI upper (pp)": df["Bonferroni CI upper"],
    "Adjusted CI excludes zero": (
        df["Adjusted CI excludes zero"]
    )
})

old_table_path = (
    FINAL_TABLES / "Table_4_Paired_Bootstrap.csv"
)

backup_path = (
    FINAL_TABLES
    / "Table_4_Paired_Bootstrap_Pre_Bonferroni.csv"
)

if old_table_path.exists() and not backup_path.exists():
    shutil.copy2(old_table_path, backup_path)

table4.to_csv(
    old_table_path,
    index=False
)

print("=" * 65)
print("UPDATED MAIN TABLE 4")
print("=" * 65)

display(table4)

print("Saved:", old_table_path)

# ----------------------------------------------------------
# 2. CREATE CORRECTED FOREST PLOT
# ----------------------------------------------------------

labels = table4["Comparison"].tolist()

estimate = table4[
    "AI minus dentist F1 (pp)"
].to_numpy(dtype=float)

low = table4[
    "Bonferroni CI lower (pp)"
].to_numpy(dtype=float)

high = table4[
    "Bonferroni CI upper (pp)"
].to_numpy(dtype=float)

assert np.all(low <= estimate)
assert np.all(high >= estimate)

y = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(10.5, 6.5))

ax.errorbar(
    estimate,
    y,
    xerr=[
        estimate - low,
        high - estimate
    ],
    fmt="o",
    markersize=7,
    linewidth=1.7,
    capsize=5
)

ax.axvline(
    0,
    linestyle="--",
    linewidth=1.2
)

ax.set_yticks(y, labels)
ax.invert_yaxis()

ax.set_xlabel(
    "F1 difference: AI minus dentist (percentage points)"
)

ax.set_title(
    "Bonferroni-Adjusted Paired Bootstrap Comparisons"
)

ax.grid(axis="x", alpha=0.25)

ax.set_xlim(
    min(0, low.min()) - 2,
    max(0, high.max()) + 2
)

fig.text(
    0.5,
    0.01,
    "20,000 bootstrap replicates; six comparisons; "
    "99.17% individual CIs",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.04, 1, 1])

forest_path = (
    FINAL_FIGURES / "Figure_2C_Bonferroni_Forest.png"
)

fig.savefig(
    forest_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("\nCorrected forest plot:", forest_path)

# ----------------------------------------------------------
# 3. REBUILD COMBINED FIGURE 2 (PANELS A-D)
# ----------------------------------------------------------

panel_paths = [
    ORIGINAL_FIGURES / "Figure_1_Five_Way_F1.png",
    ORIGINAL_FIGURES / "Figure_2_Per_Class_F1_Heatmap.png",
    forest_path,
    ORIGINAL_FIGURES
    / "Figure_4_Precision_Recall_Operating_Points.png"
]

for path in panel_paths:
    assert path.is_file(), f"Missing panel: {path}"

panel_titles = [
    "A. Overall F1 performance",
    "B. Class-specific F1 heatmap",
    "C. Bonferroni-adjusted F1 differences",
    "D. Precision-recall operating points"
]

fig, axes = plt.subplots(
    2, 2,
    figsize=(18, 13)
)

for ax, path, title in zip(
    axes.flat,
    panel_paths,
    panel_titles
):
    image = plt.imread(str(path))

    ax.imshow(image)
    ax.set_title(title, fontsize=13, pad=12)
    ax.axis("off")

fig.suptitle(
    "Comparative Performance of AI Models "
    "and Human Observers",
    fontsize=18,
    y=0.995
)

fig.tight_layout(rect=[0, 0, 1, 0.975])

combined_path = (
    FINAL_FIGURES
    / "Figure_2_Combined_Statistics_Bonferroni.png"
)

fig.savefig(
    combined_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

print("\n" + "=" * 65)
print("MAIN MANUSCRIPT MATERIALS UPDATED")
print("=" * 65)

print("Main Table 4:", old_table_path)
print("Corrected Figure 2C:", forest_path)
print("Combined Figure 2:", combined_path)

print("\nSTATUS: TABLE 4 AND FIGURE 2 UPDATED")


In [ ]:

from google.colab import drive, files

drive.mount("/content/drive")

# Upload dentalopg_prepare_public_release.py
uploaded = files.upload()

script = next(
    name for name in uploaded
    if name == "dentalopg_prepare_public_release.py"
)

exec(
    compile(
        uploaded[script].decode("utf-8"),
        script,
        "exec"
    )
)


In [ ]:

# ==========================================================
# STEP 16B - AUDIT STAGED FILES AND LOCATE COLAB NOTEBOOKS
# READ ONLY - NOTHING IS UPLOADED OR MODIFIED
# ==========================================================

from pathlib import Path
import pandas as pd

DRIVE = Path("/content/drive/MyDrive")
ROOT = DRIVE / "DentalOPG_Research_2026"

AUDIT = (
    ROOT /
    "07_Public_Release_Candidate_PRIVATE_AUDIT.csv"
)

assert AUDIT.exists(), "Private audit report not found."

df = pd.read_csv(AUDIT).fillna("")

print("=" * 65)
print("1. FILES INCLUDED IN THE RELEASE CANDIDATE")
print("=" * 65)

staged = df[df["status"] == "STAGED"]

for _, row in staged.iterrows():
    print(
        f"{row['candidate_path']} "
        f"<-- {row['source']}"
    )

print("\n" + "=" * 65)
print("2. SOURCE CODE CURRENTLY INCLUDED")
print("=" * 65)

code = staged[
    staged["candidate_path"].str.startswith("code/")
]

if len(code):
    for _, row in code.iterrows():
        print(row["source"])
else:
    print("NO SOURCE CODE FOUND")

print("\n" + "=" * 65)
print("3. POSSIBLE RESEARCH NOTEBOOKS IN GOOGLE DRIVE")
print("=" * 65)

folders = [
    ROOT,
    DRIVE / "Colab Notebooks"
]

found = []

for folder in folders:
    if not folder.exists():
        continue

    for p in folder.rglob("*"):
        if not p.is_file():
            continue

        if p.suffix.lower() not in [".ipynb", ".py"]:
            continue

        if "07_Public_Release_Candidate" in str(p):
            continue

        found.append(p)

found = sorted(
    set(found),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

print("Total candidate code files:", len(found))

for i, p in enumerate(found[:60], start=1):
    size_kb = p.stat().st_size / 1024
    print(f"{i:02d}. {p}")
    print(f"    Size: {size_kb:.1f} KB")

print("\n" + "=" * 65)
print("4. INITIAL AUDIT SUMMARY")
print("=" * 65)

print(df["status"].value_counts().to_string())

print("\nSTATUS: SOURCE CODE INVENTORY COMPLETED")
